In [11]:
# -*- coding: utf-8 -*-
"""
ALIGN (B-scale) → MATCH (ORIGINAL SCALE) → EXPORT ALL CSVS
- A,B px/um로 스케일정합 (A를 B 스케일로 NEAREST 리샘플)
- 바이너리 & AND-겹침 최대가 되는 (dy,dx) [B-스케일 정수픽셀] 탐색 (coarse→refine)
- 이동 후 A_on_B 생성 (라벨 보존)
- 매칭은 '원본 스케일(A-스케일)'에서 수행:
    * B 센트로이드를 A-스케일로 변환, 이동벡터도 A-스케일로 환산
    * 헝가리안(언매치 허용)
- pairs.csv에 두 스케일 좌표/벡터/거리/µm단위/형태특징/IoU까지 전부 기록
"""

import os
import cv2
import numpy as np
import tifffile as tiff
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import distance
from scipy.optimize import linear_sum_assignment
from skimage.measure import regionprops

# ======================= 사용자 설정 =======================
MASK_A_PATH = r"C:\Users\oxfil\0916_cellpose_formathcedimages\10X\Position_similar_ROIA004_10_new_seg.npy"
MASK_B_PATH = r"C:\Users\oxfil\Cell_optical_media_comparision\July_22_ROI_A_004_RGB_seg.npy"
IMG_B_PATH  = r"C:\Users\oxfil\Cell_optical_media_comparision\July_22_ROI_A_004_RGB.tif"  # 옵션(오버레이용)
OUT_DIR = r"."

# 스케일(px/um)
PX_PER_UM_A = 1.00
PX_PER_UM_B = 2.68  # scale = PX_PER_UM_B / PX_PER_UM_A

# 탐색 범위/간격 (B 좌표계, 정수 픽셀)
COARSE_RADIUS_YX = (500, 500)
COARSE_STRIDE    = 60
REFINE_RADIUS_YX = (60, 60)
REFINE_STRIDE    = 2

# 매칭 파라미터(원본 A-스케일에서의 게이팅을 위해 환산 사용)
FINAL_MAX_DIST_BPX   = 100.0
UNMATCHED_PENALTY_B  = 200.0

UPSCALE_VIS = 1
INVERT_VIEW = False
# ==========================================================


def load_mask(path):
    ext = os.path.splitext(path.lower())[1]
    if ext in (".tif", ".tiff"):
        return np.asarray(tiff.imread(path)).astype(np.int32)
    elif ext == ".npy":
        data = np.load(path, allow_pickle=True)
        if isinstance(data, dict) and "masks" in data:
            return np.asarray(data["masks"]).astype(np.int32)
        if isinstance(data, np.ndarray):
            if data.dtype == object and isinstance(data.flat[0], dict) and "masks" in data.flat[0]:
                return np.asarray(data.flat[0]["masks"]).astype(np.int32)
            if data.ndim >= 2:
                return data.astype(np.int32)
        raise TypeError("Unsupported npy structure.")
    else:
        arr = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if arr is None:
            raise FileNotFoundError(path)
        return np.asarray(arr).astype(np.int32)


def nearest_resample_labels(src, scale):
    H, W = src.shape
    W2 = int(round(W * scale))
    H2 = int(round(H * scale))
    return cv2.resize(src, (W2, H2), interpolation=cv2.INTER_NEAREST)


def to_binary(mask):
    return (mask > 0).astype(np.uint8)


def overlap_count(binA, binB, dy, dx):
    Ha, Wa = binA.shape
    Hb, Wb = binB.shape
    dy = int(dy)
    dx = int(dx)

    y0A = max(0, -dy)
    y1A = min(Ha, Hb - dy)
    x0A = max(0, -dx)
    x1A = min(Wa, Wb - dx)

    if y1A <= y0A or x1A <= x0A:
        return 0

    y0B = max(0, dy)
    y1B = y0B + (y1A - y0A)
    x0B = max(0, dx)
    x1B = x0B + (x1A - x0A)

    Aov = binA[y0A:y1A, x0A:x1A]
    Bov = binB[y0B:y1B, x0B:x1B]
    return int((Aov * Bov).sum())


def grid_search_peak(binA, binB, center_dy, center_dx, rad_yx, stride, tag):
    ry, rx = rad_yx
    best = {"ov": -1, "dy": 0, "dx": 0}
    ys = range(center_dy - ry, center_dy + ry + 1, stride)
    xs = range(center_dx - rx, center_dx + rx + 1, stride)
    tried = 0

    for dy in ys:
        for dx in xs:
            tried += 1
            ov = overlap_count(binA, binB, dy, dx)
            if ov > best["ov"]:
                best.update({"ov": ov, "dy": int(dy), "dx": int(dx)})

    print(f"[{tag}] tried={tried}  best_ov={best['ov']}  shift=({best['dy']},{best['dx']})")
    return best


def translate_labels_to_canvas(src, dy, dx, canvas_shape):
    HB, WB = canvas_shape
    out = np.zeros((HB, WB), dtype=src.dtype)
    dy = int(dy)
    dx = int(dx)
    H, W = src.shape

    y0s = max(0, -dy)
    y1s = min(W if False else H, HB - dy)
    x0s = max(0, -dx)
    x1s = min(W, WB - dx)

    y0d = max(0, dy)
    y1d = y0d + (y1s - y0s)
    x0d = max(0, dx)
    x1d = x0d + (x1s - x0s)

    if y1s > y0s and x1s > x0s:
        out[y0d:y1d, x0d:x1d] = src[y0s:y1s, x0s:x1s]
    return out


def extract_features(mask):
    """
    Shape-only PCA / pair similarity를 위해 shape-related features 확장
    """
    rows = []
    for p in regionprops(mask):
        y, x = p.centroid
        area = float(p.area)
        peri = float(p.perimeter)

        major = float(p.major_axis_length)
        minor = float(p.minor_axis_length)
        aspect = (major / minor) if minor > 0 else np.nan
        circ = (4.0 * np.pi * area) / (peri**2 + 1e-12)

        rows.append(dict(
            label=int(p.label),
            cx=float(x),
            cy=float(y),

            # size-related
            area=area,
            perimeter=peri,
            major_axis_length=major,
            minor_axis_length=minor,
            equivalent_diameter=float(p.equivalent_diameter),

            # shape-only 핵심
            aspect=aspect,
            circularity=circ,
            solidity=float(p.solidity) if hasattr(p, "solidity") else np.nan,
            eccentricity=float(p.eccentricity) if hasattr(p, "eccentricity") else np.nan,
            extent=float(p.extent) if hasattr(p, "extent") else np.nan,

            # 보조 shape descriptors
            orientation=float(p.orientation) if hasattr(p, "orientation") else np.nan,
            convex_area=float(p.convex_area) if hasattr(p, "convex_area") else np.nan,
            bbox_area=float((p.bbox[2] - p.bbox[0]) * (p.bbox[3] - p.bbox[1])),
        ))
    return pd.DataFrame(rows)


def hungarian_with_unmatched(Axy, Bxy, max_dist, unmatched_penalty):
    Axy = np.asarray(Axy, float)
    Bxy = np.asarray(Bxy, float)
    nA, nB = len(Axy), len(Bxy)

    if nA == 0 and nB == 0:
        return [], [], []
    if nA == 0:
        return [], [], list(range(nB))
    if nB == 0:
        return [], list(range(nA)), []

    C = distance.cdist(Axy, Bxy)
    BIG = 1e9
    C = np.where(C <= max_dist, C, BIG)

    PA = np.full((nA, nA), unmatched_penalty, float)
    PB = np.full((nB, nB), unmatched_penalty, float)
    cost = np.vstack([np.hstack([C, PA]), np.hstack([PB, np.zeros((nB, nA))])])

    r, c = linear_sum_assignment(cost)
    matches = []
    usedA = np.zeros(nA, bool)
    usedB = np.zeros(nB, bool)

    for i, j in zip(r, c):
        if i < nA and j < nB and C[i, j] < BIG:
            matches.append((i, j, float(C[i, j])))
            usedA[i] = True
            usedB[j] = True

    unA = [i for i in range(nA) if not usedA[i]]
    unB = [j for j in range(nB) if not usedB[j]]
    return matches, unA, unB


def load_bg_image(path, shape_hw, invert=False):
    H, W = shape_hw
    if path and os.path.exists(path):
        img = tiff.imread(path) if path.lower().endswith((".tif", ".tiff")) else cv2.imread(path, cv2.IMREAD_UNCHANGED)

        if img.ndim == 2:
            img8 = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            if invert:
                img8 = 255 - img8
            bg = cv2.cvtColor(img8, cv2.COLOR_GRAY2BGR)
        else:
            img8 = img if img.dtype == np.uint8 else cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            if invert:
                img8 = 255 - img8
            bg = img8[:, :, :3]

        return cv2.resize(bg, (W, H), interpolation=cv2.INTER_LANCZOS4)
    else:
        return np.zeros((H, W, 3), dtype=np.uint8)


def draw_vectors(bg_bgr, Axy_B, Bxy_B, matches, upscale, out_png):
    H, W = bg_bgr.shape[:2]
    canvas = cv2.resize(bg_bgr, (W * upscale, H * upscale), interpolation=cv2.INTER_LANCZOS4)

    def T(pt):
        return (int(round(pt[0] * upscale)), int(round(pt[1] * upscale)))

    r = max(2, upscale // 2)

    for x, y in Bxy_B:
        cv2.circle(canvas, T((x, y)), r, (255, 200, 0), -1)
    for x, y in Axy_B:
        cv2.circle(canvas, T((x, y)), r, (0, 220, 0), -1)

    for iA, iB, _ in matches:
        a = Axy_B[iA]
        b = Bxy_B[iB]
        cv2.arrowedLine(
            canvas, T((a[0], a[1])), T((b[0], b[1])),
            (255, 255, 255), max(1, upscale), tipLength=0.25, line_type=cv2.LINE_AA
        )

    cv2.imwrite(out_png, canvas)


def compute_iou_for_pair(A_on_B, Bmask, labelA, labelB):
    Abin = (A_on_B == int(labelA))
    Bbin = (Bmask == int(labelB))
    inter = int(np.logical_and(Abin, Bbin).sum())
    uni = int(np.logical_or(Abin, Bbin).sum())
    return (inter / uni) if uni > 0 else 0.0


def scatter_and_corr(x, y, name, out_dir):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    if len(x) < 2:
        print(f"[{name}] 유효 페어 부족")
        return

    r = np.corrcoef(x, y)[0, 1]
    a, b = np.polyfit(x, y, 1)
    xx = np.linspace(x.min(), x.max(), 100)
    yy = a * xx + b

    plt.figure(figsize=(5, 5))
    plt.scatter(x, y, s=12, alpha=0.6)
    plt.plot(xx, yy, lw=2)
    plt.xlabel(f"{name} (A)")
    plt.ylabel(f"{name} (B)")
    plt.title(f"{name}: r={r:.3f}")
    plt.tight_layout()

    out = os.path.join(out_dir, f"scatter_{name}.png")
    plt.savefig(out, dpi=200)
    plt.close()
    print(f"[Saved] {out}")


def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    # --- Load ---
    A = load_mask(MASK_A_PATH)
    B = load_mask(MASK_B_PATH)
    print(f"[SHAPE] A={A.shape}, B={B.shape}")

    # --- Scale to B for alignment ---
    scale = PX_PER_UM_B / PX_PER_UM_A
    inv_scale = 1.0 / scale
    A_scaled = nearest_resample_labels(A, scale)
    print(f"[SCALE] {scale:.4f}  A_scaled={A_scaled.shape}  B={B.shape}")

    # --- AND-overlap peak (coarse→refine) ---
    binA = to_binary(A_scaled)
    binB = to_binary(B)

    best_c = grid_search_peak(binA, binB, 0, 0, COARSE_RADIUS_YX, COARSE_STRIDE, "COARSE")
    best_r = grid_search_peak(binA, binB, best_c["dy"], best_c["dx"], REFINE_RADIUS_YX, REFINE_STRIDE, "REFINE")
    dy_Bpx, dx_Bpx, ov = best_r["dy"], best_r["dx"], best_r["ov"]
    print(f"[SELECT] dy_B={dy_Bpx}, dx_B={dx_Bpx}, overlap_px={ov}")

    # --- Place A on B-canvas (labels preserved) ---
    A_on_B = translate_labels_to_canvas(A_scaled, dy_Bpx, dx_Bpx, canvas_shape=B.shape)
    tiff.imwrite(os.path.join(OUT_DIR, "A_scaled_shifted_on_B.tif"), A_on_B.astype(np.int32))

    # --- Features in ORIGINAL scale (A,B) ---
    dfA = extract_features(A)
    dfB = extract_features(B)

    # --- Build centroids in both spaces ---
    Axy_A = dfA[["cx", "cy"]].to_numpy()
    Bxy_B = dfB[["cx", "cy"]].to_numpy()
    Bxy_A = Bxy_B * inv_scale
    Axy_B = Axy_A * scale + np.array([dx_Bpx, dy_Bpx])[None, :]

    # --- Translation vector in ALL units ---
    dy_Apx = dy_Bpx * inv_scale
    dx_Apx = dx_Bpx * inv_scale
    dy_um = dy_Bpx / PX_PER_UM_B
    dx_um = dx_Bpx / PX_PER_UM_B

    # --- Matching in ORIGINAL A-scale ---
    FINAL_MAX_DIST_A = FINAL_MAX_DIST_BPX * inv_scale
    UNMATCHED_PENALTY_A = UNMATCHED_PENALTY_B * inv_scale
    Axy_A_shifted = Axy_A + np.array([dx_Apx, dy_Apx])[None, :]

    matches, unA, unB = hungarian_with_unmatched(
        Axy_A_shifted, Bxy_A, FINAL_MAX_DIST_A, UNMATCHED_PENALTY_A
    )
    print(f"[MATCH] matches={len(matches)}  unA={len(unA)}  unB={len(unB)}  gate_Apx={FINAL_MAX_DIST_A:.1f}")

    # --- Export CSVs ---
    pairs_rows = []
    for iA, iB, dist_Apx in matches:
        labelA = int(dfA.iloc[iA]["label"])
        labelB = int(dfB.iloc[iB]["label"])

        Ax_A, Ay_A = float(dfA.iloc[iA]["cx"]), float(dfA.iloc[iA]["cy"])
        Bx_B, By_B = float(dfB.iloc[iB]["cx"]), float(dfB.iloc[iB]["cy"])

        Ax_B = Ax_A * scale + dx_Bpx
        Ay_B = Ay_A * scale + dy_Bpx
        Bx_A = Bx_B * inv_scale
        By_A = By_B * inv_scale

        dist_Bpx = np.hypot((Ax_B - Bx_B), (Ay_B - By_B))
        dist_um = dist_Bpx / PX_PER_UM_B
        iou = compute_iou_for_pair(A_on_B, B, labelA, labelB)

        pairs_rows.append(dict(
            # indices / labels
            idxA=iA, idxB=iB, labelA=labelA, labelB=labelB,

            # translation
            shift_dx_Bpx=dx_Bpx, shift_dy_Bpx=dy_Bpx,
            shift_dx_Apx=dx_Apx, shift_dy_Apx=dy_Apx,
            shift_dx_um=dx_um, shift_dy_um=dy_um,

            # coordinates (both scales)
            Ax_Apx=Ax_A, Ay_Apx=Ay_A,
            Ax_Bpx=Ax_B, Ay_Bpx=Ay_B,
            Bx_Bpx=Bx_B, By_Bpx=By_B,
            Bx_Apx=Bx_A, By_Apx=By_A,

            # distances
            dist_Apx=float(dist_Apx),
            dist_Bpx=float(dist_Bpx),
            dist_um=float(dist_um),

            # original features
            areaA=dfA.iloc[iA]["area"],
            areaB=dfB.iloc[iB]["area"],
            periA=dfA.iloc[iA]["perimeter"],
            periB=dfB.iloc[iB]["perimeter"],

            majorA=dfA.iloc[iA]["major_axis_length"],
            majorB=dfB.iloc[iB]["major_axis_length"],
            minorA=dfA.iloc[iA]["minor_axis_length"],
            minorB=dfB.iloc[iB]["minor_axis_length"],
            equiv_diamA=dfA.iloc[iA]["equivalent_diameter"],
            equiv_diamB=dfB.iloc[iB]["equivalent_diameter"],

            aspectA=dfA.iloc[iA]["aspect"],
            aspectB=dfB.iloc[iB]["aspect"],
            circA=dfA.iloc[iA]["circularity"],
            circB=dfB.iloc[iB]["circularity"],
            solidityA=dfA.iloc[iA]["solidity"],
            solidityB=dfB.iloc[iB]["solidity"],
            eccentricityA=dfA.iloc[iA]["eccentricity"],
            eccentricityB=dfB.iloc[iB]["eccentricity"],
            extentA=dfA.iloc[iA]["extent"],
            extentB=dfB.iloc[iB]["extent"],
            orientationA=dfA.iloc[iA]["orientation"],
            orientationB=dfB.iloc[iB]["orientation"],
            convex_areaA=dfA.iloc[iA]["convex_area"],
            convex_areaB=dfB.iloc[iB]["convex_area"],
            bbox_areaA=dfA.iloc[iA]["bbox_area"],
            bbox_areaB=dfB.iloc[iB]["bbox_area"],

            # physical area
            areaA_um2=dfA.iloc[iA]["area"] / (PX_PER_UM_A**2),
            areaB_um2=dfB.iloc[iB]["area"] / (PX_PER_UM_B**2),

            # quality
            iou_Bscale=float(iou)
        ))

    df_pairs = pd.DataFrame(pairs_rows)
    df_pairs.to_csv(os.path.join(OUT_DIR, "pairs.csv"), index=False)
    print("[Saved] pairs.csv")

    if len(unA) > 0:
        pd.DataFrame({
            "idxA": unA,
            "labelA": [int(dfA.iloc[i]["label"]) for i in unA]
        }).to_csv(os.path.join(OUT_DIR, "unmatched_A.csv"), index=False)

    if len(unB) > 0:
        pd.DataFrame({
            "idxB": unB,
            "labelB": [int(dfB.iloc[j]["label"]) for j in unB]
        }).to_csv(os.path.join(OUT_DIR, "unmatched_B.csv"), index=False)

    pd.DataFrame([dict(
        PX_PER_UM_A=PX_PER_UM_A, PX_PER_UM_B=PX_PER_UM_B, scale=scale,
        dy_Bpx=dy_Bpx, dx_Bpx=dx_Bpx, dy_Apx=dy_Apx, dx_Apx=dx_Apx,
        dy_um=dy_um, dx_um=dx_um,
        overlap_peak_px=ov,
        FINAL_MAX_DIST_BPX=FINAL_MAX_DIST_BPX,
        FINAL_MAX_DIST_Apx=FINAL_MAX_DIST_A,
        matches=len(matches), unmatched_A=len(unA), unmatched_B=len(unB),
        A_shape_H=A.shape[0], A_shape_W=A.shape[1],
        B_shape_H=B.shape[0], B_shape_W=B.shape[1],
    )]).to_csv(os.path.join(OUT_DIR, "summary.csv"), index=False)
    print("[Saved] summary.csv")

    # --- Basic plots ---
    if len(df_pairs) >= 2:
        scatter_and_corr(df_pairs["aspectA"],       df_pairs["aspectB"],       "aspect",        OUT_DIR)
        scatter_and_corr(df_pairs["circA"],         df_pairs["circB"],         "circularity",   OUT_DIR)
        scatter_and_corr(df_pairs["solidityA"],     df_pairs["solidityB"],     "solidity",      OUT_DIR)
        scatter_and_corr(df_pairs["eccentricityA"], df_pairs["eccentricityB"], "eccentricity",  OUT_DIR)
        scatter_and_corr(df_pairs["extentA"],       df_pairs["extentB"],       "extent",        OUT_DIR)

    # --- Visualization ---
    bgB = load_bg_image(IMG_B_PATH, (B.shape[0], B.shape[1]), invert=INVERT_VIEW)
    draw_vectors(
        bgB, Axy_B=Axy_B, Bxy_B=Bxy_B, matches=matches,
        upscale=UPSCALE_VIS, out_png=os.path.join(OUT_DIR, "overlay_vectors.png")
    )
    print(f"[Saved] {os.path.join(OUT_DIR, 'overlay_vectors.png')}")


if __name__ == "__main__":
    main()

[SHAPE] A=(1490, 2240), B=(3984, 6000)
[SCALE] 2.6800  A_scaled=(3993, 6003)  B=(3984, 6000)
[COARSE] tried=289  best_ov=1813956  shift=(-20,-20)
[REFINE] tried=3721  best_ov=1883360  shift=(0,-20)
[SELECT] dy_B=0, dx_B=-20, overlap_px=1883360
[MATCH] matches=210  unA=99  unB=44  gate_Apx=37.3
[Saved] pairs.csv
[Saved] summary.csv
[Saved] .\scatter_aspect.png
[Saved] .\scatter_circularity.png
[Saved] .\scatter_solidity.png
[Saved] .\scatter_eccentricity.png


<tifffile.TiffFile 'July_22_ROI_A_004_RGB.tif'> OME series cannot handle discontiguous storage ((3984, 6000, 3) != (3, 3984, 6000))


[Saved] .\scatter_extent.png
[Saved] .\overlay_vectors.png


In [20]:

import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from scipy.stats import mannwhitneyu, gaussian_kde

# =========================
# USER SETTINGS
# =========================
PAIR_CSV_DIR = r"C:\Users\oxfil\Pair_for_PCA"
GLOB_PATTERN = "*.csv"
OUT_DIR = r".\shape_pair_pca_analysis"

# 실제 네 컬럼명 기준
FEATURE_PAIRS = {
    "aspect": ("aspectA", "aspectB"),
    "circularity": ("circA", "circB"),
    "eccentricity": ("eccentricityA", "eccentricityB"),
    "extent": ("extentA", "extentB"),
}

# scaling
SCALER_TYPE = "standard"   # "standard" or "robust"

# random comparison
N_RANDOM_PER_REAL = 20
N_PERMUTATIONS = 2000
RANDOM_SEED = 42

# plotting
FIG_DPI = 300
POINT_SIZE = 14
POINT_ALPHA = 0.28
SAMPLED_PAIR_LINES = 120

USE_DENSITY_CONTOUR = True
CONTOUR_LEVELS = 5
CONTOUR_GRID_N = 200

# optional filtering
MIN_IOU = None      # 예: 0.05
MAX_DIST_UM = None  # 예: 20

# =========================


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def load_all_pairs(pair_dir, pattern):
    files = glob(os.path.join(pair_dir, pattern), recursive=True)
    if not files:
        raise FileNotFoundError(f"No pairs.csv found in: {pair_dir}")

    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["source_file"] = os.path.basename(f)
        df["source_path"] = f
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True)
    return data, files


def filter_pairs(df):
    out = df.copy()

    if MIN_IOU is not None and "iou_Bscale" in out.columns:
        out = out[out["iou_Bscale"] >= MIN_IOU].copy()

    if MAX_DIST_UM is not None and "dist_um" in out.columns:
        out = out[out["dist_um"] <= MAX_DIST_UM].copy()

    return out


def validate_columns(df, feature_pairs):
    missing = []
    for _, (ca, cb) in feature_pairs.items():
        if ca not in df.columns:
            missing.append(ca)
        if cb not in df.columns:
            missing.append(cb)
    if missing:
        raise ValueError(f"Missing columns: {missing}")


def prepare_feature_matrices(df, feature_pairs):
    feature_names = list(feature_pairs.keys())
    colsA = [feature_pairs[f][0] for f in feature_names]
    colsB = [feature_pairs[f][1] for f in feature_names]

    meta_cols = []
    for c in ["idxA", "idxB", "labelA", "labelB", "source_file", "source_path", "dist_um", "iou_Bscale"]:
        if c in df.columns:
            meta_cols.append(c)

    sub = df[colsA + colsB + meta_cols].copy()
    sub = sub.replace([np.inf, -np.inf], np.nan).dropna()

    A = sub[colsA].to_numpy(float)
    B = sub[colsB].to_numpy(float)

    A_df = pd.DataFrame(A, columns=feature_names, index=sub.index)
    B_df = pd.DataFrame(B, columns=feature_names, index=sub.index)

    return sub, A_df, B_df, feature_names


def get_scaler(kind="standard"):
    if kind.lower() == "robust":
        return RobustScaler()
    return StandardScaler()


def joint_normalize(A_df, B_df, scaler_type="standard"):
    scaler = get_scaler(scaler_type)
    pooled = pd.concat([A_df, B_df], axis=0).to_numpy(float)
    scaler.fit(pooled)

    ZA = scaler.transform(A_df.to_numpy(float))
    ZB = scaler.transform(B_df.to_numpy(float))
    return scaler, ZA, ZB


def run_pca(ZA, ZB, n_components=2):
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    pooled = np.vstack([ZA, ZB])
    proj = pca.fit_transform(pooled)
    PA = proj[:len(ZA)]
    PB = proj[len(ZA):]
    return pca, PA, PB


def euclidean_pair_distance(A, B):
    return np.sqrt(np.sum((A - B) ** 2, axis=1))


def random_pair_distances(ZA, ZB, n_random_per_real=20, seed=42):
    rng = np.random.default_rng(seed)
    n = len(ZA)
    out = []

    for i in range(n):
        idx = rng.integers(0, n, size=n_random_per_real)
        idx = np.where(idx == i, (idx + 1) % n, idx)
        d = np.sqrt(np.sum((ZA[i][None, :] - ZB[idx]) ** 2, axis=1))
        out.extend(d.tolist())

    return np.asarray(out, dtype=float)


def permutation_test(real_dist, ZA, ZB, n_perm=2000, seed=42):
    rng = np.random.default_rng(seed)
    observed = np.mean(real_dist)

    perm_means = np.zeros(n_perm, dtype=float)
    n = len(ZA)

    for k in range(n_perm):
        perm_idx = rng.permutation(n)
        d = np.sqrt(np.sum((ZA - ZB[perm_idx]) ** 2, axis=1))
        perm_means[k] = np.mean(d)

    p_emp = (np.sum(perm_means <= observed) + 1) / (n_perm + 1)
    return observed, perm_means, p_emp


def add_density_contour(ax, x, y, levels=5, grid_n=200, linewidth=1.2, alpha=0.9):
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 20:
        return

    values = np.vstack([x, y])
    kde = gaussian_kde(values)

    xmin, xmax = np.min(x), np.max(x)
    ymin, ymax = np.min(y), np.max(y)

    xpad = 0.08 * (xmax - xmin + 1e-12)
    ypad = 0.08 * (ymax - ymin + 1e-12)

    xx, yy = np.meshgrid(
        np.linspace(xmin - xpad, xmax + xpad, grid_n),
        np.linspace(ymin - ypad, ymax + ypad, grid_n)
    )
    coords = np.vstack([xx.ravel(), yy.ravel()])
    zz = kde(coords).reshape(xx.shape)

    zmin, zmax = np.nanmin(zz), np.nanmax(zz)
    levs = np.linspace(zmin + 0.2 * (zmax - zmin), zmax, levels)
    ax.contour(xx, yy, zz, levels=levs, linewidths=linewidth, alpha=alpha)


def plot_pca_overlay(PA, PB, explained, out_path):
    fig, ax = plt.subplots(figsize=(6.2, 5.4))

    ax.scatter(PA[:, 0], PA[:, 1], s=POINT_SIZE, alpha=POINT_ALPHA, label="Method A")
    ax.scatter(PB[:, 0], PB[:, 1], s=POINT_SIZE, alpha=POINT_ALPHA, label="Method B")

    if USE_DENSITY_CONTOUR:
        add_density_contour(ax, PA[:, 0], PA[:, 1], levels=CONTOUR_LEVELS, grid_n=CONTOUR_GRID_N)
        add_density_contour(ax, PB[:, 0], PB[:, 1], levels=CONTOUR_LEVELS, grid_n=CONTOUR_GRID_N)

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)


def plot_sampled_pair_lines(PA, PB, explained, out_path, sample_n=120, seed=42):
    rng = np.random.default_rng(seed)
    n = len(PA)
    sample_n = min(sample_n, n)
    idx = rng.choice(n, size=sample_n, replace=False)

    fig, ax = plt.subplots(figsize=(6.2, 5.4))
    ax.scatter(PA[:, 0], PA[:, 1], s=8, alpha=0.16, label="Method A")
    ax.scatter(PB[:, 0], PB[:, 1], s=8, alpha=0.16, label="Method B")

    for i in idx:
        ax.plot([PA[i, 0], PB[i, 0]], [PA[i, 1], PB[i, 1]], lw=0.7, alpha=0.6)

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)


def plot_real_vs_random(real_dist, rand_dist, p_mwu, p_emp, out_path):
    fig, ax = plt.subplots(figsize=(5.8, 5.0))

    data = [real_dist, rand_dist]
    labels = ["Real pair", "Random pair"]

    vp = ax.violinplot(data, showmeans=False, showmedians=True, showextrema=False)
    for body in vp["bodies"]:
        body.set_alpha(0.45)

    rng = np.random.default_rng(RANDOM_SEED)
    for i, arr in enumerate(data, start=1):
        x = rng.normal(i, 0.05, size=len(arr))
        ax.scatter(x, arr, s=6, alpha=0.08)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(labels)
    ax.set_ylabel("Distance in normalized shape space")
    ax.set_title(
        f"MWU p = {p_mwu:.3e}\nPermutation p = {p_emp:.3e}\n"
        f"Median(real) = {np.median(real_dist):.3f}, Median(random) = {np.median(rand_dist):.3f}",
        fontsize=10
    )

    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)


def plot_loadings(pca, feature_names, out_png, out_csv):
    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_names,
        columns=[f"PC{i+1}" for i in range(pca.components_.shape[0])]
    )
    loadings.to_csv(out_csv)

    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    x = np.arange(len(feature_names))
    ax.bar(x - 0.18, loadings["PC1"], width=0.36, label="PC1")
    ax.bar(x + 0.18, loadings["PC2"], width=0.36, label="PC2")
    ax.set_xticks(x)
    ax.set_xticklabels(feature_names, rotation=30, ha="right")
    ax.set_ylabel("Loading")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)


def save_pair_level(sub, PA, PB, real_dist, out_csv):
    out = sub.copy()
    out["A_PC1"] = PA[:, 0]
    out["A_PC2"] = PA[:, 1]
    out["B_PC1"] = PB[:, 0]
    out["B_PC2"] = PB[:, 1]
    out["shape_pair_distance"] = real_dist
    out.to_csv(out_csv, index=False)


def save_summary(real_dist, rand_dist, pca, p_mwu, p_emp, out_csv):
    summary = pd.DataFrame([{
        "n_real_pairs": len(real_dist),
        "n_random_pairs": len(rand_dist),
        "real_mean": float(np.mean(real_dist)),
        "real_median": float(np.median(real_dist)),
        "random_mean": float(np.mean(rand_dist)),
        "random_median": float(np.median(rand_dist)),
        "mannwhitney_p": float(p_mwu),
        "permutation_p": float(p_emp),
        "pc1_var_ratio": float(pca.explained_variance_ratio_[0]),
        "pc2_var_ratio": float(pca.explained_variance_ratio_[1]),
    }])
    summary.to_csv(out_csv, index=False)


def main():
    ensure_dir(OUT_DIR)

    df, files = load_all_pairs(PAIR_CSV_DIR, GLOB_PATTERN)
    print(f"Loaded files: {len(files)}")
    print(f"Rows before filtering: {len(df)}")

    validate_columns(df, FEATURE_PAIRS)
    df = filter_pairs(df)
    print(f"Rows after filtering: {len(df)}")

    sub, A_df, B_df, feature_names = prepare_feature_matrices(df, FEATURE_PAIRS)
    print(f"Valid rows for PCA/similarity: {len(sub)}")

    scaler, ZA, ZB = joint_normalize(A_df, B_df, scaler_type=SCALER_TYPE)
    pca, PA, PB = run_pca(ZA, ZB, n_components=2)

    real_dist = euclidean_pair_distance(ZA, ZB)
    rand_dist = random_pair_distances(
        ZA, ZB,
        n_random_per_real=N_RANDOM_PER_REAL,
        seed=RANDOM_SEED
    )

    _, p_mwu = mannwhitneyu(real_dist, rand_dist, alternative="less")
    _, perm_means, p_emp = permutation_test(
        real_dist, ZA, ZB,
        n_perm=N_PERMUTATIONS,
        seed=RANDOM_SEED
    )

    save_pair_level(
        sub, PA, PB, real_dist,
        os.path.join(OUT_DIR, "pair_level_shape_similarity.csv")
    )
    save_summary(
        real_dist, rand_dist, pca, p_mwu, p_emp,
        os.path.join(OUT_DIR, "summary_shape_similarity.csv")
    )

    plot_pca_overlay(
        PA, PB, pca.explained_variance_ratio_,
        os.path.join(OUT_DIR, "pca_overlay_A_vs_B.png")
    )
    plot_sampled_pair_lines(
        PA, PB, pca.explained_variance_ratio_,
        os.path.join(OUT_DIR, "pca_pairs_connected.png"),
        sample_n=SAMPLED_PAIR_LINES,
        seed=RANDOM_SEED
    )
    plot_real_vs_random(
        real_dist, rand_dist, p_mwu, p_emp,
        os.path.join(OUT_DIR, "real_vs_random_pair_distance.png")
    )
    plot_loadings(
        pca, feature_names,
        os.path.join(OUT_DIR, "pca_loadings.png"),
        os.path.join(OUT_DIR, "pca_loadings.csv")
    )

    print("Saved:")
    print(" - pair_level_shape_similarity.csv")
    print(" - summary_shape_similarity.csv")
    print(" - pca_overlay_A_vs_B.png")
    print(" - pca_pairs_connected.png")
    print(" - real_vs_random_pair_distance.png")
    print(" - pca_loadings.png")
    print(" - pca_loadings.csv")


if __name__ == "__main__":
    main()

Loaded files: 6
Rows before filtering: 1071
Rows after filtering: 1071
Valid rows for PCA/similarity: 1071
Saved:
 - pair_level_shape_similarity.csv
 - summary_shape_similarity.csv
 - pca_overlay_A_vs_B.png
 - pca_pairs_connected.png
 - real_vs_random_pair_distance.png
 - pca_loadings.png
 - pca_loadings.csv


In [4]:
# -*- coding: utf-8 -*-

import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from scipy.stats import mannwhitneyu, gaussian_kde

# =========================
# USER SETTINGS
# =========================
PAIR_CSV_DIR = r"C:\Users\oxfil\Pair_for_PCA"
GLOB_PATTERN = "*.csv"
OUT_DIR = r".\shape_pair_pca_analysis416"


# 실제 네 pairs.csv 컬럼명 기준
FEATURE_PAIRS = {
    "aspect": ("aspectA", "aspectB"),
    "circularity": ("circA", "circB"),
    "solidity": ("solidityA","solidityB"),
    "eccentricity": ("eccentricityA", "eccentricityB"),
    "extent": ("extentA", "extentB"),
}

# scaling
SCALER_TYPE = "standard"   # "standard" or "robust"

# filtering
MIN_IOU = None      # 예: 0.05
MAX_DIST_UM = 100 # 예: 20

# random / permutation
N_RANDOM_PER_REAL = 20
N_PERMUTATIONS = 2000
RANDOM_SEED = 80

# plotting
FIG_DPI = 300
POINT_SIZE = 16
POINT_ALPHA = 0.28
SAMPLED_PAIR_LINES = 120

USE_DENSITY_CONTOUR = True
CONTOUR_LEVELS = 5
CONTOUR_GRID_N = 200

# =========================
# GraphPad-like style
# =========================
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.linewidth": 1.2,
    "axes.grid": False,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COLOR_A = "#000080"
COLOR_B = "#028A0F"
COLOR_LINE = "#4D4D4D"
COLOR_REAL = "#4C78A8"
COLOR_RANDOM = "#B0B0B0"

# =========================


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def load_all_pairs(pair_dir, pattern):
    files = glob(os.path.join(pair_dir, pattern), recursive=True)
    if not files:
        raise FileNotFoundError(f"No pairs.csv found in: {pair_dir}")

    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["source_file"] = os.path.basename(f)
        df["source_path"] = f
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True)
    return data, files


def filter_pairs(df):
    out = df.copy()

    if MIN_IOU is not None and "iou_Bscale" in out.columns:
        out = out[out["iou_Bscale"] >= MIN_IOU].copy()

    if MAX_DIST_UM is not None and "dist_um" in out.columns:
        out = out[out["dist_um"] <= MAX_DIST_UM].copy()

    return out


def validate_columns(df, feature_pairs):
    missing = []
    for _, (ca, cb) in feature_pairs.items():
        if ca not in df.columns:
            missing.append(ca)
        if cb not in df.columns:
            missing.append(cb)
    if missing:
        raise ValueError(f"Missing columns: {missing}")


def prepare_feature_matrices(df, feature_pairs):
    feature_names = list(feature_pairs.keys())
    colsA = [feature_pairs[f][0] for f in feature_names]
    colsB = [feature_pairs[f][1] for f in feature_names]

    meta_cols = []
    for c in ["idxA", "idxB", "labelA", "labelB", "source_file", "source_path", "dist_um", "iou_Bscale"]:
        if c in df.columns:
            meta_cols.append(c)

    sub = df[colsA + colsB + meta_cols].copy()
    sub = sub.replace([np.inf, -np.inf], np.nan).dropna()

    A = sub[colsA].to_numpy(float)
    B = sub[colsB].to_numpy(float)

    A_df = pd.DataFrame(A, columns=feature_names, index=sub.index)
    B_df = pd.DataFrame(B, columns=feature_names, index=sub.index)

    return sub, A_df, B_df, feature_names


def get_scaler(kind="standard"):
    if kind.lower() == "robust":
        return RobustScaler()
    return StandardScaler()


def joint_normalize(A_df, B_df, scaler_type="standard"):
    scaler = get_scaler(scaler_type)
    pooled = pd.concat([A_df, B_df], axis=0).to_numpy(float)
    scaler.fit(pooled)

    ZA = scaler.transform(A_df.to_numpy(float))
    ZB = scaler.transform(B_df.to_numpy(float))
    return scaler, ZA, ZB


def run_pca(ZA, ZB, n_components=2):
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    pooled = np.vstack([ZA, ZB])
    proj = pca.fit_transform(pooled)
    PA = proj[:len(ZA)]
    PB = proj[len(ZA):]
    return pca, PA, PB


def euclidean_pair_distance(A, B):
    return np.sqrt(np.sum((A - B) ** 2, axis=1))


def random_pair_distances_equal_n(ZA, ZB, meta_df, seed=42):
    """
    real pair 수와 동일한 개수의 random one-to-one shuffled pair 생성
    """
    rng = np.random.default_rng(seed)
    n = len(ZA)

    perm = rng.permutation(n)

    # 너무 엄밀하게 하려면 자기 자신 pairing 회피
    same = perm == np.arange(n)
    if np.any(same):
        perm[same] = np.roll(perm[same], 1)

    rand_dist = np.sqrt(np.sum((ZA - ZB[perm]) ** 2, axis=1))

    rows = []
    for i in range(n):
        rows.append({
            "comparison_type": "Random pair",
            "distance": float(rand_dist[i]),
            "real_pair_index": int(i),
            "random_partner_index": int(perm[i]),
            "source_file_A": meta_df.iloc[i]["source_file"] if "source_file" in meta_df.columns else None,
            "source_file_B": meta_df.iloc[perm[i]]["source_file"] if "source_file" in meta_df.columns else None,
        })

    return rand_dist, pd.DataFrame(rows)


def permutation_test(real_dist, ZA, ZB, n_perm=2000, seed=42):
    rng = np.random.default_rng(seed)
    observed = np.mean(real_dist)

    perm_means = np.zeros(n_perm, dtype=float)
    n = len(ZA)

    for k in range(n_perm):
        perm_idx = rng.permutation(n)
        d = np.sqrt(np.sum((ZA - ZB[perm_idx]) ** 2, axis=1))
        perm_means[k] = np.mean(d)

    p_emp = (np.sum(perm_means <= observed) + 1) / (n_perm + 1)
    return observed, perm_means, p_emp


def add_density_contour(ax, x, y, color, levels=5, grid_n=200, linewidth=1.2, alpha=0.9):
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 20:
        return

    values = np.vstack([x, y])
    kde = gaussian_kde(values)

    xmin, xmax = np.min(x), np.max(x)
    ymin, ymax = np.min(y), np.max(y)

    xpad = 0.08 * (xmax - xmin + 1e-12)
    ypad = 0.08 * (ymax - ymin + 1e-12)

    xx, yy = np.meshgrid(
        np.linspace(xmin - xpad, xmax + xpad, grid_n),
        np.linspace(ymin - ypad, ymax + ypad, grid_n)
    )
    coords = np.vstack([xx.ravel(), yy.ravel()])
    zz = kde(coords).reshape(xx.shape)

    zmin, zmax = np.nanmin(zz), np.nanmax(zz)
    levs = np.linspace(zmin + 0.2 * (zmax - zmin), zmax, levels)
    ax.contour(
    xx, yy, zz,
    levels=levs,
    linewidths=linewidth,
    alpha=alpha,
    colors=[color],
    zorder=4
    )


def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", width=1.2, length=5)


def plot_pca_overlay(PA, PB, explained, out_path, seed=42):
    fig, ax = plt.subplots(figsize=(6.0, 5.2))

    # 1) contour는 각 조건별로 그대로 유지
    if USE_DENSITY_CONTOUR:
        add_density_contour(
            ax, PA[:, 0], PA[:, 1],
            color=COLOR_A,
            levels=CONTOUR_LEVELS,
            grid_n=CONTOUR_GRID_N,
            linewidth=0.9,
            alpha=0.45
        )
        add_density_contour(
            ax, PB[:, 0], PB[:, 1],
            color=COLOR_B,
            levels=CONTOUR_LEVELS,
            grid_n=CONTOUR_GRID_N,
            linewidth=0.9,
            alpha=0.45
        )

    # 2) scatter만 합쳐서 랜덤 순서로 그리기
    dfA = pd.DataFrame({
        "PC1": PA[:, 0],
        "PC2": PA[:, 1],
        "color": COLOR_A,
    })
    dfB = pd.DataFrame({
        "PC1": PB[:, 0],
        "PC2": PB[:, 1],
        "color": COLOR_B,
    })

    plot_df = pd.concat([dfA, dfB], ignore_index=True)
    plot_df = plot_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    for _, row in plot_df.iterrows():
        ax.scatter(
            row["PC1"], row["PC2"],
            s=POINT_SIZE,
            alpha=POINT_ALPHA,
            color=row["color"],
            edgecolors="none",
            zorder=2
        )

    # legend dummy handles
    ax.scatter([], [], s=POINT_SIZE*2, color=COLOR_A, alpha=0.7, label="Method A")
    ax.scatter([], [], s=POINT_SIZE*2, color=COLOR_B, alpha=0.7, label="Method B")

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
    ax.legend(frameon=False, fontsize=11)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_sampled_pair_lines(PA, PB, explained, out_path, sample_n=120, seed=42):
    rng = np.random.default_rng(seed)
    n = len(PA)
    sample_n = min(sample_n, n)
    idx = rng.choice(n, size=sample_n, replace=False)

    fig, ax = plt.subplots(figsize=(6.0, 5.2))
    ax.scatter(PA[:, 0], PA[:, 1], s=10, alpha=0.16, color=COLOR_A, label="Method A")
    ax.scatter(PB[:, 0], PB[:, 1], s=10, alpha=0.16, color=COLOR_B, label="Method B")

    for i in idx:
        ax.plot([PA[i, 0], PB[i, 0]], [PA[i, 1], PB[i, 1]], lw=0.7, alpha=0.55, color=COLOR_LINE)

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
    ax.legend(frameon=False, fontsize=12)
    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_real_vs_random_box(real_dist, rand_dist, p_mwu, p_emp, out_path):
    fig, ax = plt.subplots(figsize=(4.8, 5.4))

    data = [real_dist, rand_dist]
    positions = [1, 2]

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=0.45,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color="black", linewidth=1.6),
        whiskerprops=dict(color="black", linewidth=1.2),
        capprops=dict(color="black", linewidth=1.2),
        boxprops=dict(linewidth=1.2, color="black"),
    )

    bp["boxes"][0].set_facecolor(COLOR_REAL)
    bp["boxes"][0].set_alpha(0.45)
    bp["boxes"][1].set_facecolor(COLOR_RANDOM)
    bp["boxes"][1].set_alpha(0.65)

    rng = np.random.default_rng(RANDOM_SEED)
    for pos, arr, color in zip(positions, data, [COLOR_REAL, COLOR_RANDOM]):
        x = rng.normal(pos, 0.06, size=len(arr))
        ax.scatter(x, arr, s=7, alpha=0.10, color=color, edgecolors="none")

    y_max = max(np.max(real_dist), np.max(rand_dist))
    y_line = y_max * 1.08
    y_text = y_max * 1.13

    ax.plot([1, 1, 2, 2], [y_line*0.99, y_line, y_line, y_line*0.99], color="black", lw=1.0)
    ax.text(1.5, y_text, f"MWU p = {p_mwu:.2e}\nPerm p = {p_emp:.2e}",
            ha="center", va="bottom", fontsize=10)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Real pair", "Random pair"], fontsize=12)
    ax.set_ylabel("Distance in normalized shape space", fontsize=14)
    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_loadings(pca, feature_names, out_pdf, out_csv):
    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_names,
        columns=[f"PC{i+1}" for i in range(pca.components_.shape[0])]
    )
    loadings.to_csv(out_csv, index=True)

    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    x = np.arange(len(feature_names))

    ax.bar(x - 0.18, loadings["PC1"], width=0.36, color=COLOR_A, alpha=0.8, label="PC1")
    ax.bar(x + 0.18, loadings["PC2"], width=0.36, color=COLOR_B, alpha=0.8, label="PC2")

    ax.set_xticks(x)
    ax.set_xticklabels(feature_names, rotation=30, ha="right")
    ax.set_ylabel("Loading", fontsize=14)
    ax.legend(frameon=False, fontsize=11)
    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_pdf, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def save_pair_level(sub, PA, PB, real_dist, out_csv):
    out = sub.copy()
    out["A_PC1"] = PA[:, 0]
    out["A_PC2"] = PA[:, 1]
    out["B_PC1"] = PB[:, 0]
    out["B_PC2"] = PB[:, 1]
    out["shape_pair_distance"] = real_dist
    out.to_csv(out_csv, index=False)


def save_summary(real_dist, rand_dist, pca, p_mwu, p_emp, out_csv):
    summary = pd.DataFrame([{
        "n_real_pairs": len(real_dist),
        "n_random_pairs": len(rand_dist),
        "real_mean": float(np.mean(real_dist)),
        "real_median": float(np.median(real_dist)),
        "random_mean": float(np.mean(rand_dist)),
        "random_median": float(np.median(rand_dist)),
        "mannwhitney_p": float(p_mwu),
        "permutation_p": float(p_emp),
        "pc1_var_ratio": float(pca.explained_variance_ratio_[0]),
        "pc2_var_ratio": float(pca.explained_variance_ratio_[1]),
    }])
    summary.to_csv(out_csv, index=False)


def save_spss_pair_distance_long(sub, real_dist, random_long_df, out_csv):
    """
    SPSS에서 바로 비교 가능한 long-format:
    comparison_type / distance / source_file / idxA / idxB ...
    """
    real_rows = []
    for i in range(len(sub)):
        row = {
            "comparison_type": "Real pair",
            "distance": float(real_dist[i]),
            "pair_row_index": int(i),
        }
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                row[c] = sub.iloc[i][c]
        real_rows.append(row)

    df_real = pd.DataFrame(real_rows)
    df_long = pd.concat([df_real, random_long_df], ignore_index=True)
    df_long.to_csv(out_csv, index=False)


def save_spss_pca_scores_long(sub, PA, PB, out_csv):
    """
    SPSS에서 modality factor로 반복 없이 바로 다룰 수 있는 long-format PCA scores
    """
    rows = []
    for i in range(len(sub)):
        meta = {}
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                meta[c] = sub.iloc[i][c]

        rows.append({
            **meta,
            "pair_row_index": int(i),
            "method": "A",
            "PC1": float(PA[i, 0]),
            "PC2": float(PA[i, 1]),
        })
        rows.append({
            **meta,
            "pair_row_index": int(i),
            "method": "B",
            "PC1": float(PB[i, 0]),
            "PC2": float(PB[i, 1]),
        })

    pd.DataFrame(rows).to_csv(out_csv, index=False)

def plot_real_vs_random_violin(real_dist, rand_dist, p_mwu, p_emp, out_path):
    fig, ax = plt.subplots(figsize=(4.8, 5.4))

    data = [real_dist, rand_dist]
    positions = [1, 2]

    vp = ax.violinplot(
        data,
        positions=positions,
        widths=0.7,
        showmeans=False,
        showmedians=False,
        showextrema=False
    )

    # body color
    for body, color, alpha in zip(vp["bodies"], [COLOR_REAL, COLOR_RANDOM], [0.55, 0.65]):
        body.set_facecolor(color)
        body.set_edgecolor("black")
        body.set_linewidth(1.0)
        body.set_alpha(alpha)

    # median + IQR manually
    for pos, arr in zip(positions, data):
        q1, med, q3 = np.percentile(arr, [25, 50, 75])
        ax.plot([pos, pos], [q1, q3], color="black", lw=2.0, zorder=3)
        ax.plot([pos - 0.12, pos + 0.12], [med, med], color="black", lw=2.0, zorder=4)

    y_max = max(np.max(real_dist), np.max(rand_dist))
    y_line = y_max * 1.08
    y_text = y_max * 1.13

    ax.plot([1, 1, 2, 2], [y_line*0.99, y_line, y_line, y_line*0.99], color="black", lw=1.0)
    ax.text(
        1.5, y_text,
        f"MWU p = {p_mwu:.2e}\nPerm p = {p_emp:.2e}",
        ha="center", va="bottom", fontsize=10
    )

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Real pair", "Random pair"], fontsize=12)
    ax.set_ylabel("Distance in normalized shape space", fontsize=14)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)
    
def main():
    ensure_dir(OUT_DIR)

    df, files = load_all_pairs(PAIR_CSV_DIR, GLOB_PATTERN)
    print(f"Loaded files: {len(files)}")
    print(f"Rows before filtering: {len(df)}")

    validate_columns(df, FEATURE_PAIRS)
    df = filter_pairs(df)
    print(f"Rows after filtering: {len(df)}")

    sub, A_df, B_df, feature_names = prepare_feature_matrices(df, FEATURE_PAIRS)
    print(f"Valid rows for PCA/similarity: {len(sub)}")

    scaler, ZA, ZB = joint_normalize(A_df, B_df, scaler_type=SCALER_TYPE)
    pca, PA, PB = run_pca(ZA, ZB, n_components=2)

    real_dist = euclidean_pair_distance(ZA, ZB)
    rand_dist, random_long_df = random_pair_distances_equal_n(
        ZA, ZB, sub,
        seed=RANDOM_SEED
    )

    _, p_mwu = mannwhitneyu(real_dist, rand_dist, alternative="less")
    _, perm_means, p_emp = permutation_test(
        real_dist, ZA, ZB,
        n_perm=N_PERMUTATIONS,
        seed=RANDOM_SEED
    )

    # Main result tables
    save_pair_level(
        sub, PA, PB, real_dist,
        os.path.join(OUT_DIR, "pair_level_shape_similarity.csv")
    )
    save_summary(
        real_dist, rand_dist, pca, p_mwu, p_emp,
        os.path.join(OUT_DIR, "summary_shape_similarity.csv")
    )

    # SPSS-ready tables
    save_spss_pair_distance_long(
        sub, real_dist, random_long_df,
        os.path.join(OUT_DIR, "spss_pair_distance_long.csv")
    )
    save_spss_pca_scores_long(
        sub, PA, PB,
        os.path.join(OUT_DIR, "spss_pca_scores_long.csv")
    )

    # Figures: all PDF
    plot_pca_overlay(
        PA, PB, pca.explained_variance_ratio_,
        os.path.join(OUT_DIR, "pca_overlay_A_vs_B.pdf")
    )
    plot_sampled_pair_lines(
        PA, PB, pca.explained_variance_ratio_,
        os.path.join(OUT_DIR, "pca_pairs_connected.pdf"),
        sample_n=SAMPLED_PAIR_LINES,
        seed=RANDOM_SEED
    )
    plot_real_vs_random_box(
        real_dist, rand_dist, p_mwu, p_emp,
        os.path.join(OUT_DIR, "real_vs_random_pair_distance_box.pdf")
    )
    plot_loadings(
        pca, feature_names,
        os.path.join(OUT_DIR, "pca_loadings.pdf"),
        os.path.join(OUT_DIR, "pca_loadings.csv")
    )
    
    plot_real_vs_random_violin(
    real_dist, rand_dist, p_mwu, p_emp,
    os.path.join(OUT_DIR, "real_vs_random_pair_distance_violin.pdf")
    )

    print("Saved:")
    print(" - pair_level_shape_similarity.csv")
    print(" - summary_shape_similarity.csv")
    print(" - spss_pair_distance_long.csv")
    print(" - spss_pca_scores_long.csv")
    print(" - pca_overlay_A_vs_B.pdf")
    print(" - pca_pairs_connected.pdf")
    print(" - real_vs_random_pair_distance_box.pdf")
    print(" - pca_loadings.pdf")
    print(" - pca_loadings.csv")


if __name__ == "__main__":
    main()

Loaded files: 6
Rows before filtering: 1071
Rows after filtering: 1071
Valid rows for PCA/similarity: 1071
Saved:
 - pair_level_shape_similarity.csv
 - summary_shape_similarity.csv
 - spss_pair_distance_long.csv
 - spss_pca_scores_long.csv
 - pca_overlay_A_vs_B.pdf
 - pca_pairs_connected.pdf
 - real_vs_random_pair_distance_box.pdf
 - pca_loadings.pdf
 - pca_loadings.csv


In [6]:
# -*- coding: utf-8 -*-

import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from scipy.stats import mannwhitneyu, gaussian_kde, wilcoxon

# =========================
# USER SETTINGS
# =========================
PAIR_CSV_DIR = r"C:\Users\oxfil\Pair_for_PCA"
GLOB_PATTERN = "*.csv"
OUT_DIR = r".\shape_pair_pca_analysis416"

# 실제 네 pairs.csv 컬럼명 기준
FEATURE_PAIRS = {
    "aspect": ("aspectA", "aspectB"),
    "circularity": ("circA", "circB"),
    "solidity": ("solidityA", "solidityB"),
    "eccentricity": ("eccentricityA", "eccentricityB"),
    "extent": ("extentA", "extentB"),
}

# scaling
SCALER_TYPE = "standard"   # "standard" or "robust"

# filtering
MIN_IOU = None
MAX_DIST_UM = 100

# random / permutation
N_RANDOM_PER_REAL = 20
N_PERMUTATIONS = 2000
RANDOM_SEED = 80

# plotting
FIG_DPI = 300
POINT_SIZE = 16
POINT_ALPHA = 0.28
SAMPLED_PAIR_LINES = 120

USE_DENSITY_CONTOUR = True
CONTOUR_LEVELS = 5
CONTOUR_GRID_N = 200

# =========================
# GraphPad-like style
# =========================
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.linewidth": 1.2,
    "axes.grid": False,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COLOR_A = "#000080"
COLOR_B = "#028A0F"
COLOR_LINE = "#4D4D4D"
COLOR_REAL = "#4C78A8"
COLOR_RANDOM = "#B0B0B0"


# =========================
# Basic utilities
# =========================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def load_all_pairs(pair_dir, pattern):
    files = glob(os.path.join(pair_dir, pattern), recursive=True)
    if not files:
        raise FileNotFoundError(f"No pairs.csv found in: {pair_dir}")

    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["source_file"] = os.path.basename(f)
        df["source_path"] = f
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True)
    return data, files


def filter_pairs(df):
    out = df.copy()

    if MIN_IOU is not None and "iou_Bscale" in out.columns:
        out = out[out["iou_Bscale"] >= MIN_IOU].copy()

    if MAX_DIST_UM is not None and "dist_um" in out.columns:
        out = out[out["dist_um"] <= MAX_DIST_UM].copy()

    return out


def validate_columns(df, feature_pairs):
    missing = []
    for _, (ca, cb) in feature_pairs.items():
        if ca not in df.columns:
            missing.append(ca)
        if cb not in df.columns:
            missing.append(cb)
    if missing:
        raise ValueError(f"Missing columns: {missing}")


def prepare_feature_matrices(df, feature_pairs):
    feature_names = list(feature_pairs.keys())
    colsA = [feature_pairs[f][0] for f in feature_names]
    colsB = [feature_pairs[f][1] for f in feature_names]

    meta_cols = []
    for c in ["idxA", "idxB", "labelA", "labelB", "source_file", "source_path", "dist_um", "iou_Bscale"]:
        if c in df.columns:
            meta_cols.append(c)

    sub = df[colsA + colsB + meta_cols].copy()
    sub = sub.replace([np.inf, -np.inf], np.nan).dropna()

    A = sub[colsA].to_numpy(float)
    B = sub[colsB].to_numpy(float)

    A_df = pd.DataFrame(A, columns=feature_names, index=sub.index)
    B_df = pd.DataFrame(B, columns=feature_names, index=sub.index)

    return sub, A_df, B_df, feature_names


def get_scaler(kind="standard"):
    if kind.lower() == "robust":
        return RobustScaler()
    return StandardScaler()


def joint_normalize(A_df, B_df, scaler_type="standard"):
    scaler = get_scaler(scaler_type)
    pooled = pd.concat([A_df, B_df], axis=0).to_numpy(float)
    scaler.fit(pooled)

    ZA = scaler.transform(A_df.to_numpy(float))
    ZB = scaler.transform(B_df.to_numpy(float))
    return scaler, ZA, ZB


def run_pca(ZA, ZB, n_components=2):
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    pooled = np.vstack([ZA, ZB])
    proj = pca.fit_transform(pooled)
    PA = proj[:len(ZA)]
    PB = proj[len(ZA):]
    return pca, PA, PB


def euclidean_pair_distance(A, B):
    return np.sqrt(np.sum((A - B) ** 2, axis=1))


def random_pair_distances_equal_n(ZA, ZB, meta_df, seed=42):
    rng = np.random.default_rng(seed)
    n = len(ZA)

    perm = rng.permutation(n)

    same = perm == np.arange(n)
    if np.any(same):
        perm[same] = np.roll(perm[same], 1)

    rand_dist = np.sqrt(np.sum((ZA - ZB[perm]) ** 2, axis=1))

    rows = []
    for i in range(n):
        rows.append({
            "comparison_type": "Random pair",
            "distance": float(rand_dist[i]),
            "real_pair_index": int(i),
            "random_partner_index": int(perm[i]),
            "source_file_A": meta_df.iloc[i]["source_file"] if "source_file" in meta_df.columns else None,
            "source_file_B": meta_df.iloc[perm[i]]["source_file"] if "source_file" in meta_df.columns else None,
        })

    return rand_dist, pd.DataFrame(rows)


def permutation_test(real_dist, ZA, ZB, n_perm=2000, seed=42):
    rng = np.random.default_rng(seed)
    observed = np.mean(real_dist)

    perm_means = np.zeros(n_perm, dtype=float)
    n = len(ZA)

    for k in range(n_perm):
        perm_idx = rng.permutation(n)
        d = np.sqrt(np.sum((ZA - ZB[perm_idx]) ** 2, axis=1))
        perm_means[k] = np.mean(d)

    p_emp = (np.sum(perm_means <= observed) + 1) / (n_perm + 1)
    return observed, perm_means, p_emp


# =========================
# Plot utilities
# =========================
def add_density_contour(ax, x, y, color, levels=5, grid_n=200, linewidth=1.2, alpha=0.9):
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 20:
        return

    values = np.vstack([x, y])
    kde = gaussian_kde(values)

    xmin, xmax = np.min(x), np.max(x)
    ymin, ymax = np.min(y), np.max(y)

    xpad = 0.08 * (xmax - xmin + 1e-12)
    ypad = 0.08 * (ymax - ymin + 1e-12)

    xx, yy = np.meshgrid(
        np.linspace(xmin - xpad, xmax + xpad, grid_n),
        np.linspace(ymin - ypad, ymax + ypad, grid_n)
    )
    coords = np.vstack([xx.ravel(), yy.ravel()])
    zz = kde(coords).reshape(xx.shape)

    zmin, zmax = np.nanmin(zz), np.nanmax(zz)
    levs = np.linspace(zmin + 0.2 * (zmax - zmin), zmax, levels)

    ax.contour(
        xx, yy, zz,
        levels=levs,
        linewidths=linewidth,
        alpha=alpha,
        colors=[color],
        zorder=4
    )


def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", width=1.2, length=5)


def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    return "ns"


# =========================
# Main PCA plots
# =========================
def plot_pca_overlay(PA, PB, explained, out_path, seed=42):
    fig, ax = plt.subplots(figsize=(6.0, 5.2))

    if USE_DENSITY_CONTOUR:
        add_density_contour(
            ax, PA[:, 0], PA[:, 1],
            color=COLOR_A,
            levels=CONTOUR_LEVELS,
            grid_n=CONTOUR_GRID_N,
            linewidth=0.9,
            alpha=0.45
        )
        add_density_contour(
            ax, PB[:, 0], PB[:, 1],
            color=COLOR_B,
            levels=CONTOUR_LEVELS,
            grid_n=CONTOUR_GRID_N,
            linewidth=0.9,
            alpha=0.45
        )

    dfA = pd.DataFrame({
        "PC1": PA[:, 0],
        "PC2": PA[:, 1],
        "color": COLOR_A,
    })
    dfB = pd.DataFrame({
        "PC1": PB[:, 0],
        "PC2": PB[:, 1],
        "color": COLOR_B,
    })

    plot_df = pd.concat([dfA, dfB], ignore_index=True)
    plot_df = plot_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    for _, row in plot_df.iterrows():
        ax.scatter(
            row["PC1"], row["PC2"],
            s=POINT_SIZE,
            alpha=POINT_ALPHA,
            color=row["color"],
            edgecolors="none",
            zorder=2
        )

    ax.scatter([], [], s=POINT_SIZE * 2, color=COLOR_A, alpha=0.7, label="Method A")
    ax.scatter([], [], s=POINT_SIZE * 2, color=COLOR_B, alpha=0.7, label="Method B")

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
    ax.legend(frameon=False, fontsize=11)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_sampled_pair_lines(PA, PB, explained, out_path, sample_n=120, seed=42):
    rng = np.random.default_rng(seed)
    n = len(PA)
    sample_n = min(sample_n, n)
    idx = rng.choice(n, size=sample_n, replace=False)

    fig, ax = plt.subplots(figsize=(6.0, 5.2))
    ax.scatter(PA[:, 0], PA[:, 1], s=10, alpha=0.16, color=COLOR_A, label="Method A")
    ax.scatter(PB[:, 0], PB[:, 1], s=10, alpha=0.16, color=COLOR_B, label="Method B")

    for i in idx:
        ax.plot([PA[i, 0], PB[i, 0]], [PA[i, 1], PB[i, 1]], lw=0.7, alpha=0.55, color=COLOR_LINE)

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
    ax.legend(frameon=False, fontsize=12)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_real_vs_random_box(real_dist, rand_dist, p_mwu, p_emp, out_path):
    fig, ax = plt.subplots(figsize=(4.8, 5.4))

    data = [real_dist, rand_dist]
    positions = [1, 2]

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=0.45,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color="black", linewidth=1.6),
        whiskerprops=dict(color="black", linewidth=1.2),
        capprops=dict(color="black", linewidth=1.2),
        boxprops=dict(linewidth=1.2, color="black"),
    )

    bp["boxes"][0].set_facecolor(COLOR_REAL)
    bp["boxes"][0].set_alpha(0.45)
    bp["boxes"][1].set_facecolor(COLOR_RANDOM)
    bp["boxes"][1].set_alpha(0.65)

    rng = np.random.default_rng(RANDOM_SEED)
    for pos, arr, color in zip(positions, data, [COLOR_REAL, COLOR_RANDOM]):
        x = rng.normal(pos, 0.06, size=len(arr))
        ax.scatter(x, arr, s=7, alpha=0.10, color=color, edgecolors="none")

    y_max = max(np.max(real_dist), np.max(rand_dist))
    y_line = y_max * 1.08
    y_text = y_max * 1.13

    ax.plot([1, 1, 2, 2], [y_line*0.99, y_line, y_line, y_line*0.99], color="black", lw=1.0)
    ax.text(1.5, y_text, f"MWU p = {p_mwu:.2e}\nPerm p = {p_emp:.2e}",
            ha="center", va="bottom", fontsize=10)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Real pair", "Random pair"], fontsize=12)
    ax.set_ylabel("Distance in normalized shape space", fontsize=14)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_real_vs_random_violin(real_dist, rand_dist, p_mwu, p_emp, out_path):
    fig, ax = plt.subplots(figsize=(4.8, 5.4))

    data = [real_dist, rand_dist]
    positions = [1, 2]

    vp = ax.violinplot(
        data,
        positions=positions,
        widths=0.7,
        showmeans=False,
        showmedians=False,
        showextrema=False
    )

    for body, color, alpha in zip(vp["bodies"], [COLOR_REAL, COLOR_RANDOM], [0.55, 0.65]):
        body.set_facecolor(color)
        body.set_edgecolor("black")
        body.set_linewidth(1.0)
        body.set_alpha(alpha)

    for pos, arr in zip(positions, data):
        q1, med, q3 = np.percentile(arr, [25, 50, 75])
        ax.plot([pos, pos], [q1, q3], color="black", lw=2.0, zorder=3)
        ax.plot([pos - 0.12, pos + 0.12], [med, med], color="black", lw=2.0, zorder=4)

    y_max = max(np.max(real_dist), np.max(rand_dist))
    y_line = y_max * 1.08
    y_text = y_max * 1.13

    ax.plot([1, 1, 2, 2], [y_line*0.99, y_line, y_line, y_line*0.99], color="black", lw=1.0)
    ax.text(
        1.5, y_text,
        f"MWU p = {p_mwu:.2e}\nPerm p = {p_emp:.2e}",
        ha="center", va="bottom", fontsize=10
    )

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Real pair", "Random pair"], fontsize=12)
    ax.set_ylabel("Distance in normalized shape space", fontsize=14)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_loadings(pca, feature_names, out_pdf, out_csv):
    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_names,
        columns=[f"PC{i+1}" for i in range(pca.components_.shape[0])]
    )
    loadings.to_csv(out_csv, index=True)

    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    x = np.arange(len(feature_names))

    ax.bar(x - 0.18, loadings["PC1"], width=0.36, color=COLOR_A, alpha=0.8, label="PC1")
    ax.bar(x + 0.18, loadings["PC2"], width=0.36, color=COLOR_B, alpha=0.8, label="PC2")

    ax.set_xticks(x)
    ax.set_xticklabels(feature_names, rotation=30, ha="right")
    ax.set_ylabel("Loading", fontsize=14)
    ax.legend(frameon=False, fontsize=11)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_pdf, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


# =========================
# Per-feature paired comparison
# =========================
def save_featurewise_paired_stats(sub, feature_pairs, out_csv):
    rows = []

    for feat, (colA, colB) in feature_pairs.items():
        x = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)
        y = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)

        valid = np.isfinite(x) & np.isfinite(y)
        x = x[valid]
        y = y[valid]

        if len(x) == 0:
            rows.append({
                "feature": feat,
                "n": 0,
                "A_mean": np.nan,
                "B_mean": np.nan,
                "A_median": np.nan,
                "B_median": np.nan,
                "median_diff_A_minus_B": np.nan,
                "mean_diff_A_minus_B": np.nan,
                "wilcoxon_stat": np.nan,
                "wilcoxon_p": np.nan,
            })
            continue

        d = x - y

        if np.allclose(d, 0):
            stat = 0.0
            pval = 1.0
        else:
            try:
                stat, pval = wilcoxon(x, y, alternative="two-sided", zero_method="wilcox")
            except ValueError:
                stat, pval = np.nan, np.nan

        rows.append({
            "feature": feat,
            "n": int(len(x)),
            "A_mean": float(np.mean(x)),
            "B_mean": float(np.mean(y)),
            "A_median": float(np.median(x)),
            "B_median": float(np.median(y)),
            "median_diff_A_minus_B": float(np.median(d)),
            "mean_diff_A_minus_B": float(np.mean(d)),
            "wilcoxon_stat": float(stat) if np.isfinite(stat) else np.nan,
            "wilcoxon_p": float(pval) if np.isfinite(pval) else np.nan,
        })

    pd.DataFrame(rows).to_csv(out_csv, index=False)


def save_featurewise_long_for_spss(sub, feature_pairs, out_csv):
    rows = []
    for i in range(len(sub)):
        meta = {}
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                meta[c] = sub.iloc[i][c]

        for feat, (colA, colB) in feature_pairs.items():
            rows.append({
                **meta,
                "pair_row_index": int(i),
                "feature": feat,
                "method": "A",
                "value": sub.iloc[i][colA]
            })
            rows.append({
                **meta,
                "pair_row_index": int(i),
                "feature": feat,
                "method": "B",
                "value": sub.iloc[i][colB]
            })

    pd.DataFrame(rows).to_csv(out_csv, index=False)


def plot_featurewise_paired_boxplots(sub, feature_pairs, out_dir):
    rng = np.random.default_rng(RANDOM_SEED)

    for feat, (colA, colB) in feature_pairs.items():
        x = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)
        y = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)

        valid = np.isfinite(x) & np.isfinite(y)
        x = x[valid]
        y = y[valid]

        if len(x) == 0:
            continue

        d = x - y
        if np.allclose(d, 0):
            stat = 0.0
            pval = 1.0
        else:
            try:
                stat, pval = wilcoxon(x, y, alternative="two-sided", zero_method="wilcox")
            except ValueError:
                stat, pval = np.nan, np.nan

        fig, ax = plt.subplots(figsize=(4.8, 5.6))

        bp = ax.boxplot(
            [x, y],
            positions=[1, 2],
            widths=0.5,
            patch_artist=True,
            showfliers=False,
            medianprops=dict(color="black", linewidth=1.6),
            whiskerprops=dict(color="black", linewidth=1.2),
            capprops=dict(color="black", linewidth=1.2),
            boxprops=dict(linewidth=1.2, color="black"),
        )

        bp["boxes"][0].set_facecolor(COLOR_A)
        bp["boxes"][0].set_alpha(0.45)
        bp["boxes"][1].set_facecolor(COLOR_B)
        bp["boxes"][1].set_alpha(0.45)

        n = len(x)
        draw_n = n if n <= 1500 else 1500
        idx = rng.choice(n, size=draw_n, replace=False)

        for i in idx:
            x1 = 1 + rng.normal(0, 0.03)
            x2 = 2 + rng.normal(0, 0.03)
            ax.plot([x1, x2], [x[i], y[i]], color="0.6", alpha=0.15, lw=0.6, zorder=1)

        ax.scatter(1 + rng.normal(0, 0.04, size=n), x, s=10, alpha=0.25, color=COLOR_A, edgecolors="none", zorder=2)
        ax.scatter(2 + rng.normal(0, 0.04, size=n), y, s=10, alpha=0.25, color=COLOR_B, edgecolors="none", zorder=2)

        ymax = np.nanmax(np.r_[x, y])
        ymin = np.nanmin(np.r_[x, y])
        yr = ymax - ymin if ymax > ymin else 1.0

        yline = ymax + 0.08 * yr
        ytext = ymax + 0.13 * yr

        ax.plot([1, 1, 2, 2], [yline - 0.01 * yr, yline, yline, yline - 0.01 * yr], color="black", lw=1.0)

        if np.isfinite(pval):
            ax.text(
                1.5, ytext,
                f"Wilcoxon p = {pval:.2e}\n{p_to_stars(pval)}",
                ha="center", va="bottom", fontsize=10
            )
        else:
            ax.text(
                1.5, ytext,
                "Wilcoxon p = NA",
                ha="center", va="bottom", fontsize=10
            )

        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Method A", "Method B"], fontsize=12)
        ax.set_ylabel(feat, fontsize=14)

        style_axes(ax)
        fig.tight_layout()
        fig.savefig(
            os.path.join(out_dir, f"paired_boxplot_{feat}_A_vs_B.pdf"),
            dpi=FIG_DPI,
            bbox_inches="tight",
            transparent=True
        )
        plt.close(fig)


# =========================
# Save tables
# =========================
def save_pair_level(sub, PA, PB, real_dist, out_csv):
    out = sub.copy()
    out["A_PC1"] = PA[:, 0]
    out["A_PC2"] = PA[:, 1]
    out["B_PC1"] = PB[:, 0]
    out["B_PC2"] = PB[:, 1]
    out["shape_pair_distance"] = real_dist
    out.to_csv(out_csv, index=False)


def save_summary(real_dist, rand_dist, pca, p_mwu, p_emp, out_csv):
    summary = pd.DataFrame([{
        "n_real_pairs": len(real_dist),
        "n_random_pairs": len(rand_dist),
        "real_mean": float(np.mean(real_dist)),
        "real_median": float(np.median(real_dist)),
        "random_mean": float(np.mean(rand_dist)),
        "random_median": float(np.median(rand_dist)),
        "mannwhitney_p": float(p_mwu),
        "permutation_p": float(p_emp),
        "pc1_var_ratio": float(pca.explained_variance_ratio_[0]),
        "pc2_var_ratio": float(pca.explained_variance_ratio_[1]),
    }])
    summary.to_csv(out_csv, index=False)


def save_spss_pair_distance_long(sub, real_dist, random_long_df, out_csv):
    real_rows = []
    for i in range(len(sub)):
        row = {
            "comparison_type": "Real pair",
            "distance": float(real_dist[i]),
            "pair_row_index": int(i),
        }
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                row[c] = sub.iloc[i][c]
        real_rows.append(row)

    df_real = pd.DataFrame(real_rows)
    df_long = pd.concat([df_real, random_long_df], ignore_index=True)
    df_long.to_csv(out_csv, index=False)


def save_spss_pca_scores_long(sub, PA, PB, out_csv):
    rows = []
    for i in range(len(sub)):
        meta = {}
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                meta[c] = sub.iloc[i][c]

        rows.append({
            **meta,
            "pair_row_index": int(i),
            "method": "A",
            "PC1": float(PA[i, 0]),
            "PC2": float(PA[i, 1]),
        })
        rows.append({
            **meta,
            "pair_row_index": int(i),
            "method": "B",
            "PC1": float(PB[i, 0]),
            "PC2": float(PB[i, 1]),
        })

    pd.DataFrame(rows).to_csv(out_csv, index=False)


# =========================
# Main
# =========================
def main():
    ensure_dir(OUT_DIR)

    df, files = load_all_pairs(PAIR_CSV_DIR, GLOB_PATTERN)
    print(f"Loaded files: {len(files)}")
    print(f"Rows before filtering: {len(df)}")

    validate_columns(df, FEATURE_PAIRS)
    df = filter_pairs(df)
    print(f"Rows after filtering: {len(df)}")

    sub, A_df, B_df, feature_names = prepare_feature_matrices(df, FEATURE_PAIRS)
    print(f"Valid rows for PCA/similarity: {len(sub)}")

    scaler, ZA, ZB = joint_normalize(A_df, B_df, scaler_type=SCALER_TYPE)
    pca, PA, PB = run_pca(ZA, ZB, n_components=2)

    real_dist = euclidean_pair_distance(ZA, ZB)
    rand_dist, random_long_df = random_pair_distances_equal_n(
        ZA, ZB, sub,
        seed=RANDOM_SEED
    )

    _, p_mwu = mannwhitneyu(real_dist, rand_dist, alternative="less")
    _, perm_means, p_emp = permutation_test(
        real_dist, ZA, ZB,
        n_perm=N_PERMUTATIONS,
        seed=RANDOM_SEED
    )

    # Main result tables
    save_pair_level(
        sub, PA, PB, real_dist,
        os.path.join(OUT_DIR, "pair_level_shape_similarity.csv")
    )
    save_summary(
        real_dist, rand_dist, pca, p_mwu, p_emp,
        os.path.join(OUT_DIR, "summary_shape_similarity.csv")
    )

    # SPSS-ready tables
    save_spss_pair_distance_long(
        sub, real_dist, random_long_df,
        os.path.join(OUT_DIR, "spss_pair_distance_long.csv")
    )
    save_spss_pca_scores_long(
        sub, PA, PB,
        os.path.join(OUT_DIR, "spss_pca_scores_long.csv")
    )
    save_featurewise_paired_stats(
        sub, FEATURE_PAIRS,
        os.path.join(OUT_DIR, "featurewise_paired_stats.csv")
    )
    save_featurewise_long_for_spss(
        sub, FEATURE_PAIRS,
        os.path.join(OUT_DIR, "spss_featurewise_long.csv")
    )

    # Figures: all PDF
    plot_pca_overlay(
        PA, PB, pca.explained_variance_ratio_,
        os.path.join(OUT_DIR, "pca_overlay_A_vs_B.pdf")
    )
    plot_sampled_pair_lines(
        PA, PB, pca.explained_variance_ratio_,
        os.path.join(OUT_DIR, "pca_pairs_connected.pdf"),
        sample_n=SAMPLED_PAIR_LINES,
        seed=RANDOM_SEED
    )
    plot_real_vs_random_box(
        real_dist, rand_dist, p_mwu, p_emp,
        os.path.join(OUT_DIR, "real_vs_random_pair_distance_box.pdf")
    )
    plot_real_vs_random_violin(
        real_dist, rand_dist, p_mwu, p_emp,
        os.path.join(OUT_DIR, "real_vs_random_pair_distance_violin.pdf")
    )
    plot_loadings(
        pca, feature_names,
        os.path.join(OUT_DIR, "pca_loadings.pdf"),
        os.path.join(OUT_DIR, "pca_loadings.csv")
    )

    # Per-feature paired boxplots
    plot_featurewise_paired_boxplots(
        sub, FEATURE_PAIRS, OUT_DIR
    )

    print("Saved:")
    print(" - pair_level_shape_similarity.csv")
    print(" - summary_shape_similarity.csv")
    print(" - spss_pair_distance_long.csv")
    print(" - spss_pca_scores_long.csv")
    print(" - featurewise_paired_stats.csv")
    print(" - spss_featurewise_long.csv")
    print(" - pca_overlay_A_vs_B.pdf")
    print(" - pca_pairs_connected.pdf")
    print(" - real_vs_random_pair_distance_box.pdf")
    print(" - real_vs_random_pair_distance_violin.pdf")
    print(" - pca_loadings.pdf")
    print(" - pca_loadings.csv")

    for feat in FEATURE_PAIRS.keys():
        print(f" - paired_boxplot_{feat}_A_vs_B.pdf")


if __name__ == "__main__":
    main()

Loaded files: 6
Rows before filtering: 1071
Rows after filtering: 1071
Valid rows for PCA/similarity: 1071
Saved:
 - pair_level_shape_similarity.csv
 - summary_shape_similarity.csv
 - spss_pair_distance_long.csv
 - spss_pca_scores_long.csv
 - featurewise_paired_stats.csv
 - spss_featurewise_long.csv
 - pca_overlay_A_vs_B.pdf
 - pca_pairs_connected.pdf
 - real_vs_random_pair_distance_box.pdf
 - real_vs_random_pair_distance_violin.pdf
 - pca_loadings.pdf
 - pca_loadings.csv
 - paired_boxplot_aspect_A_vs_B.pdf
 - paired_boxplot_circularity_A_vs_B.pdf
 - paired_boxplot_solidity_A_vs_B.pdf
 - paired_boxplot_eccentricity_A_vs_B.pdf
 - paired_boxplot_extent_A_vs_B.pdf


In [12]:
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from scipy.stats import mannwhitneyu, gaussian_kde

# =========================
# USER SETTINGS
# =========================
PAIR_CSV_DIR = r"C:\Users\oxfil\Pair_for_PCA"
GLOB_PATTERN = "*.csv"
OUT_DIR = r".\shape_pair_pca_analysis416"

# 실제 네 pairs.csv 컬럼명 기준
FEATURE_PAIRS = {
    "aspect": ("aspectA", "aspectB"),
    "circularity": ("circA", "circB"),
    "solidity": ("solidityA", "solidityB"),
    "eccentricity": ("eccentricityA", "eccentricityB"),
    "extent": ("extentA", "extentB"),
}

# scaling
SCALER_TYPE = "standard"   # "standard" or "robust"

# filtering
MIN_IOU = None
MAX_DIST_UM = 100

# random / permutation
N_RANDOM_PER_REAL = 20
N_PERMUTATIONS = 2000
RANDOM_SEED = 80

# plotting
FIG_DPI = 300
POINT_SIZE = 16
POINT_ALPHA = 0.28
SAMPLED_PAIR_LINES = 120

USE_DENSITY_CONTOUR = True
CONTOUR_LEVELS = 5
CONTOUR_GRID_N = 200

# =========================
# GraphPad-like style
# =========================
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.linewidth": 1.2,
    "axes.grid": False,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COLOR_A = "#000080"
COLOR_B = "#028A0F"
COLOR_LINE = "#4D4D4D"
COLOR_REAL = "#4C78A8"
COLOR_RANDOM = "#B0B0B0"

# =========================


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def load_all_pairs(pair_dir, pattern):
    files = glob(os.path.join(pair_dir, pattern), recursive=True)
    if not files:
        raise FileNotFoundError(f"No pairs.csv found in: {pair_dir}")

    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["source_file"] = os.path.basename(f)
        df["source_path"] = f
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True)
    return data, files


def filter_pairs(df):
    out = df.copy()

    if MIN_IOU is not None and "iou_Bscale" in out.columns:
        out = out[out["iou_Bscale"] >= MIN_IOU].copy()

    if MAX_DIST_UM is not None and "dist_um" in out.columns:
        out = out[out["dist_um"] <= MAX_DIST_UM].copy()

    return out


def validate_columns(df, feature_pairs):
    missing = []
    for _, (ca, cb) in feature_pairs.items():
        if ca not in df.columns:
            missing.append(ca)
        if cb not in df.columns:
            missing.append(cb)
    if missing:
        raise ValueError(f"Missing columns: {missing}")


def prepare_feature_matrices(df, feature_pairs):
    feature_names = list(feature_pairs.keys())
    colsA = [feature_pairs[f][0] for f in feature_names]
    colsB = [feature_pairs[f][1] for f in feature_names]

    meta_cols = []
    for c in ["idxA", "idxB", "labelA", "labelB", "source_file", "source_path", "dist_um", "iou_Bscale"]:
        if c in df.columns:
            meta_cols.append(c)

    sub = df[colsA + colsB + meta_cols].copy()
    sub = sub.replace([np.inf, -np.inf], np.nan).dropna()

    A = sub[colsA].to_numpy(float)
    B = sub[colsB].to_numpy(float)

    A_df = pd.DataFrame(A, columns=feature_names, index=sub.index)
    B_df = pd.DataFrame(B, columns=feature_names, index=sub.index)

    return sub, A_df, B_df, feature_names


def get_scaler(kind="standard"):
    if kind.lower() == "robust":
        return RobustScaler()
    return StandardScaler()


def joint_normalize(A_df, B_df, scaler_type="standard"):
    scaler = get_scaler(scaler_type)
    pooled = pd.concat([A_df, B_df], axis=0).to_numpy(float)
    scaler.fit(pooled)

    ZA = scaler.transform(A_df.to_numpy(float))
    ZB = scaler.transform(B_df.to_numpy(float))
    return scaler, ZA, ZB


def run_pca(ZA, ZB, n_components=2):
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    pooled = np.vstack([ZA, ZB])
    proj = pca.fit_transform(pooled)
    PA = proj[:len(ZA)]
    PB = proj[len(ZA):]
    return pca, PA, PB


def euclidean_pair_distance(A, B):
    return np.sqrt(np.sum((A - B) ** 2, axis=1))


def random_pair_distances_equal_n(ZA, ZB, meta_df, seed=42):
    """
    real pair 수와 동일한 개수의 random one-to-one shuffled pair 생성
    """
    rng = np.random.default_rng(seed)
    n = len(ZA)

    perm = rng.permutation(n)

    same = perm == np.arange(n)
    if np.any(same):
        perm[same] = np.roll(perm[same], 1)

    rand_dist = np.sqrt(np.sum((ZA - ZB[perm]) ** 2, axis=1))

    rows = []
    for i in range(n):
        rows.append({
            "comparison_type": "Random pair",
            "distance": float(rand_dist[i]),
            "real_pair_index": int(i),
            "random_partner_index": int(perm[i]),
            "source_file_A": meta_df.iloc[i]["source_file"] if "source_file" in meta_df.columns else None,
            "source_file_B": meta_df.iloc[perm[i]]["source_file"] if "source_file" in meta_df.columns else None,
        })

    return rand_dist, pd.DataFrame(rows)


def permutation_test(real_dist, ZA, ZB, n_perm=2000, seed=42):
    rng = np.random.default_rng(seed)
    observed = np.mean(real_dist)

    perm_means = np.zeros(n_perm, dtype=float)
    n = len(ZA)

    for k in range(n_perm):
        perm_idx = rng.permutation(n)
        d = np.sqrt(np.sum((ZA - ZB[perm_idx]) ** 2, axis=1))
        perm_means[k] = np.mean(d)

    p_emp = (np.sum(perm_means <= observed) + 1) / (n_perm + 1)
    return observed, perm_means, p_emp


def add_density_contour(ax, x, y, color, levels=5, grid_n=200, linewidth=1.2, alpha=0.9):
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 20:
        return

    values = np.vstack([x, y])
    kde = gaussian_kde(values)

    xmin, xmax = np.min(x), np.max(x)
    ymin, ymax = np.min(y), np.max(y)

    xpad = 0.08 * (xmax - xmin + 1e-12)
    ypad = 0.08 * (ymax - ymin + 1e-12)

    xx, yy = np.meshgrid(
        np.linspace(xmin - xpad, xmax + xpad, grid_n),
        np.linspace(ymin - ypad, ymax + ypad, grid_n)
    )
    coords = np.vstack([xx.ravel(), yy.ravel()])
    zz = kde(coords).reshape(xx.shape)

    zmin, zmax = np.nanmin(zz), np.nanmax(zz)
    levs = np.linspace(zmin + 0.2 * (zmax - zmin), zmax, levels)
    ax.contour(
        xx, yy, zz,
        levels=levs,
        linewidths=linewidth,
        alpha=alpha,
        colors=[color],
        zorder=4
    )


def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", width=1.2, length=5)


def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    return "ns"


def plot_pca_overlay(PA, PB, explained, out_path, seed=42):
    fig, ax = plt.subplots(figsize=(6.0, 5.2))

    if USE_DENSITY_CONTOUR:
        add_density_contour(
            ax, PA[:, 0], PA[:, 1],
            color=COLOR_A,
            levels=CONTOUR_LEVELS,
            grid_n=CONTOUR_GRID_N,
            linewidth=0.9,
            alpha=0.45
        )
        add_density_contour(
            ax, PB[:, 0], PB[:, 1],
            color=COLOR_B,
            levels=CONTOUR_LEVELS,
            grid_n=CONTOUR_GRID_N,
            linewidth=0.9,
            alpha=0.45
        )

    dfA = pd.DataFrame({
        "PC1": PA[:, 0],
        "PC2": PA[:, 1],
        "color": COLOR_A,
    })
    dfB = pd.DataFrame({
        "PC1": PB[:, 0],
        "PC2": PB[:, 1],
        "color": COLOR_B,
    })

    plot_df = pd.concat([dfA, dfB], ignore_index=True)
    plot_df = plot_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    for _, row in plot_df.iterrows():
        ax.scatter(
            row["PC1"], row["PC2"],
            s=POINT_SIZE,
            alpha=POINT_ALPHA,
            color=row["color"],
            edgecolors="none",
            zorder=2
        )

    ax.scatter([], [], s=POINT_SIZE * 2, color=COLOR_A, alpha=0.7, label="Method A")
    ax.scatter([], [], s=POINT_SIZE * 2, color=COLOR_B, alpha=0.7, label="Method B")

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
    ax.legend(frameon=False, fontsize=11)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_sampled_pair_lines(PA, PB, explained, out_path, sample_n=120, seed=42):
    rng = np.random.default_rng(seed)
    n = len(PA)
    sample_n = min(sample_n, n)
    idx = rng.choice(n, size=sample_n, replace=False)

    fig, ax = plt.subplots(figsize=(6.0, 5.2))
    ax.scatter(PA[:, 0], PA[:, 1], s=10, alpha=0.16, color=COLOR_A, label="Method A")
    ax.scatter(PB[:, 0], PB[:, 1], s=10, alpha=0.16, color=COLOR_B, label="Method B")

    for i in idx:
        ax.plot([PA[i, 0], PB[i, 0]], [PA[i, 1], PB[i, 1]], lw=0.7, alpha=0.55, color=COLOR_LINE)

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
    ax.legend(frameon=False, fontsize=12)
    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_real_vs_random_box(real_dist, rand_dist, p_mwu, p_emp, out_path):
    fig, ax = plt.subplots(figsize=(4.8, 5.4))

    data = [real_dist, rand_dist]
    positions = [1, 2]

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=0.45,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color="black", linewidth=1.6),
        whiskerprops=dict(color="black", linewidth=1.2),
        capprops=dict(color="black", linewidth=1.2),
        boxprops=dict(linewidth=1.2, color="black"),
    )

    bp["boxes"][0].set_facecolor(COLOR_REAL)
    bp["boxes"][0].set_alpha(0.45)
    bp["boxes"][1].set_facecolor(COLOR_RANDOM)
    bp["boxes"][1].set_alpha(0.65)

    rng = np.random.default_rng(RANDOM_SEED)
    for pos, arr, color in zip(positions, data, [COLOR_REAL, COLOR_RANDOM]):
        x = rng.normal(pos, 0.06, size=len(arr))
        ax.scatter(x, arr, s=7, alpha=0.10, color=color, edgecolors="none")

    y_max = max(np.max(real_dist), np.max(rand_dist))
    y_line = y_max * 1.08
    y_text = y_max * 1.13

    ax.plot([1, 1, 2, 2], [y_line*0.99, y_line, y_line, y_line*0.99], color="black", lw=1.0)
    ax.text(1.5, y_text, f"MWU p = {p_mwu:.2e}\nPerm p = {p_emp:.2e}",
            ha="center", va="bottom", fontsize=10)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Real pair", "Random pair"], fontsize=12)
    ax.set_ylabel("Distance in normalized shape space", fontsize=14)
    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_real_vs_random_violin(real_dist, rand_dist, p_mwu, p_emp, out_path):
    fig, ax = plt.subplots(figsize=(4.8, 5.4))

    data = [real_dist, rand_dist]
    positions = [1, 2]

    vp = ax.violinplot(
        data,
        positions=positions,
        widths=0.7,
        showmeans=False,
        showmedians=False,
        showextrema=False
    )

    for body, color, alpha in zip(vp["bodies"], [COLOR_REAL, COLOR_RANDOM], [0.55, 0.65]):
        body.set_facecolor(color)
        body.set_edgecolor("black")
        body.set_linewidth(1.0)
        body.set_alpha(alpha)

    for pos, arr in zip(positions, data):
        q1, med, q3 = np.percentile(arr, [25, 50, 75])
        ax.plot([pos, pos], [q1, q3], color="black", lw=2.0, zorder=3)
        ax.plot([pos - 0.12, pos + 0.12], [med, med], color="black", lw=2.0, zorder=4)

    y_max = max(np.max(real_dist), np.max(rand_dist))
    y_line = y_max * 1.08
    y_text = y_max * 1.13

    ax.plot([1, 1, 2, 2], [y_line*0.99, y_line, y_line, y_line*0.99], color="black", lw=1.0)
    ax.text(
        1.5, y_text,
        f"MWU p = {p_mwu:.2e}\nPerm p = {p_emp:.2e}",
        ha="center", va="bottom", fontsize=10
    )

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Real pair", "Random pair"], fontsize=12)
    ax.set_ylabel("Distance in normalized shape space", fontsize=14)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_loadings(pca, feature_names, out_pdf, out_csv):
    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_names,
        columns=[f"PC{i+1}" for i in range(pca.components_.shape[0])]
    )
    loadings.to_csv(out_csv, index=True)

    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    x = np.arange(len(feature_names))

    ax.bar(x - 0.18, loadings["PC1"], width=0.36, color=COLOR_A, alpha=0.8, label="PC1")
    ax.bar(x + 0.18, loadings["PC2"], width=0.36, color=COLOR_B, alpha=0.8, label="PC2")

    ax.set_xticks(x)
    ax.set_xticklabels(feature_names, rotation=30, ha="right")
    ax.set_ylabel("Loading", fontsize=14)
    ax.legend(frameon=False, fontsize=11)
    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_pdf, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def save_pair_level(sub, PA, PB, real_dist, out_csv):
    out = sub.copy()
    out["A_PC1"] = PA[:, 0]
    out["A_PC2"] = PA[:, 1]
    out["B_PC1"] = PB[:, 0]
    out["B_PC2"] = PB[:, 1]
    out["shape_pair_distance"] = real_dist
    out.to_csv(out_csv, index=False)


def save_summary(real_dist, rand_dist, pca, p_mwu, p_emp, out_csv):
    summary = pd.DataFrame([{
        "n_real_pairs": len(real_dist),
        "n_random_pairs": len(rand_dist),
        "real_mean": float(np.mean(real_dist)),
        "real_median": float(np.median(real_dist)),
        "random_mean": float(np.mean(rand_dist)),
        "random_median": float(np.median(rand_dist)),
        "mannwhitney_p": float(p_mwu),
        "permutation_p": float(p_emp),
        "pc1_var_ratio": float(pca.explained_variance_ratio_[0]),
        "pc2_var_ratio": float(pca.explained_variance_ratio_[1]),
    }])
    summary.to_csv(out_csv, index=False)


def save_spss_pair_distance_long(sub, real_dist, random_long_df, out_csv):
    real_rows = []
    for i in range(len(sub)):
        row = {
            "comparison_type": "Real pair",
            "distance": float(real_dist[i]),
            "pair_row_index": int(i),
        }
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                row[c] = sub.iloc[i][c]
        real_rows.append(row)

    df_real = pd.DataFrame(real_rows)
    df_long = pd.concat([df_real, random_long_df], ignore_index=True)
    df_long.to_csv(out_csv, index=False)


def save_spss_pca_scores_long(sub, PA, PB, out_csv):
    rows = []
    for i in range(len(sub)):
        meta = {}
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                meta[c] = sub.iloc[i][c]

        rows.append({
            **meta,
            "pair_row_index": int(i),
            "method": "A",
            "PC1": float(PA[i, 0]),
            "PC2": float(PA[i, 1]),
        })
        rows.append({
            **meta,
            "pair_row_index": int(i),
            "method": "B",
            "PC1": float(PB[i, 0]),
            "PC2": float(PB[i, 1]),
        })

    pd.DataFrame(rows).to_csv(out_csv, index=False)


# =========================
# NEW: unpaired feature distribution analysis
# =========================
def save_featurewise_unpaired_stats(sub, feature_pairs, out_csv):
    rows = []

    for feat, (colA, colB) in feature_pairs.items():
        x = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)
        y = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        if len(x) == 0 or len(y) == 0:
            rows.append({
                "feature": feat,
                "n_A": len(x),
                "n_B": len(y),
                "A_mean": np.nan,
                "B_mean": np.nan,
                "A_median": np.nan,
                "B_median": np.nan,
                "A_std": np.nan,
                "B_std": np.nan,
                "mean_diff_A_minus_B": np.nan,
                "median_diff_A_minus_B": np.nan,
                "mwu_stat": np.nan,
                "mwu_p": np.nan,
            })
            continue

        stat, pval = mannwhitneyu(x, y, alternative="two-sided")

        rows.append({
            "feature": feat,
            "n_A": int(len(x)),
            "n_B": int(len(y)),
            "A_mean": float(np.mean(x)),
            "B_mean": float(np.mean(y)),
            "A_median": float(np.median(x)),
            "B_median": float(np.median(y)),
            "A_std": float(np.std(x)),
            "B_std": float(np.std(y)),
            "mean_diff_A_minus_B": float(np.mean(x) - np.mean(y)),
            "median_diff_A_minus_B": float(np.median(x) - np.median(y)),
            "mwu_stat": float(stat),
            "mwu_p": float(pval),
        })

    pd.DataFrame(rows).to_csv(out_csv, index=False)


def save_featurewise_unpaired_long(sub, feature_pairs, out_csv):
    rows = []

    for i in range(len(sub)):
        meta = {}
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                meta[c] = sub.iloc[i][c]

        for feat, (colA, colB) in feature_pairs.items():
            valA = sub.iloc[i][colA]
            valB = sub.iloc[i][colB]

            rows.append({
                **meta,
                "pair_row_index": int(i),
                "feature": feat,
                "method": "A",
                "value": valA
            })
            rows.append({
                **meta,
                "pair_row_index": int(i),
                "feature": feat,
                "method": "B",
                "value": valB
            })

    pd.DataFrame(rows).to_csv(out_csv, index=False)


def plot_featurewise_unpaired_boxplots(sub, feature_pairs, out_dir):
    rng = np.random.default_rng(RANDOM_SEED)

    for feat, (colA, colB) in feature_pairs.items():
        x = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)
        y = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        if len(x) == 0 or len(y) == 0:
            continue

        stat, pval = mannwhitneyu(x, y, alternative="two-sided")

        fig, ax = plt.subplots(figsize=(4.8, 5.6))

        bp = ax.boxplot(
            [x, y],
            positions=[1, 2],
            widths=0.5,
            patch_artist=True,
            showfliers=False,
            medianprops=dict(color="black", linewidth=1.6),
            whiskerprops=dict(color="black", linewidth=1.2),
            capprops=dict(color="black", linewidth=1.2),
            boxprops=dict(linewidth=1.2, color="black"),
        )

        bp["boxes"][0].set_facecolor(COLOR_A)
        bp["boxes"][0].set_alpha(0.45)
        bp["boxes"][1].set_facecolor(COLOR_B)
        bp["boxes"][1].set_alpha(0.45)

        ax.scatter(1 + rng.normal(0, 0.05, size=len(x)), x, s=10, alpha=0.20,
                   color=COLOR_A, edgecolors="none", zorder=2)
        ax.scatter(2 + rng.normal(0, 0.05, size=len(y)), y, s=10, alpha=0.20,
                   color=COLOR_B, edgecolors="none", zorder=2)

        ymax = np.nanmax(np.r_[x, y])
        ymin = np.nanmin(np.r_[x, y])
        yr = ymax - ymin if ymax > ymin else 1.0

        yline = ymax + 0.08 * yr
        ytext = ymax + 0.13 * yr

        ax.plot([1, 1, 2, 2], [yline - 0.01 * yr, yline, yline, yline - 0.01 * yr],
                color="black", lw=1.0)
        ax.text(1.5, ytext, f"MWU p = {pval:.2e}\n{p_to_stars(pval)}",
                ha="center", va="bottom", fontsize=10)

        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Method A", "Method B"], fontsize=12)
        ax.set_ylabel(feat, fontsize=14)

        style_axes(ax)
        fig.tight_layout()
        fig.savefig(
            os.path.join(out_dir, f"unpaired_boxplot_{feat}_A_vs_B.pdf"),
            dpi=FIG_DPI,
            bbox_inches="tight",
            transparent=True
        )
        plt.close(fig)


def plot_featurewise_unpaired_violinplots(sub, feature_pairs, out_dir):
    for feat, (colA, colB) in feature_pairs.items():
        x = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)
        y = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        if len(x) == 0 or len(y) == 0:
            continue

        stat, pval = mannwhitneyu(x, y, alternative="two-sided")

        fig, ax = plt.subplots(figsize=(4.8, 5.6))

        data = [x, y]
        positions = [1, 2]

        vp = ax.violinplot(
            data,
            positions=positions,
            widths=0.7,
            showmeans=False,
            showmedians=False,
            showextrema=False
        )

        for body, color, alpha in zip(vp["bodies"], [COLOR_A, COLOR_B], [0.45, 0.45]):
            body.set_facecolor(color)
            body.set_edgecolor("black")
            body.set_linewidth(1.0)
            body.set_alpha(alpha)

        for pos, arr in zip(positions, data):
            q1, med, q3 = np.percentile(arr, [25, 50, 75])
            ax.plot([pos, pos], [q1, q3], color="black", lw=2.0, zorder=3)
            ax.plot([pos - 0.12, pos + 0.12], [med, med], color="black", lw=2.0, zorder=4)

        ymax = np.nanmax(np.r_[x, y])
        ymin = np.nanmin(np.r_[x, y])
        yr = ymax - ymin if ymax > ymin else 1.0

        yline = ymax + 0.08 * yr
        ytext = ymax + 0.13 * yr

        ax.plot([1, 1, 2, 2], [yline - 0.01 * yr, yline, yline, yline - 0.01 * yr],
                color="black", lw=1.0)
        ax.text(1.5, ytext, f"MWU p = {pval:.2e}\n{p_to_stars(pval)}",
                ha="center", va="bottom", fontsize=10)

        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Method A", "Method B"], fontsize=12)
        ax.set_ylabel(feat, fontsize=14)

        style_axes(ax)
        fig.tight_layout()
        fig.savefig(
            os.path.join(out_dir, f"unpaired_violin_{feat}_A_vs_B.pdf"),
            dpi=FIG_DPI,
            bbox_inches="tight",
            transparent=True
        )
        plt.close(fig)


def main():
    ensure_dir(OUT_DIR)

    df, files = load_all_pairs(PAIR_CSV_DIR, GLOB_PATTERN)
    print(f"Loaded files: {len(files)}")
    print(f"Rows before filtering: {len(df)}")

    validate_columns(df, FEATURE_PAIRS)
    df = filter_pairs(df)
    print(f"Rows after filtering: {len(df)}")

    sub, A_df, B_df, feature_names = prepare_feature_matrices(df, FEATURE_PAIRS)
    print(f"Valid rows for PCA/similarity: {len(sub)}")

    scaler, ZA, ZB = joint_normalize(A_df, B_df, scaler_type=SCALER_TYPE)
    pca, PA, PB = run_pca(ZA, ZB, n_components=2)

    real_dist = euclidean_pair_distance(ZA, ZB)
    rand_dist, random_long_df = random_pair_distances_equal_n(
        ZA, ZB, sub,
        seed=RANDOM_SEED
    )

    _, p_mwu = mannwhitneyu(real_dist, rand_dist, alternative="less")
    _, perm_means, p_emp = permutation_test(
        real_dist, ZA, ZB,
        n_perm=N_PERMUTATIONS,
        seed=RANDOM_SEED
    )

    # Main result tables
    save_pair_level(
        sub, PA, PB, real_dist,
        os.path.join(OUT_DIR, "pair_level_shape_similarity.csv")
    )
    save_summary(
        real_dist, rand_dist, pca, p_mwu, p_emp,
        os.path.join(OUT_DIR, "summary_shape_similarity.csv")
    )

    # SPSS-ready tables
    save_spss_pair_distance_long(
        sub, real_dist, random_long_df,
        os.path.join(OUT_DIR, "spss_pair_distance_long.csv")
    )
    save_spss_pca_scores_long(
        sub, PA, PB,
        os.path.join(OUT_DIR, "spss_pca_scores_long.csv")
    )

    # NEW: feature-wise overall distribution comparison
    save_featurewise_unpaired_stats(
        sub, FEATURE_PAIRS,
        os.path.join(OUT_DIR, "featurewise_unpaired_stats.csv")
    )
    save_featurewise_unpaired_long(
        sub, FEATURE_PAIRS,
        os.path.join(OUT_DIR, "spss_featurewise_unpaired_long.csv")
    )

    # Figures
    plot_pca_overlay(
        PA, PB, pca.explained_variance_ratio_,
        os.path.join(OUT_DIR, "pca_overlay_A_vs_B.pdf")
    )
    plot_sampled_pair_lines(
        PA, PB, pca.explained_variance_ratio_,
        os.path.join(OUT_DIR, "pca_pairs_connected.pdf"),
        sample_n=SAMPLED_PAIR_LINES,
        seed=RANDOM_SEED
    )
    plot_real_vs_random_box(
        real_dist, rand_dist, p_mwu, p_emp,
        os.path.join(OUT_DIR, "real_vs_random_pair_distance_box.pdf")
    )
    plot_real_vs_random_violin(
        real_dist, rand_dist, p_mwu, p_emp,
        os.path.join(OUT_DIR, "real_vs_random_pair_distance_violin.pdf")
    )
    plot_loadings(
        pca, feature_names,
        os.path.join(OUT_DIR, "pca_loadings.pdf"),
        os.path.join(OUT_DIR, "pca_loadings.csv")
    )

    # NEW: feature-wise box / violin
    plot_featurewise_unpaired_boxplots(
        sub, FEATURE_PAIRS, OUT_DIR
    )
    plot_featurewise_unpaired_violinplots(
        sub, FEATURE_PAIRS, OUT_DIR
    )

    print("Saved:")
    print(" - pair_level_shape_similarity.csv")
    print(" - summary_shape_similarity.csv")
    print(" - spss_pair_distance_long.csv")
    print(" - spss_pca_scores_long.csv")
    print(" - featurewise_unpaired_stats.csv")
    print(" - spss_featurewise_unpaired_long.csv")
    print(" - pca_overlay_A_vs_B.pdf")
    print(" - pca_pairs_connected.pdf")
    print(" - real_vs_random_pair_distance_box.pdf")
    print(" - real_vs_random_pair_distance_violin.pdf")
    print(" - pca_loadings.pdf")
    print(" - pca_loadings.csv")

    for feat in FEATURE_PAIRS.keys():
        print(f" - unpaired_boxplot_{feat}_A_vs_B.pdf")
        print(f" - unpaired_violin_{feat}_A_vs_B.pdf")


if __name__ == "__main__":
    main()

Loaded files: 6
Rows before filtering: 1071
Rows after filtering: 1071
Valid rows for PCA/similarity: 1071
Saved:
 - pair_level_shape_similarity.csv
 - summary_shape_similarity.csv
 - spss_pair_distance_long.csv
 - spss_pca_scores_long.csv
 - featurewise_unpaired_stats.csv
 - spss_featurewise_unpaired_long.csv
 - pca_overlay_A_vs_B.pdf
 - pca_pairs_connected.pdf
 - real_vs_random_pair_distance_box.pdf
 - real_vs_random_pair_distance_violin.pdf
 - pca_loadings.pdf
 - pca_loadings.csv
 - unpaired_boxplot_aspect_A_vs_B.pdf
 - unpaired_violin_aspect_A_vs_B.pdf
 - unpaired_boxplot_circularity_A_vs_B.pdf
 - unpaired_violin_circularity_A_vs_B.pdf
 - unpaired_boxplot_solidity_A_vs_B.pdf
 - unpaired_violin_solidity_A_vs_B.pdf
 - unpaired_boxplot_eccentricity_A_vs_B.pdf
 - unpaired_violin_eccentricity_A_vs_B.pdf
 - unpaired_boxplot_extent_A_vs_B.pdf
 - unpaired_violin_extent_A_vs_B.pdf


In [16]:
# -*- coding: utf-8 -*-

import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from scipy.stats import mannwhitneyu, gaussian_kde

# =========================
# USER SETTINGS
# =========================
PAIR_CSV_DIR = r"C:\Users\oxfil\Pair_for_PCA"
GLOB_PATTERN = "*.csv"
OUT_DIR = r".\shape_pair_pca_analysis416"

FEATURE_PAIRS = {
    "aspect": ("aspectA", "aspectB"),
    "circularity": ("circA", "circB"),
    "solidity": ("solidityA", "solidityB"),
    "eccentricity": ("eccentricityA", "eccentricityB"),
    "extent": ("extentA", "extentB"),
}

SCALER_TYPE = "standard"   # "standard" or "robust"

MIN_IOU = None
MAX_DIST_UM = 100

N_PERMUTATIONS = 2000
RANDOM_SEED = 80

FIG_DPI = 300
POINT_SIZE = 16
POINT_ALPHA = 0.28
SAMPLED_PAIR_LINES = 120

USE_DENSITY_CONTOUR = True
CONTOUR_LEVELS = 5
CONTOUR_GRID_N = 200

# =========================
# GraphPad-like style
# =========================
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.linewidth": 1.2,
    "axes.grid": False,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COLOR_A = "#000080"
COLOR_B = "#028A0F"
COLOR_LINE = "#4D4D4D"
COLOR_REAL = "#4C78A8"
COLOR_RANDOM = "#B0B0B0"


# =========================
# Basic utilities
# =========================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def load_all_pairs(pair_dir, pattern):
    files = glob(os.path.join(pair_dir, pattern), recursive=True)
    if not files:
        raise FileNotFoundError(f"No pairs.csv found in: {pair_dir}")

    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["source_file"] = os.path.basename(f)
        df["source_path"] = f
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True)
    return data, files


def filter_pairs(df):
    out = df.copy()

    if MIN_IOU is not None and "iou_Bscale" in out.columns:
        out = out[out["iou_Bscale"] >= MIN_IOU].copy()

    if MAX_DIST_UM is not None and "dist_um" in out.columns:
        out = out[out["dist_um"] <= MAX_DIST_UM].copy()

    return out


def validate_columns(df, feature_pairs):
    missing = []
    for _, (ca, cb) in feature_pairs.items():
        if ca not in df.columns:
            missing.append(ca)
        if cb not in df.columns:
            missing.append(cb)
    if missing:
        raise ValueError(f"Missing columns: {missing}")


def prepare_feature_matrices(df, feature_pairs):
    feature_names = list(feature_pairs.keys())
    colsA = [feature_pairs[f][0] for f in feature_names]
    colsB = [feature_pairs[f][1] for f in feature_names]

    meta_cols = []
    for c in ["idxA", "idxB", "labelA", "labelB", "source_file", "source_path", "dist_um", "iou_Bscale"]:
        if c in df.columns:
            meta_cols.append(c)

    sub = df[colsA + colsB + meta_cols].copy()
    sub = sub.replace([np.inf, -np.inf], np.nan).dropna()

    A = sub[colsA].to_numpy(float)
    B = sub[colsB].to_numpy(float)

    A_df = pd.DataFrame(A, columns=feature_names, index=sub.index)
    B_df = pd.DataFrame(B, columns=feature_names, index=sub.index)

    return sub, A_df, B_df, feature_names


def get_scaler(kind="standard"):
    if kind.lower() == "robust":
        return RobustScaler()
    return StandardScaler()


def joint_normalize(A_df, B_df, scaler_type="standard"):
    scaler = get_scaler(scaler_type)
    pooled = pd.concat([A_df, B_df], axis=0).to_numpy(float)
    scaler.fit(pooled)

    ZA = scaler.transform(A_df.to_numpy(float))
    ZB = scaler.transform(B_df.to_numpy(float))
    return scaler, ZA, ZB


def run_pca(ZA, ZB, n_components=2):
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    pooled = np.vstack([ZA, ZB])
    proj = pca.fit_transform(pooled)
    PA = proj[:len(ZA)]
    PB = proj[len(ZA):]
    return pca, PA, PB


def euclidean_pair_distance(A, B):
    return np.sqrt(np.sum((A - B) ** 2, axis=1))


def random_pair_distances_equal_n(ZA, ZB, meta_df, seed=42):
    rng = np.random.default_rng(seed)
    n = len(ZA)

    perm = rng.permutation(n)
    same = perm == np.arange(n)
    if np.any(same):
        perm[same] = np.roll(perm[same], 1)

    rand_dist = np.sqrt(np.sum((ZA - ZB[perm]) ** 2, axis=1))

    rows = []
    for i in range(n):
        rows.append({
            "comparison_type": "Random pair",
            "distance": float(rand_dist[i]),
            "real_pair_index": int(i),
            "random_partner_index": int(perm[i]),
            "source_file_A": meta_df.iloc[i]["source_file"] if "source_file" in meta_df.columns else None,
            "source_file_B": meta_df.iloc[perm[i]]["source_file"] if "source_file" in meta_df.columns else None,
        })

    return rand_dist, pd.DataFrame(rows)


def permutation_test(real_dist, ZA, ZB, n_perm=2000, seed=42):
    rng = np.random.default_rng(seed)
    observed = np.mean(real_dist)

    perm_means = np.zeros(n_perm, dtype=float)
    n = len(ZA)

    for k in range(n_perm):
        perm_idx = rng.permutation(n)
        d = np.sqrt(np.sum((ZA - ZB[perm_idx]) ** 2, axis=1))
        perm_means[k] = np.mean(d)

    p_emp = (np.sum(perm_means <= observed) + 1) / (n_perm + 1)
    return observed, perm_means, p_emp


# =========================
# Effect size / percent diff
# =========================
def cliffs_delta(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    nx = len(x)
    ny = len(y)
    if nx == 0 or ny == 0:
        return np.nan

    gt = 0
    lt = 0
    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)

    return float((gt - lt) / (nx * ny))


def cliffs_delta_label(delta):
    if not np.isfinite(delta):
        return "NA"
    ad = abs(delta)
    if ad < 0.147:
        return "negligible"
    elif ad < 0.33:
        return "small"
    elif ad < 0.474:
        return "medium"
    return "large"


def percent_shift_from_median(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if len(x) == 0 or len(y) == 0:
        return np.nan

    med_x = np.median(x)
    med_y = np.median(y)

    denom = abs(med_y) if abs(med_y) > 1e-12 else np.nan
    if not np.isfinite(denom):
        return np.nan

    return float(100.0 * (med_x - med_y) / denom)


def symmetric_percent_difference_from_means(x, y, absolute=False):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if len(x) == 0 or len(y) == 0:
        return np.nan

    mean_x = np.mean(x)
    mean_y = np.mean(y)
    denom = (mean_x + mean_y) / 2.0

    if abs(denom) < 1e-12:
        return np.nan

    val = 100.0 * (mean_x - mean_y) / denom
    if absolute:
        val = abs(val)
    return float(val)


# =========================
# Plot utilities
# =========================
def add_density_contour(ax, x, y, color, levels=5, grid_n=200, linewidth=1.2, alpha=0.9):
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 20:
        return

    values = np.vstack([x, y])
    kde = gaussian_kde(values)

    xmin, xmax = np.min(x), np.max(x)
    ymin, ymax = np.min(y), np.max(y)

    xpad = 0.08 * (xmax - xmin + 1e-12)
    ypad = 0.08 * (ymax - ymin + 1e-12)

    xx, yy = np.meshgrid(
        np.linspace(xmin - xpad, xmax + xpad, grid_n),
        np.linspace(ymin - ypad, ymax + ypad, grid_n)
    )
    coords = np.vstack([xx.ravel(), yy.ravel()])
    zz = kde(coords).reshape(xx.shape)

    zmin, zmax = np.nanmin(zz), np.nanmax(zz)
    levs = np.linspace(zmin + 0.2 * (zmax - zmin), zmax, levels)

    ax.contour(xx, yy, zz, levels=levs, linewidths=linewidth, alpha=alpha, colors=[color], zorder=4)


def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", width=1.2, length=5)


def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    return "ns"


# =========================
# PCA / distance plots
# =========================
def plot_pca_overlay(PA, PB, explained, out_path, seed=42):
    fig, ax = plt.subplots(figsize=(6.0, 5.2))

    if USE_DENSITY_CONTOUR:
        add_density_contour(ax, PA[:, 0], PA[:, 1], color=COLOR_A,
                            levels=CONTOUR_LEVELS, grid_n=CONTOUR_GRID_N,
                            linewidth=0.9, alpha=0.45)
        add_density_contour(ax, PB[:, 0], PB[:, 1], color=COLOR_B,
                            levels=CONTOUR_LEVELS, grid_n=CONTOUR_GRID_N,
                            linewidth=0.9, alpha=0.45)

    dfA = pd.DataFrame({"PC1": PA[:, 0], "PC2": PA[:, 1], "color": COLOR_A})
    dfB = pd.DataFrame({"PC1": PB[:, 0], "PC2": PB[:, 1], "color": COLOR_B})
    plot_df = pd.concat([dfA, dfB], ignore_index=True)
    plot_df = plot_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    for _, row in plot_df.iterrows():
        ax.scatter(row["PC1"], row["PC2"], s=POINT_SIZE, alpha=POINT_ALPHA,
                   color=row["color"], edgecolors="none", zorder=2)

    ax.scatter([], [], s=POINT_SIZE * 2, color=COLOR_A, alpha=0.7, label="Method A")
    ax.scatter([], [], s=POINT_SIZE * 2, color=COLOR_B, alpha=0.7, label="Method B")

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
    ax.legend(frameon=False, fontsize=11)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_sampled_pair_lines(PA, PB, explained, out_path, sample_n=120, seed=42):
    rng = np.random.default_rng(seed)
    n = len(PA)
    sample_n = min(sample_n, n)
    idx = rng.choice(n, size=sample_n, replace=False)

    fig, ax = plt.subplots(figsize=(6.0, 5.2))
    ax.scatter(PA[:, 0], PA[:, 1], s=10, alpha=0.16, color=COLOR_A, label="Method A")
    ax.scatter(PB[:, 0], PB[:, 1], s=10, alpha=0.16, color=COLOR_B, label="Method B")

    for i in idx:
        ax.plot([PA[i, 0], PB[i, 0]], [PA[i, 1], PB[i, 1]], lw=0.7, alpha=0.55, color=COLOR_LINE)

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
    ax.legend(frameon=False, fontsize=12)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_real_vs_random_box(real_dist, rand_dist, p_mwu, p_emp, out_path):
    fig, ax = plt.subplots(figsize=(4.8, 5.4))
    data = [real_dist, rand_dist]
    positions = [1, 2]

    bp = ax.boxplot(
        data, positions=positions, widths=0.45, patch_artist=True, showfliers=False,
        medianprops=dict(color="black", linewidth=1.6),
        whiskerprops=dict(color="black", linewidth=1.2),
        capprops=dict(color="black", linewidth=1.2),
        boxprops=dict(linewidth=1.2, color="black"),
    )

    bp["boxes"][0].set_facecolor(COLOR_REAL)
    bp["boxes"][0].set_alpha(0.45)
    bp["boxes"][1].set_facecolor(COLOR_RANDOM)
    bp["boxes"][1].set_alpha(0.65)

    rng = np.random.default_rng(RANDOM_SEED)
    for pos, arr, color in zip(positions, data, [COLOR_REAL, COLOR_RANDOM]):
        x = rng.normal(pos, 0.06, size=len(arr))
        ax.scatter(x, arr, s=7, alpha=0.10, color=color, edgecolors="none")

    y_max = max(np.max(real_dist), np.max(rand_dist))
    y_line = y_max * 1.08
    y_text = y_max * 1.13

    ax.plot([1, 1, 2, 2], [y_line * 0.99, y_line, y_line, y_line * 0.99], color="black", lw=1.0)
    ax.text(1.5, y_text, f"MWU p = {p_mwu:.2e}\nPerm p = {p_emp:.2e}",
            ha="center", va="bottom", fontsize=10)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Real pair", "Random pair"], fontsize=12)
    ax.set_ylabel("Distance in normalized shape space", fontsize=14)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_real_vs_random_violin(real_dist, rand_dist, p_mwu, p_emp, out_path):
    fig, ax = plt.subplots(figsize=(4.8, 5.4))
    data = [real_dist, rand_dist]
    positions = [1, 2]

    vp = ax.violinplot(data, positions=positions, widths=0.7,
                       showmeans=False, showmedians=False, showextrema=False)

    for body, color, alpha in zip(vp["bodies"], [COLOR_REAL, COLOR_RANDOM], [0.55, 0.65]):
        body.set_facecolor(color)
        body.set_edgecolor("black")
        body.set_linewidth(1.0)
        body.set_alpha(alpha)

    for pos, arr in zip(positions, data):
        q1, med, q3 = np.percentile(arr, [25, 50, 75])
        ax.plot([pos, pos], [q1, q3], color="black", lw=2.0, zorder=3)
        ax.plot([pos - 0.12, pos + 0.12], [med, med], color="black", lw=2.0, zorder=4)

    y_max = max(np.max(real_dist), np.max(rand_dist))
    y_line = y_max * 1.08
    y_text = y_max * 1.13

    ax.plot([1, 1, 2, 2], [y_line * 0.99, y_line, y_line, y_line * 0.99], color="black", lw=1.0)
    ax.text(1.5, y_text, f"MWU p = {p_mwu:.2e}\nPerm p = {p_emp:.2e}",
            ha="center", va="bottom", fontsize=10)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Real pair", "Random pair"], fontsize=12)
    ax.set_ylabel("Distance in normalized shape space", fontsize=14)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_loadings(pca, feature_names, out_pdf, out_csv):
    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_names,
        columns=[f"PC{i+1}" for i in range(pca.components_.shape[0])]
    )
    loadings.to_csv(out_csv, index=True)

    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    x = np.arange(len(feature_names))
    ax.bar(x - 0.18, loadings["PC1"], width=0.36, color=COLOR_A, alpha=0.8, label="PC1")
    ax.bar(x + 0.18, loadings["PC2"], width=0.36, color=COLOR_B, alpha=0.8, label="PC2")

    ax.set_xticks(x)
    ax.set_xticklabels(feature_names, rotation=30, ha="right")
    ax.set_ylabel("Loading", fontsize=14)
    ax.legend(frameon=False, fontsize=11)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_pdf, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


# =========================
# Save tables
# =========================
def save_pair_level(sub, PA, PB, real_dist, out_csv):
    out = sub.copy()
    out["A_PC1"] = PA[:, 0]
    out["A_PC2"] = PA[:, 1]
    out["B_PC1"] = PB[:, 0]
    out["B_PC2"] = PB[:, 1]
    out["shape_pair_distance"] = real_dist
    out.to_csv(out_csv, index=False)


def save_summary(real_dist, rand_dist, pca, p_mwu, p_emp, out_csv):
    summary = pd.DataFrame([{
        "n_real_pairs": len(real_dist),
        "n_random_pairs": len(rand_dist),
        "real_mean": float(np.mean(real_dist)),
        "real_median": float(np.median(real_dist)),
        "random_mean": float(np.mean(rand_dist)),
        "random_median": float(np.median(rand_dist)),
        "mannwhitney_p": float(p_mwu),
        "permutation_p": float(p_emp),
        "pc1_var_ratio": float(pca.explained_variance_ratio_[0]),
        "pc2_var_ratio": float(pca.explained_variance_ratio_[1]),
    }])
    summary.to_csv(out_csv, index=False)


def save_spss_pair_distance_long(sub, real_dist, random_long_df, out_csv):
    real_rows = []
    for i in range(len(sub)):
        row = {
            "comparison_type": "Real pair",
            "distance": float(real_dist[i]),
            "pair_row_index": int(i),
        }
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                row[c] = sub.iloc[i][c]
        real_rows.append(row)

    df_real = pd.DataFrame(real_rows)
    df_long = pd.concat([df_real, random_long_df], ignore_index=True)
    df_long.to_csv(out_csv, index=False)


def save_spss_pca_scores_long(sub, PA, PB, out_csv):
    rows = []
    for i in range(len(sub)):
        meta = {}
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                meta[c] = sub.iloc[i][c]

        rows.append({
            **meta,
            "pair_row_index": int(i),
            "method": "A",
            "PC1": float(PA[i, 0]),
            "PC2": float(PA[i, 1]),
        })
        rows.append({
            **meta,
            "pair_row_index": int(i),
            "method": "B",
            "PC1": float(PB[i, 0]),
            "PC2": float(PB[i, 1]),
        })

    pd.DataFrame(rows).to_csv(out_csv, index=False)


# =========================
# Unpaired feature distribution analysis
# =========================
def save_featurewise_unpaired_stats(sub, feature_pairs, out_csv):
    rows = []

    for feat, (colA, colB) in feature_pairs.items():
        x = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)
        y = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        if len(x) == 0 or len(y) == 0:
            rows.append({
                "feature": feat,
                "n_A": len(x),
                "n_B": len(y),
                "A_mean": np.nan,
                "B_mean": np.nan,
                "A_median": np.nan,
                "B_median": np.nan,
                "A_std": np.nan,
                "B_std": np.nan,
                "mean_diff_A_minus_B": np.nan,
                "median_diff_A_minus_B": np.nan,
                "median_percent_shift_A_vs_B": np.nan,
                "mean_percent_diff_sym_A_vs_B": np.nan,
                "mean_percent_abs_diff_sym_A_vs_B": np.nan,
                "mwu_stat": np.nan,
                "mwu_p": np.nan,
                "cliffs_delta": np.nan,
                "cliffs_delta_magnitude": "NA",
            })
            continue

        stat, pval = mannwhitneyu(x, y, alternative="two-sided")
        delta = cliffs_delta(x, y)

        rows.append({
            "feature": feat,
            "n_A": int(len(x)),
            "n_B": int(len(y)),
            "A_mean": float(np.mean(x)),
            "B_mean": float(np.mean(y)),
            "A_median": float(np.median(x)),
            "B_median": float(np.median(y)),
            "A_std": float(np.std(x)),
            "B_std": float(np.std(y)),
            "mean_diff_A_minus_B": float(np.mean(x) - np.mean(y)),
            "median_diff_A_minus_B": float(np.median(x) - np.median(y)),
            "median_percent_shift_A_vs_B": percent_shift_from_median(x, y),
            "mean_percent_diff_sym_A_vs_B": symmetric_percent_difference_from_means(x, y, absolute=False),
            "mean_percent_abs_diff_sym_A_vs_B": symmetric_percent_difference_from_means(x, y, absolute=True),
            "mwu_stat": float(stat),
            "mwu_p": float(pval),
            "cliffs_delta": float(delta),
            "cliffs_delta_magnitude": cliffs_delta_label(delta),
        })

    pd.DataFrame(rows).to_csv(out_csv, index=False)


def save_featurewise_unpaired_long(sub, feature_pairs, out_csv):
    rows = []

    for i in range(len(sub)):
        meta = {}
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                meta[c] = sub.iloc[i][c]

        for feat, (colA, colB) in feature_pairs.items():
            rows.append({
                **meta,
                "pair_row_index": int(i),
                "feature": feat,
                "method": "A",
                "value": sub.iloc[i][colA]
            })
            rows.append({
                **meta,
                "pair_row_index": int(i),
                "feature": feat,
                "method": "B",
                "value": sub.iloc[i][colB]
            })

    pd.DataFrame(rows).to_csv(out_csv, index=False)


def plot_featurewise_unpaired_boxplots(sub, feature_pairs, out_dir):
    rng = np.random.default_rng(RANDOM_SEED)

    for feat, (colA, colB) in feature_pairs.items():
        x = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)
        y = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        if len(x) == 0 or len(y) == 0:
            continue

        stat, pval = mannwhitneyu(x, y, alternative="two-sided")
        delta = cliffs_delta(x, y)
        delta_label = cliffs_delta_label(delta)
        med_diff = float(np.median(x) - np.median(y))
        mean_pct = symmetric_percent_difference_from_means(x, y, absolute=False)

        fig, ax = plt.subplots(figsize=(4.8, 6.1))

        bp = ax.boxplot(
            [x, y], positions=[1, 2], widths=0.5, patch_artist=True, showfliers=False,
            medianprops=dict(color="black", linewidth=1.6),
            whiskerprops=dict(color="black", linewidth=1.2),
            capprops=dict(color="black", linewidth=1.2),
            boxprops=dict(linewidth=1.2, color="black"),
        )

        bp["boxes"][0].set_facecolor(COLOR_A)
        bp["boxes"][0].set_alpha(0.45)
        bp["boxes"][1].set_facecolor(COLOR_B)
        bp["boxes"][1].set_alpha(0.45)

        ax.scatter(1 + rng.normal(0, 0.05, size=len(x)), x, s=10, alpha=0.20,
                   color=COLOR_A, edgecolors="none", zorder=2)
        ax.scatter(2 + rng.normal(0, 0.05, size=len(y)), y, s=10, alpha=0.20,
                   color=COLOR_B, edgecolors="none", zorder=2)

        ymax = np.nanmax(np.r_[x, y])
        ymin = np.nanmin(np.r_[x, y])
        yr = ymax - ymin if ymax > ymin else 1.0
        yline = ymax + 0.08 * yr
        ytext = ymax + 0.13 * yr

        ax.plot([1, 1, 2, 2], [yline - 0.01 * yr, yline, yline, yline - 0.01 * yr],
                color="black", lw=1.0)
        ax.text(
            1.5, ytext,
            f"MWU p = {pval:.2e}\nCliff's δ = {delta:.3f} ({delta_label})\nMedian diff = {med_diff:.4g}\nSym mean % diff = {mean_pct:.3g}%",
            ha="center", va="bottom", fontsize=10
        )

        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Method A", "Method B"], fontsize=12)
        ax.set_ylabel(feat, fontsize=14)

        style_axes(ax)
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, f"unpaired_boxplot_{feat}_A_vs_B.pdf"),
                    dpi=FIG_DPI, bbox_inches="tight", transparent=True)
        plt.close(fig)


def plot_featurewise_unpaired_violinplots(sub, feature_pairs, out_dir):
    for feat, (colA, colB) in feature_pairs.items():
        x = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)
        y = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        if len(x) == 0 or len(y) == 0:
            continue

        stat, pval = mannwhitneyu(x, y, alternative="two-sided")
        delta = cliffs_delta(x, y)
        delta_label = cliffs_delta_label(delta)
        med_diff = float(np.median(x) - np.median(y))
        mean_pct = symmetric_percent_difference_from_means(x, y, absolute=False)

        fig, ax = plt.subplots(figsize=(4.8, 6.1))
        data = [x, y]
        positions = [1, 2]

        vp = ax.violinplot(data, positions=positions, widths=0.7,
                           showmeans=False, showmedians=False, showextrema=False)

        for body, color, alpha in zip(vp["bodies"], [COLOR_A, COLOR_B], [0.45, 0.45]):
            body.set_facecolor(color)
            body.set_edgecolor("black")
            body.set_linewidth(1.0)
            body.set_alpha(alpha)

        for pos, arr in zip(positions, data):
            q1, med, q3 = np.percentile(arr, [25, 50, 75])
            ax.plot([pos, pos], [q1, q3], color="black", lw=2.0, zorder=3)
            ax.plot([pos - 0.12, pos + 0.12], [med, med], color="black", lw=2.0, zorder=4)

        ymax = np.nanmax(np.r_[x, y])
        ymin = np.nanmin(np.r_[x, y])
        yr = ymax - ymin if ymax > ymin else 1.0
        yline = ymax + 0.08 * yr
        ytext = ymax + 0.13 * yr

        ax.plot([1, 1, 2, 2], [yline - 0.01 * yr, yline, yline, yline - 0.01 * yr],
                color="black", lw=1.0)
        ax.text(
            1.5, ytext,
            f"MWU p = {pval:.2e}\nCliff's δ = {delta:.3f} ({delta_label})\nMedian diff = {med_diff:.4g}\nSym mean % diff = {mean_pct:.3g}%",
            ha="center", va="bottom", fontsize=10
        )

        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Method A", "Method B"], fontsize=12)
        ax.set_ylabel(feat, fontsize=14)

        style_axes(ax)
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, f"unpaired_violin_{feat}_A_vs_B.pdf"),
                    dpi=FIG_DPI, bbox_inches="tight", transparent=True)
        plt.close(fig)


# =========================
# Main
# =========================
def main():
    ensure_dir(OUT_DIR)

    df, files = load_all_pairs(PAIR_CSV_DIR, GLOB_PATTERN)
    print(f"Loaded files: {len(files)}")
    print(f"Rows before filtering: {len(df)}")

    validate_columns(df, FEATURE_PAIRS)
    df = filter_pairs(df)
    print(f"Rows after filtering: {len(df)}")

    sub, A_df, B_df, feature_names = prepare_feature_matrices(df, FEATURE_PAIRS)
    print(f"Valid rows for PCA/similarity: {len(sub)}")

    scaler, ZA, ZB = joint_normalize(A_df, B_df, scaler_type=SCALER_TYPE)
    pca, PA, PB = run_pca(ZA, ZB, n_components=2)

    real_dist = euclidean_pair_distance(ZA, ZB)
    rand_dist, random_long_df = random_pair_distances_equal_n(ZA, ZB, sub, seed=RANDOM_SEED)

    _, p_mwu = mannwhitneyu(real_dist, rand_dist, alternative="less")
    _, perm_means, p_emp = permutation_test(real_dist, ZA, ZB, n_perm=N_PERMUTATIONS, seed=RANDOM_SEED)

    save_pair_level(sub, PA, PB, real_dist, os.path.join(OUT_DIR, "pair_level_shape_similarity.csv"))
    save_summary(real_dist, rand_dist, pca, p_mwu, p_emp, os.path.join(OUT_DIR, "summary_shape_similarity.csv"))

    save_spss_pair_distance_long(sub, real_dist, random_long_df,
                                 os.path.join(OUT_DIR, "spss_pair_distance_long.csv"))
    save_spss_pca_scores_long(sub, PA, PB,
                              os.path.join(OUT_DIR, "spss_pca_scores_long.csv"))

    save_featurewise_unpaired_stats(
        sub, FEATURE_PAIRS,
        os.path.join(OUT_DIR, "featurewise_unpaired_stats.csv")
    )
    save_featurewise_unpaired_long(
        sub, FEATURE_PAIRS,
        os.path.join(OUT_DIR, "spss_featurewise_unpaired_long.csv")
    )

    plot_pca_overlay(PA, PB, pca.explained_variance_ratio_,
                     os.path.join(OUT_DIR, "pca_overlay_A_vs_B.pdf"))
    plot_sampled_pair_lines(PA, PB, pca.explained_variance_ratio_,
                            os.path.join(OUT_DIR, "pca_pairs_connected.pdf"),
                            sample_n=SAMPLED_PAIR_LINES, seed=RANDOM_SEED)
    plot_real_vs_random_box(real_dist, rand_dist, p_mwu, p_emp,
                            os.path.join(OUT_DIR, "real_vs_random_pair_distance_box.pdf"))
    plot_real_vs_random_violin(real_dist, rand_dist, p_mwu, p_emp,
                               os.path.join(OUT_DIR, "real_vs_random_pair_distance_violin.pdf"))
    plot_loadings(pca, feature_names,
                  os.path.join(OUT_DIR, "pca_loadings.pdf"),
                  os.path.join(OUT_DIR, "pca_loadings.csv"))

    plot_featurewise_unpaired_boxplots(sub, FEATURE_PAIRS, OUT_DIR)
    plot_featurewise_unpaired_violinplots(sub, FEATURE_PAIRS, OUT_DIR)

    print("Saved:")
    print(" - pair_level_shape_similarity.csv")
    print(" - summary_shape_similarity.csv")
    print(" - spss_pair_distance_long.csv")
    print(" - spss_pca_scores_long.csv")
    print(" - featurewise_unpaired_stats.csv")
    print(" - spss_featurewise_unpaired_long.csv")
    print(" - pca_overlay_A_vs_B.pdf")
    print(" - pca_pairs_connected.pdf")
    print(" - real_vs_random_pair_distance_box.pdf")
    print(" - real_vs_random_pair_distance_violin.pdf")
    print(" - pca_loadings.pdf")
    print(" - pca_loadings.csv")

    for feat in FEATURE_PAIRS.keys():
        print(f" - unpaired_boxplot_{feat}_A_vs_B.pdf")
        print(f" - unpaired_violin_{feat}_A_vs_B.pdf")


if __name__ == "__main__":
    main()

Loaded files: 6
Rows before filtering: 1071
Rows after filtering: 1071
Valid rows for PCA/similarity: 1071
Saved:
 - pair_level_shape_similarity.csv
 - summary_shape_similarity.csv
 - spss_pair_distance_long.csv
 - spss_pca_scores_long.csv
 - featurewise_unpaired_stats.csv
 - spss_featurewise_unpaired_long.csv
 - pca_overlay_A_vs_B.pdf
 - pca_pairs_connected.pdf
 - real_vs_random_pair_distance_box.pdf
 - real_vs_random_pair_distance_violin.pdf
 - pca_loadings.pdf
 - pca_loadings.csv
 - unpaired_boxplot_aspect_A_vs_B.pdf
 - unpaired_violin_aspect_A_vs_B.pdf
 - unpaired_boxplot_circularity_A_vs_B.pdf
 - unpaired_violin_circularity_A_vs_B.pdf
 - unpaired_boxplot_solidity_A_vs_B.pdf
 - unpaired_violin_solidity_A_vs_B.pdf
 - unpaired_boxplot_eccentricity_A_vs_B.pdf
 - unpaired_violin_eccentricity_A_vs_B.pdf
 - unpaired_boxplot_extent_A_vs_B.pdf
 - unpaired_violin_extent_A_vs_B.pdf


In [1]:
# -*- coding: utf-8 -*-

import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from scipy.stats import mannwhitneyu, gaussian_kde, pearsonr, spearmanr

# =========================
# USER SETTINGS
# =========================
PAIR_CSV_DIR = r"C:\Users\oxfil\Pair_for_PCA"
GLOB_PATTERN = "*.csv"
OUT_DIR = r".\shape_pair_pca_analysis416"

FEATURE_PAIRS = {
    "aspect": ("aspectA", "aspectB"),
    "circularity": ("circA", "circB"),
    "solidity": ("solidityA", "solidityB"),
    "eccentricity": ("eccentricityA", "eccentricityB"),
    "extent": ("extentA", "extentB"),
}

SCALER_TYPE = "standard"   # "standard" or "robust"

MIN_IOU = None
MAX_DIST_UM = 100

N_PERMUTATIONS = 2000
RANDOM_SEED = 80

FIG_DPI = 300
POINT_SIZE = 16
POINT_ALPHA = 0.28
SAMPLED_PAIR_LINES = 120

USE_DENSITY_CONTOUR = True
CONTOUR_LEVELS = 5
CONTOUR_GRID_N = 200

# =========================
# GraphPad-like style
# =========================
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.linewidth": 1.2,
    "axes.grid": False,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COLOR_A = "#000080"
COLOR_B = "#028A0F"
COLOR_LINE = "#4D4D4D"
COLOR_REAL = "#4C78A8"
COLOR_RANDOM = "#B0B0B0"


# =========================
# Basic utilities
# =========================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def load_all_pairs(pair_dir, pattern):
    files = glob(os.path.join(pair_dir, pattern), recursive=True)
    if not files:
        raise FileNotFoundError(f"No pairs.csv found in: {pair_dir}")

    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["source_file"] = os.path.basename(f)
        df["source_path"] = f
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True)
    return data, files


def filter_pairs(df):
    out = df.copy()

    if MIN_IOU is not None and "iou_Bscale" in out.columns:
        out = out[out["iou_Bscale"] >= MIN_IOU].copy()

    if MAX_DIST_UM is not None and "dist_um" in out.columns:
        out = out[out["dist_um"] <= MAX_DIST_UM].copy()

    return out


def validate_columns(df, feature_pairs):
    missing = []
    for _, (ca, cb) in feature_pairs.items():
        if ca not in df.columns:
            missing.append(ca)
        if cb not in df.columns:
            missing.append(cb)
    if missing:
        raise ValueError(f"Missing columns: {missing}")


def prepare_feature_matrices(df, feature_pairs):
    feature_names = list(feature_pairs.keys())
    colsA = [feature_pairs[f][0] for f in feature_names]
    colsB = [feature_pairs[f][1] for f in feature_names]

    meta_cols = []
    for c in ["idxA", "idxB", "labelA", "labelB", "source_file", "source_path", "dist_um", "iou_Bscale"]:
        if c in df.columns:
            meta_cols.append(c)

    sub = df[colsA + colsB + meta_cols].copy()
    sub = sub.replace([np.inf, -np.inf], np.nan).dropna()

    A = sub[colsA].to_numpy(float)
    B = sub[colsB].to_numpy(float)

    A_df = pd.DataFrame(A, columns=feature_names, index=sub.index)
    B_df = pd.DataFrame(B, columns=feature_names, index=sub.index)

    return sub, A_df, B_df, feature_names


def get_scaler(kind="standard"):
    if kind.lower() == "robust":
        return RobustScaler()
    return StandardScaler()


def joint_normalize(A_df, B_df, scaler_type="standard"):
    scaler = get_scaler(scaler_type)
    pooled = pd.concat([A_df, B_df], axis=0).to_numpy(float)
    scaler.fit(pooled)

    ZA = scaler.transform(A_df.to_numpy(float))
    ZB = scaler.transform(B_df.to_numpy(float))
    return scaler, ZA, ZB


def run_pca(ZA, ZB, n_components=2):
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    pooled = np.vstack([ZA, ZB])
    proj = pca.fit_transform(pooled)
    PA = proj[:len(ZA)]
    PB = proj[len(ZA):]
    return pca, PA, PB


def euclidean_pair_distance(A, B):
    return np.sqrt(np.sum((A - B) ** 2, axis=1))


def random_pair_distances_equal_n(ZA, ZB, meta_df, seed=42):
    rng = np.random.default_rng(seed)
    n = len(ZA)

    perm = rng.permutation(n)
    same = perm == np.arange(n)
    if np.any(same):
        perm[same] = np.roll(perm[same], 1)

    rand_dist = np.sqrt(np.sum((ZA - ZB[perm]) ** 2, axis=1))

    rows = []
    for i in range(n):
        rows.append({
            "comparison_type": "Random pair",
            "distance": float(rand_dist[i]),
            "real_pair_index": int(i),
            "random_partner_index": int(perm[i]),
            "source_file_A": meta_df.iloc[i]["source_file"] if "source_file" in meta_df.columns else None,
            "source_file_B": meta_df.iloc[perm[i]]["source_file"] if "source_file" in meta_df.columns else None,
        })

    return rand_dist, pd.DataFrame(rows)


def permutation_test(real_dist, ZA, ZB, n_perm=2000, seed=42):
    rng = np.random.default_rng(seed)
    observed = np.mean(real_dist)

    perm_means = np.zeros(n_perm, dtype=float)
    n = len(ZA)

    for k in range(n_perm):
        perm_idx = rng.permutation(n)
        d = np.sqrt(np.sum((ZA - ZB[perm_idx]) ** 2, axis=1))
        perm_means[k] = np.mean(d)

    p_emp = (np.sum(perm_means <= observed) + 1) / (n_perm + 1)
    return observed, perm_means, p_emp


# =========================
# Effect size / percent diff
# =========================
def cliffs_delta(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    nx = len(x)
    ny = len(y)
    if nx == 0 or ny == 0:
        return np.nan

    gt = 0
    lt = 0
    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)

    return float((gt - lt) / (nx * ny))


def cliffs_delta_label(delta):
    if not np.isfinite(delta):
        return "NA"
    ad = abs(delta)
    if ad < 0.147:
        return "negligible"
    elif ad < 0.33:
        return "small"
    elif ad < 0.474:
        return "medium"
    return "large"


def percent_shift_from_median(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if len(x) == 0 or len(y) == 0:
        return np.nan

    med_x = np.median(x)
    med_y = np.median(y)

    denom = abs(med_y) if abs(med_y) > 1e-12 else np.nan
    if not np.isfinite(denom):
        return np.nan

    return float(100.0 * (med_x - med_y) / denom)


def symmetric_percent_difference_from_means(x, y, absolute=False):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if len(x) == 0 or len(y) == 0:
        return np.nan

    mean_x = np.mean(x)
    mean_y = np.mean(y)
    denom = (mean_x + mean_y) / 2.0

    if abs(denom) < 1e-12:
        return np.nan

    val = 100.0 * (mean_x - mean_y) / denom
    if absolute:
        val = abs(val)
    return float(val)


# =========================
# Plot utilities
# =========================
def add_density_contour(ax, x, y, color, levels=5, grid_n=200, linewidth=1.2, alpha=0.9):
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 20:
        return

    values = np.vstack([x, y])
    kde = gaussian_kde(values)

    xmin, xmax = np.min(x), np.max(x)
    ymin, ymax = np.min(y), np.max(y)

    xpad = 0.08 * (xmax - xmin + 1e-12)
    ypad = 0.08 * (ymax - ymin + 1e-12)

    xx, yy = np.meshgrid(
        np.linspace(xmin - xpad, xmax + xpad, grid_n),
        np.linspace(ymin - ypad, ymax + ypad, grid_n)
    )
    coords = np.vstack([xx.ravel(), yy.ravel()])
    zz = kde(coords).reshape(xx.shape)

    zmin, zmax = np.nanmin(zz), np.nanmax(zz)
    levs = np.linspace(zmin + 0.2 * (zmax - zmin), zmax, levels)

    ax.contour(xx, yy, zz, levels=levs, linewidths=linewidth, alpha=alpha, colors=[color], zorder=4)


def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", width=1.2, length=5)


def p_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    return "ns"


# =========================
# PCA / distance plots
# =========================
def plot_pca_overlay(PA, PB, explained, out_path, seed=42):
    fig, ax = plt.subplots(figsize=(6.0, 5.2))

    if USE_DENSITY_CONTOUR:
        add_density_contour(ax, PA[:, 0], PA[:, 1], color=COLOR_A,
                            levels=CONTOUR_LEVELS, grid_n=CONTOUR_GRID_N,
                            linewidth=0.9, alpha=0.45)
        add_density_contour(ax, PB[:, 0], PB[:, 1], color=COLOR_B,
                            levels=CONTOUR_LEVELS, grid_n=CONTOUR_GRID_N,
                            linewidth=0.9, alpha=0.45)

    dfA = pd.DataFrame({"PC1": PA[:, 0], "PC2": PA[:, 1], "color": COLOR_A})
    dfB = pd.DataFrame({"PC1": PB[:, 0], "PC2": PB[:, 1], "color": COLOR_B})
    plot_df = pd.concat([dfA, dfB], ignore_index=True)
    plot_df = plot_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    for _, row in plot_df.iterrows():
        ax.scatter(row["PC1"], row["PC2"], s=POINT_SIZE, alpha=POINT_ALPHA,
                   color=row["color"], edgecolors="none", zorder=2)

    ax.scatter([], [], s=POINT_SIZE * 2, color=COLOR_A, alpha=0.7, label="Method A")
    ax.scatter([], [], s=POINT_SIZE * 2, color=COLOR_B, alpha=0.7, label="Method B")

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
    ax.legend(frameon=False, fontsize=11)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_sampled_pair_lines(PA, PB, explained, out_path, sample_n=120, seed=42):
    rng = np.random.default_rng(seed)
    n = len(PA)
    sample_n = min(sample_n, n)
    idx = rng.choice(n, size=sample_n, replace=False)

    fig, ax = plt.subplots(figsize=(6.0, 5.2))
    ax.scatter(PA[:, 0], PA[:, 1], s=10, alpha=0.16, color=COLOR_A, label="Method A")
    ax.scatter(PB[:, 0], PB[:, 1], s=10, alpha=0.16, color=COLOR_B, label="Method B")

    for i in idx:
        ax.plot([PA[i, 0], PB[i, 0]], [PA[i, 1], PB[i, 1]], lw=0.7, alpha=0.55, color=COLOR_LINE)

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
    ax.legend(frameon=False, fontsize=12)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_real_vs_random_box(real_dist, rand_dist, p_mwu, p_emp, out_path):
    fig, ax = plt.subplots(figsize=(4.8, 5.4))
    data = [real_dist, rand_dist]
    positions = [1, 2]

    bp = ax.boxplot(
        data, positions=positions, widths=0.45, patch_artist=True, showfliers=False,
        medianprops=dict(color="black", linewidth=1.6),
        whiskerprops=dict(color="black", linewidth=1.2),
        capprops=dict(color="black", linewidth=1.2),
        boxprops=dict(linewidth=1.2, color="black"),
    )

    bp["boxes"][0].set_facecolor(COLOR_REAL)
    bp["boxes"][0].set_alpha(0.45)
    bp["boxes"][1].set_facecolor(COLOR_RANDOM)
    bp["boxes"][1].set_alpha(0.65)

    rng = np.random.default_rng(RANDOM_SEED)
    for pos, arr, color in zip(positions, data, [COLOR_REAL, COLOR_RANDOM]):
        x = rng.normal(pos, 0.06, size=len(arr))
        ax.scatter(x, arr, s=7, alpha=0.10, color=color, edgecolors="none")

    y_max = max(np.max(real_dist), np.max(rand_dist))
    y_line = y_max * 1.08
    y_text = y_max * 1.13

    ax.plot([1, 1, 2, 2], [y_line * 0.99, y_line, y_line, y_line * 0.99], color="black", lw=1.0)
    ax.text(1.5, y_text, f"MWU p = {p_mwu:.2e}\nPerm p = {p_emp:.2e}",
            ha="center", va="bottom", fontsize=10)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Real pair", "Random pair"], fontsize=12)
    ax.set_ylabel("Distance in normalized shape space", fontsize=14)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_real_vs_random_violin(real_dist, rand_dist, p_mwu, p_emp, out_path):
    fig, ax = plt.subplots(figsize=(4.8, 5.4))
    data = [real_dist, rand_dist]
    positions = [1, 2]

    vp = ax.violinplot(data, positions=positions, widths=0.7,
                       showmeans=False, showmedians=False, showextrema=False)

    for body, color, alpha in zip(vp["bodies"], [COLOR_REAL, COLOR_RANDOM], [0.55, 0.65]):
        body.set_facecolor(color)
        body.set_edgecolor("black")
        body.set_linewidth(1.0)
        body.set_alpha(alpha)

    for pos, arr in zip(positions, data):
        q1, med, q3 = np.percentile(arr, [25, 50, 75])
        ax.plot([pos, pos], [q1, q3], color="black", lw=2.0, zorder=3)
        ax.plot([pos - 0.12, pos + 0.12], [med, med], color="black", lw=2.0, zorder=4)

    y_max = max(np.max(real_dist), np.max(rand_dist))
    y_line = y_max * 1.08
    y_text = y_max * 1.13

    ax.plot([1, 1, 2, 2], [y_line * 0.99, y_line, y_line, y_line * 0.99], color="black", lw=1.0)
    ax.text(1.5, y_text, f"MWU p = {p_mwu:.2e}\nPerm p = {p_emp:.2e}",
            ha="center", va="bottom", fontsize=10)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Real pair", "Random pair"], fontsize=12)
    ax.set_ylabel("Distance in normalized shape space", fontsize=14)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


def plot_loadings(pca, feature_names, out_pdf, out_csv):
    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_names,
        columns=[f"PC{i+1}" for i in range(pca.components_.shape[0])]
    )
    loadings.to_csv(out_csv, index=True)

    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    x = np.arange(len(feature_names))
    ax.bar(x - 0.18, loadings["PC1"], width=0.36, color=COLOR_A, alpha=0.8, label="PC1")
    ax.bar(x + 0.18, loadings["PC2"], width=0.36, color=COLOR_B, alpha=0.8, label="PC2")

    ax.set_xticks(x)
    ax.set_xticklabels(feature_names, rotation=30, ha="right")
    ax.set_ylabel("Loading", fontsize=14)
    ax.legend(frameon=False, fontsize=11)

    style_axes(ax)
    fig.tight_layout()
    fig.savefig(out_pdf, dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


# =========================
# Save tables
# =========================
def save_pair_level(sub, PA, PB, real_dist, out_csv):
    out = sub.copy()
    out["A_PC1"] = PA[:, 0]
    out["A_PC2"] = PA[:, 1]
    out["B_PC1"] = PB[:, 0]
    out["B_PC2"] = PB[:, 1]
    out["shape_pair_distance"] = real_dist
    out.to_csv(out_csv, index=False)


def save_summary(real_dist, rand_dist, pca, p_mwu, p_emp, out_csv):
    summary = pd.DataFrame([{
        "n_real_pairs": len(real_dist),
        "n_random_pairs": len(rand_dist),
        "real_mean": float(np.mean(real_dist)),
        "real_median": float(np.median(real_dist)),
        "random_mean": float(np.mean(rand_dist)),
        "random_median": float(np.median(rand_dist)),
        "mannwhitney_p": float(p_mwu),
        "permutation_p": float(p_emp),
        "pc1_var_ratio": float(pca.explained_variance_ratio_[0]),
        "pc2_var_ratio": float(pca.explained_variance_ratio_[1]),
    }])
    summary.to_csv(out_csv, index=False)


def save_spss_pair_distance_long(sub, real_dist, random_long_df, out_csv):
    real_rows = []
    for i in range(len(sub)):
        row = {
            "comparison_type": "Real pair",
            "distance": float(real_dist[i]),
            "pair_row_index": int(i),
        }
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                row[c] = sub.iloc[i][c]
        real_rows.append(row)

    df_real = pd.DataFrame(real_rows)
    df_long = pd.concat([df_real, random_long_df], ignore_index=True)
    df_long.to_csv(out_csv, index=False)


def save_spss_pca_scores_long(sub, PA, PB, out_csv):
    rows = []
    for i in range(len(sub)):
        meta = {}
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                meta[c] = sub.iloc[i][c]

        rows.append({
            **meta,
            "pair_row_index": int(i),
            "method": "A",
            "PC1": float(PA[i, 0]),
            "PC2": float(PA[i, 1]),
        })
        rows.append({
            **meta,
            "pair_row_index": int(i),
            "method": "B",
            "PC1": float(PB[i, 0]),
            "PC2": float(PB[i, 1]),
        })

    pd.DataFrame(rows).to_csv(out_csv, index=False)


# =========================
# Unpaired feature distribution analysis
# =========================
def save_featurewise_unpaired_stats(sub, feature_pairs, out_csv):
    rows = []

    for feat, (colA, colB) in feature_pairs.items():
        x = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)
        y = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        if len(x) == 0 or len(y) == 0:
            rows.append({
                "feature": feat,
                "n_A": len(x),
                "n_B": len(y),
                "A_mean": np.nan,
                "B_mean": np.nan,
                "A_median": np.nan,
                "B_median": np.nan,
                "A_std": np.nan,
                "B_std": np.nan,
                "mean_diff_A_minus_B": np.nan,
                "median_diff_A_minus_B": np.nan,
                "median_percent_shift_A_vs_B": np.nan,
                "mean_percent_diff_sym_A_vs_B": np.nan,
                "mean_percent_abs_diff_sym_A_vs_B": np.nan,
                "mwu_stat": np.nan,
                "mwu_p": np.nan,
                "cliffs_delta": np.nan,
                "cliffs_delta_magnitude": "NA",
            })
            continue

        stat, pval = mannwhitneyu(x, y, alternative="two-sided")
        delta = cliffs_delta(x, y)

        rows.append({
            "feature": feat,
            "n_A": int(len(x)),
            "n_B": int(len(y)),
            "A_mean": float(np.mean(x)),
            "B_mean": float(np.mean(y)),
            "A_median": float(np.median(x)),
            "B_median": float(np.median(y)),
            "A_std": float(np.std(x)),
            "B_std": float(np.std(y)),
            "mean_diff_A_minus_B": float(np.mean(x) - np.mean(y)),
            "median_diff_A_minus_B": float(np.median(x) - np.median(y)),
            "median_percent_shift_A_vs_B": percent_shift_from_median(x, y),
            "mean_percent_diff_sym_A_vs_B": symmetric_percent_difference_from_means(x, y, absolute=False),
            "mean_percent_abs_diff_sym_A_vs_B": symmetric_percent_difference_from_means(x, y, absolute=True),
            "mwu_stat": float(stat),
            "mwu_p": float(pval),
            "cliffs_delta": float(delta),
            "cliffs_delta_magnitude": cliffs_delta_label(delta),
        })

    pd.DataFrame(rows).to_csv(out_csv, index=False)


def save_featurewise_unpaired_long(sub, feature_pairs, out_csv):
    rows = []

    for i in range(len(sub)):
        meta = {}
        for c in ["source_file", "source_path", "idxA", "idxB", "labelA", "labelB", "dist_um", "iou_Bscale"]:
            if c in sub.columns:
                meta[c] = sub.iloc[i][c]

        for feat, (colA, colB) in feature_pairs.items():
            rows.append({
                **meta,
                "pair_row_index": int(i),
                "feature": feat,
                "method": "A",
                "value": sub.iloc[i][colA]
            })
            rows.append({
                **meta,
                "pair_row_index": int(i),
                "feature": feat,
                "method": "B",
                "value": sub.iloc[i][colB]
            })

    pd.DataFrame(rows).to_csv(out_csv, index=False)


def plot_featurewise_unpaired_boxplots(sub, feature_pairs, out_dir):
    rng = np.random.default_rng(RANDOM_SEED)

    for feat, (colA, colB) in feature_pairs.items():
        x = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)
        y = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        if len(x) == 0 or len(y) == 0:
            continue

        stat, pval = mannwhitneyu(x, y, alternative="two-sided")
        delta = cliffs_delta(x, y)
        delta_label = cliffs_delta_label(delta)
        med_diff = float(np.median(x) - np.median(y))
        mean_pct = symmetric_percent_difference_from_means(x, y, absolute=False)

        fig, ax = plt.subplots(figsize=(4.8, 6.1))

        bp = ax.boxplot(
            [x, y], positions=[1, 2], widths=0.5, patch_artist=True, showfliers=False,
            medianprops=dict(color="black", linewidth=1.6),
            whiskerprops=dict(color="black", linewidth=1.2),
            capprops=dict(color="black", linewidth=1.2),
            boxprops=dict(linewidth=1.2, color="black"),
        )

        bp["boxes"][0].set_facecolor(COLOR_A)
        bp["boxes"][0].set_alpha(0.45)
        bp["boxes"][1].set_facecolor(COLOR_B)
        bp["boxes"][1].set_alpha(0.45)

        ax.scatter(1 + rng.normal(0, 0.05, size=len(x)), x, s=10, alpha=0.20,
                   color=COLOR_A, edgecolors="none", zorder=2)
        ax.scatter(2 + rng.normal(0, 0.05, size=len(y)), y, s=10, alpha=0.20,
                   color=COLOR_B, edgecolors="none", zorder=2)

        ymax = np.nanmax(np.r_[x, y])
        ymin = np.nanmin(np.r_[x, y])
        yr = ymax - ymin if ymax > ymin else 1.0
        yline = ymax + 0.08 * yr
        ytext = ymax + 0.13 * yr

        ax.plot([1, 1, 2, 2], [yline - 0.01 * yr, yline, yline, yline - 0.01 * yr],
                color="black", lw=1.0)
        ax.text(
            1.5, ytext,
            f"MWU p = {pval:.2e}\nCliff's δ = {delta:.3f} ({delta_label})\nMedian diff = {med_diff:.4g}\nSym mean % diff = {mean_pct:.3g}%",
            ha="center", va="bottom", fontsize=10
        )

        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Method A", "Method B"], fontsize=12)
        ax.set_ylabel(feat, fontsize=14)

        style_axes(ax)
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, f"unpaired_boxplot_{feat}_A_vs_B.pdf"),
                    dpi=FIG_DPI, bbox_inches="tight", transparent=True)
        plt.close(fig)


def plot_featurewise_unpaired_violinplots(sub, feature_pairs, out_dir):
    for feat, (colA, colB) in feature_pairs.items():
        x = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)
        y = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        if len(x) == 0 or len(y) == 0:
            continue

        stat, pval = mannwhitneyu(x, y, alternative="two-sided")
        delta = cliffs_delta(x, y)
        delta_label = cliffs_delta_label(delta)
        med_diff = float(np.median(x) - np.median(y))
        mean_pct = symmetric_percent_difference_from_means(x, y, absolute=False)

        fig, ax = plt.subplots(figsize=(4.8, 6.1))
        data = [x, y]
        positions = [1, 2]

        vp = ax.violinplot(data, positions=positions, widths=0.7,
                           showmeans=False, showmedians=False, showextrema=False)

        for body, color, alpha in zip(vp["bodies"], [COLOR_A, COLOR_B], [0.45, 0.45]):
            body.set_facecolor(color)
            body.set_edgecolor("black")
            body.set_linewidth(1.0)
            body.set_alpha(alpha)

        for pos, arr in zip(positions, data):
            q1, med, q3 = np.percentile(arr, [25, 50, 75])
            ax.plot([pos, pos], [q1, q3], color="black", lw=2.0, zorder=3)
            ax.plot([pos - 0.12, pos + 0.12], [med, med], color="black", lw=2.0, zorder=4)

        ymax = np.nanmax(np.r_[x, y])
        ymin = np.nanmin(np.r_[x, y])
        yr = ymax - ymin if ymax > ymin else 1.0
        yline = ymax + 0.08 * yr
        ytext = ymax + 0.13 * yr

        ax.plot([1, 1, 2, 2], [yline - 0.01 * yr, yline, yline, yline - 0.01 * yr],
                color="black", lw=1.0)
        ax.text(
            1.5, ytext,
            f"MWU p = {pval:.2e}\nCliff's δ = {delta:.3f} ({delta_label})\nMedian diff = {med_diff:.4g}\nSym mean % diff = {mean_pct:.3g}%",
            ha="center", va="bottom", fontsize=10
        )

        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Method A", "Method B"], fontsize=12)
        ax.set_ylabel(feat, fontsize=14)

        style_axes(ax)
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, f"unpaired_violin_{feat}_A_vs_B.pdf"),
                    dpi=FIG_DPI, bbox_inches="tight", transparent=True)
        plt.close(fig)


# =========================
# Pair scatter: B on x, A on y
# =========================
def plot_featurewise_pair_scatter_with_identity(sub, feature_pairs, out_dir):
    rng = np.random.default_rng(RANDOM_SEED)

    for feat, (colA, colB) in feature_pairs.items():
        y = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)  # Method A -> y
        x = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)  # Method B -> x

        valid = np.isfinite(x) & np.isfinite(y)
        x = x[valid]
        y = y[valid]

        if len(x) < 3:
            continue

        try:
            pear_r, pear_p = pearsonr(x, y)
        except Exception:
            pear_r, pear_p = np.nan, np.nan

        try:
            spear_r, spear_p = spearmanr(x, y)
        except Exception:
            spear_r, spear_p = np.nan, np.nan

        try:
            m, b = np.polyfit(x, y, 1)
            yhat = m * x + b
            ss_res = np.sum((y - yhat) ** 2)
            ss_tot = np.sum((y - np.mean(y)) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        except Exception:
            m, b, r2 = np.nan, np.nan, np.nan

        rmse_identity = np.sqrt(np.mean((y - x) ** 2)) if len(x) > 0 else np.nan

        fig, ax = plt.subplots(figsize=(5.2, 5.2))

        ax.scatter(
            x, y,
            s=18, alpha=0.28, color=COLOR_A,
            edgecolors="none", zorder=2
        )

        vmin = np.nanmin(np.r_[x, y])
        vmax = np.nanmax(np.r_[x, y])
        vr = vmax - vmin
        if vr <= 0:
            vr = 1.0
        pad = 0.05 * vr
        lo = vmin - pad
        hi = vmax + pad

        # y=x
        ax.plot([lo, hi], [lo, hi], ls="--", lw=1.2, color="black", alpha=0.8, zorder=1)

        # fit line
        if np.isfinite(m) and np.isfinite(b):
            xx = np.linspace(lo, hi, 200)
            yy = m * xx + b
            ax.plot(xx, yy, lw=1.2, color=COLOR_B, alpha=0.9, zorder=3)

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)

        ax.set_xlabel(f"Method B ({feat})", fontsize=14)
        ax.set_ylabel(f"Method A ({feat})", fontsize=14)

        txt = (
            f"Pearson r = {pear_r:.3f}\n"
            f"Spearman ρ = {spear_r:.3f}\n"
            f"R² = {r2:.3f}\n"
            f"Slope = {m:.3f}\n"
            f"RMSE vs y=x = {rmse_identity:.4g}"
        )
        ax.text(
            0.03, 0.97, txt,
            transform=ax.transAxes,
            ha="left", va="top", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", alpha=0.9)
        )

        style_axes(ax)
        fig.tight_layout()
        fig.savefig(
            os.path.join(out_dir, f"pair_scatter_identity_{feat}_A_vs_B.pdf"),
            dpi=FIG_DPI,
            bbox_inches="tight",
            transparent=True
        )
        plt.close(fig)


# =========================
# Main
# =========================
def main():
    ensure_dir(OUT_DIR)

    df, files = load_all_pairs(PAIR_CSV_DIR, GLOB_PATTERN)
    print(f"Loaded files: {len(files)}")
    print(f"Rows before filtering: {len(df)}")

    validate_columns(df, FEATURE_PAIRS)
    df = filter_pairs(df)
    print(f"Rows after filtering: {len(df)}")

    sub, A_df, B_df, feature_names = prepare_feature_matrices(df, FEATURE_PAIRS)
    print(f"Valid rows for PCA/similarity: {len(sub)}")

    scaler, ZA, ZB = joint_normalize(A_df, B_df, scaler_type=SCALER_TYPE)
    pca, PA, PB = run_pca(ZA, ZB, n_components=2)

    real_dist = euclidean_pair_distance(ZA, ZB)
    rand_dist, random_long_df = random_pair_distances_equal_n(ZA, ZB, sub, seed=RANDOM_SEED)

    _, p_mwu = mannwhitneyu(real_dist, rand_dist, alternative="less")
    _, perm_means, p_emp = permutation_test(real_dist, ZA, ZB, n_perm=N_PERMUTATIONS, seed=RANDOM_SEED)

    save_pair_level(sub, PA, PB, real_dist, os.path.join(OUT_DIR, "pair_level_shape_similarity.csv"))
    save_summary(real_dist, rand_dist, pca, p_mwu, p_emp, os.path.join(OUT_DIR, "summary_shape_similarity.csv"))

    save_spss_pair_distance_long(sub, real_dist, random_long_df,
                                 os.path.join(OUT_DIR, "spss_pair_distance_long.csv"))
    save_spss_pca_scores_long(sub, PA, PB,
                              os.path.join(OUT_DIR, "spss_pca_scores_long.csv"))

    save_featurewise_unpaired_stats(
        sub, FEATURE_PAIRS,
        os.path.join(OUT_DIR, "featurewise_unpaired_stats.csv")
    )
    save_featurewise_unpaired_long(
        sub, FEATURE_PAIRS,
        os.path.join(OUT_DIR, "spss_featurewise_unpaired_long.csv")
    )

    plot_pca_overlay(PA, PB, pca.explained_variance_ratio_,
                     os.path.join(OUT_DIR, "pca_overlay_A_vs_B.pdf"))
    plot_sampled_pair_lines(PA, PB, pca.explained_variance_ratio_,
                            os.path.join(OUT_DIR, "pca_pairs_connected.pdf"),
                            sample_n=SAMPLED_PAIR_LINES, seed=RANDOM_SEED)
    plot_real_vs_random_box(real_dist, rand_dist, p_mwu, p_emp,
                            os.path.join(OUT_DIR, "real_vs_random_pair_distance_box.pdf"))
    plot_real_vs_random_violin(real_dist, rand_dist, p_mwu, p_emp,
                               os.path.join(OUT_DIR, "real_vs_random_pair_distance_violin.pdf"))
    plot_loadings(pca, feature_names,
                  os.path.join(OUT_DIR, "pca_loadings.pdf"),
                  os.path.join(OUT_DIR, "pca_loadings.csv"))

    plot_featurewise_unpaired_boxplots(sub, FEATURE_PAIRS, OUT_DIR)
    plot_featurewise_unpaired_violinplots(sub, FEATURE_PAIRS, OUT_DIR)
    plot_featurewise_pair_scatter_with_identity(sub, FEATURE_PAIRS, OUT_DIR)

    print("Saved:")
    print(" - pair_level_shape_similarity.csv")
    print(" - summary_shape_similarity.csv")
    print(" - spss_pair_distance_long.csv")
    print(" - spss_pca_scores_long.csv")
    print(" - featurewise_unpaired_stats.csv")
    print(" - spss_featurewise_unpaired_long.csv")
    print(" - pca_overlay_A_vs_B.pdf")
    print(" - pca_pairs_connected.pdf")
    print(" - real_vs_random_pair_distance_box.pdf")
    print(" - real_vs_random_pair_distance_violin.pdf")
    print(" - pca_loadings.pdf")
    print(" - pca_loadings.csv")

    for feat in FEATURE_PAIRS.keys():
        print(f" - unpaired_boxplot_{feat}_A_vs_B.pdf")
        print(f" - unpaired_violin_{feat}_A_vs_B.pdf")
        print(f" - pair_scatter_identity_{feat}_A_vs_B.pdf")


if __name__ == "__main__":
    main()

Loaded files: 6
Rows before filtering: 1071
Rows after filtering: 1071
Valid rows for PCA/similarity: 1071
Saved:
 - pair_level_shape_similarity.csv
 - summary_shape_similarity.csv
 - spss_pair_distance_long.csv
 - spss_pca_scores_long.csv
 - featurewise_unpaired_stats.csv
 - spss_featurewise_unpaired_long.csv
 - pca_overlay_A_vs_B.pdf
 - pca_pairs_connected.pdf
 - real_vs_random_pair_distance_box.pdf
 - real_vs_random_pair_distance_violin.pdf
 - pca_loadings.pdf
 - pca_loadings.csv
 - unpaired_boxplot_aspect_A_vs_B.pdf
 - unpaired_violin_aspect_A_vs_B.pdf
 - pair_scatter_identity_aspect_A_vs_B.pdf
 - unpaired_boxplot_circularity_A_vs_B.pdf
 - unpaired_violin_circularity_A_vs_B.pdf
 - pair_scatter_identity_circularity_A_vs_B.pdf
 - unpaired_boxplot_solidity_A_vs_B.pdf
 - unpaired_violin_solidity_A_vs_B.pdf
 - pair_scatter_identity_solidity_A_vs_B.pdf
 - unpaired_boxplot_eccentricity_A_vs_B.pdf
 - unpaired_violin_eccentricity_A_vs_B.pdf
 - pair_scatter_identity_eccentricity_A_vs_B.pdf
 

In [3]:
# -*- coding: utf-8 -*-

import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from scipy.stats import mannwhitneyu, gaussian_kde, pearsonr, spearmanr

# =========================
# USER SETTINGS
# =========================
PAIR_CSV_DIR = r"C:\Users\oxfil\Pair_for_PCA"
GLOB_PATTERN = "*.csv"
OUT_DIR = r".\shape_pair_pca_analysis416"

FEATURE_PAIRS = {
    "aspect": ("aspectA", "aspectB"),
    "circularity": ("circA", "circB"),
    "solidity": ("solidityA", "solidityB"),
    "eccentricity": ("eccentricityA", "eccentricityB"),
    "extent": ("extentA", "extentB"),
}

SCALER_TYPE = "standard"   # "standard" or "robust"

MIN_IOU = None
MAX_DIST_UM = 100

N_PERMUTATIONS = 2000
RANDOM_SEED = 80

FIG_DPI = 300
POINT_SIZE = 16
POINT_ALPHA = 0.28
SAMPLED_PAIR_LINES = 120

USE_DENSITY_CONTOUR = True
CONTOUR_LEVELS = 5
CONTOUR_GRID_N = 200

# =========================
# Manual axis limits for pair scatter
# x = Method B, y = Method A
# None 이면 자동 범위
# =========================
PAIR_SCATTER_LIMITS = {
    "aspect": (0.8, 5.0),
    "circularity": (0.2, 1.0),
    "solidity": (0.6, 1.0),
    "eccentricity": (0.0, 1.0),
    "extent": (0.0, 1.0),
}

# =========================
# GraphPad-like style
# =========================
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.linewidth": 1.2,
    "axes.grid": False,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COLOR_A = "#000080"
COLOR_B = "#028A0F"
COLOR_LINE = "#4D4D4D"
COLOR_REAL = "#4C78A8"
COLOR_RANDOM = "#B0B0B0"


# =========================
# Basic utilities
# =========================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def load_all_pairs(pair_dir, pattern):
    files = glob(os.path.join(pair_dir, pattern), recursive=True)
    if not files:
        raise FileNotFoundError(f"No pairs.csv found in: {pair_dir}")

    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["source_file"] = os.path.basename(f)
        df["source_path"] = f
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True)
    return data, files


def filter_pairs(df):
    out = df.copy()

    if MIN_IOU is not None and "iou_Bscale" in out.columns:
        out = out[out["iou_Bscale"] >= MIN_IOU].copy()

    if MAX_DIST_UM is not None and "dist_um" in out.columns:
        out = out[out["dist_um"] <= MAX_DIST_UM].copy()

    return out


def validate_columns(df, feature_pairs):
    missing = []
    for _, (ca, cb) in feature_pairs.items():
        if ca not in df.columns:
            missing.append(ca)
        if cb not in df.columns:
            missing.append(cb)
    if missing:
        raise ValueError(f"Missing columns: {missing}")


def prepare_feature_matrices(df, feature_pairs):
    feature_names = list(feature_pairs.keys())
    colsA = [feature_pairs[f][0] for f in feature_names]
    colsB = [feature_pairs[f][1] for f in feature_names]

    meta_cols = []
    for c in ["idxA", "idxB", "labelA", "labelB", "source_file", "source_path", "dist_um", "iou_Bscale"]:
        if c in df.columns:
            meta_cols.append(c)

    sub = df[colsA + colsB + meta_cols].copy()
    sub = sub.replace([np.inf, -np.inf], np.nan).dropna()

    A = sub[colsA].to_numpy(float)
    B = sub[colsB].to_numpy(float)

    A_df = pd.DataFrame(A, columns=feature_names, index=sub.index)
    B_df = pd.DataFrame(B, columns=feature_names, index=sub.index)

    return sub, A_df, B_df, feature_names


def get_scaler(kind="standard"):
    if kind.lower() == "robust":
        return RobustScaler()
    return StandardScaler()


def joint_normalize(A_df, B_df, scaler_type="standard"):
    scaler = get_scaler(scaler_type)
    pooled = pd.concat([A_df, B_df], axis=0).to_numpy(float)
    scaler.fit(pooled)

    ZA = scaler.transform(A_df.to_numpy(float))
    ZB = scaler.transform(B_df.to_numpy(float))
    return scaler, ZA, ZB


def run_pca(ZA, ZB, n_components=2):
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    pooled = np.vstack([ZA, ZB])
    proj = pca.fit_transform(pooled)
    PA = proj[:len(ZA)]
    PB = proj[len(ZA):]
    return pca, PA, PB


def euclidean_pair_distance(A, B):
    return np.sqrt(np.sum((A - B) ** 2, axis=1))


def random_pair_distances_equal_n(ZA, ZB, meta_df, seed=42):
    rng = np.random.default_rng(seed)
    n = len(ZA)

    perm = rng.permutation(n)
    same = perm == np.arange(n)
    if np.any(same):
        perm[same] = np.roll(perm[same], 1)

    rand_dist = np.sqrt(np.sum((ZA - ZB[perm]) ** 2, axis=1))

    rows = []
    for i in range(n):
        rows.append({
            "comparison_type": "Random pair",
            "distance": float(rand_dist[i]),
            "real_pair_index": int(i),
            "random_partner_index": int(perm[i]),
            "source_file_A": meta_df.iloc[i]["source_file"] if "source_file" in meta_df.columns else None,
            "source_file_B": meta_df.iloc[perm[i]]["source_file"] if "source_file" in meta_df.columns else None,
        })

    return rand_dist, pd.DataFrame(rows)


def permutation_test(real_dist, ZA, ZB, n_perm=2000, seed=42):
    rng = np.random.default_rng(seed)
    observed = np.mean(real_dist)

    perm_means = np.zeros(n_perm, dtype=float)
    n = len(ZA)

    for k in range(n_perm):
        perm_idx = rng.permutation(n)
        d = np.sqrt(np.sum((ZA - ZB[perm_idx]) ** 2, axis=1))
        perm_means[k] = np.mean(d)

    p_emp = (np.sum(perm_means <= observed) + 1) / (n_perm + 1)
    return observed, perm_means, p_emp


# =========================
# Effect size / percent diff
# =========================
def cliffs_delta(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    nx = len(x)
    ny = len(y)
    if nx == 0 or ny == 0:
        return np.nan

    gt = 0
    lt = 0
    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)

    return float((gt - lt) / (nx * ny))


def cliffs_delta_label(delta):
    if not np.isfinite(delta):
        return "NA"
    ad = abs(delta)
    if ad < 0.147:
        return "negligible"
    elif ad < 0.33:
        return "small"
    elif ad < 0.474:
        return "medium"
    return "large"


def percent_shift_from_median(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if len(x) == 0 or len(y) == 0:
        return np.nan

    med_x = np.median(x)
    med_y = np.median(y)

    denom = abs(med_y) if abs(med_y) > 1e-12 else np.nan
    if not np.isfinite(denom):
        return np.nan

    return float(100.0 * (med_x - med_y) / denom)


def symmetric_percent_difference_from_means(x, y, absolute=False):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if len(x) == 0 or len(y) == 0:
        return np.nan

    mean_x = np.mean(x)
    mean_y = np.mean(y)
    denom = (mean_x + mean_y) / 2.0

    if abs(denom) < 1e-12:
        return np.nan

    val = 100.0 * (mean_x - mean_y) / denom
    if absolute:
        val = abs(val)
    return float(val)


# =========================
# Plot utilities
# =========================
def add_density_contour(ax, x, y, color, levels=5, grid_n=200, linewidth=1.2, alpha=0.9):
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 20:
        return

    values = np.vstack([x, y])
    kde = gaussian_kde(values)

    xmin, xmax = np.min(x), np.max(x)
    ymin, ymax = np.min(y), np.max(y)

    xpad = 0.08 * (xmax - xmin + 1e-12)
    ypad = 0.08 * (ymax - ymin + 1e-12)

    xx, yy = np.meshgrid(
        np.linspace(xmin - xpad, xmax + xpad, grid_n),
        np.linspace(ymin - ypad, ymax + ypad, grid_n)
    )
    coords = np.vstack([xx.ravel(), yy.ravel()])
    zz = kde(coords).reshape(xx.shape)

    zmin, zmax = np.nanmin(zz), np.nanmax(zz)
    levs = np.linspace(zmin + 0.2 * (zmax - zmin), zmax, levels)

    ax.contour(xx, yy, zz, levels=levs, linewidths=linewidth, alpha=alpha, colors=[color], zorder=4)


def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", width=1.2, length=5)


# =========================
# Pair scatter: B on x, A on y
# =========================
def plot_featurewise_pair_scatter_with_identity(sub, feature_pairs, out_dir):
    """
    x-axis = Method B
    y-axis = Method A
    matched pairs scatter + y=x line
    축 범위는 PAIR_SCATTER_LIMITS에서 수동 지정 가능
    """
    for feat, (colA, colB) in feature_pairs.items():
        y = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)  # Method A
        x = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)  # Method B

        valid = np.isfinite(x) & np.isfinite(y)
        x = x[valid]
        y = y[valid]

        if len(x) < 3:
            continue

        try:
            pear_r, _ = pearsonr(x, y)
        except Exception:
            pear_r = np.nan

        try:
            spear_r, _ = spearmanr(x, y)
        except Exception:
            spear_r = np.nan

        try:
            m, b = np.polyfit(x, y, 1)
            yhat = m * x + b
            ss_res = np.sum((y - yhat) ** 2)
            ss_tot = np.sum((y - np.mean(y)) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        except Exception:
            m, b, r2 = np.nan, np.nan, np.nan

        rmse_identity = np.sqrt(np.mean((y - x) ** 2)) if len(x) > 0 else np.nan

        fig, ax = plt.subplots(figsize=(5.2, 5.2))

        ax.scatter(
            x, y,
            s=18, alpha=0.28, color=COLOR_A,
            edgecolors="none", zorder=2
        )

        # ===== manual axis limits =====
        manual_lim = PAIR_SCATTER_LIMITS.get(feat, None)

        if manual_lim is not None:
            lo, hi = manual_lim
        else:
            vmin = np.nanmin(np.r_[x, y])
            vmax = np.nanmax(np.r_[x, y])
            vr = vmax - vmin
            if vr <= 0:
                vr = 1.0
            pad = 0.05 * vr
            lo = vmin - pad
            hi = vmax + pad

        # identity line
        ax.plot([lo, hi], [lo, hi], ls="--", lw=1.2, color="black", alpha=0.8, zorder=1)

        # fitted line
        if np.isfinite(m) and np.isfinite(b):
            xx = np.linspace(lo, hi, 200)
            yy = m * xx + b
            ax.plot(xx, yy, lw=1.2, color=COLOR_B, alpha=0.9, zorder=3)

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)

        ax.set_xlabel(f"Method B ({feat})", fontsize=14)
        ax.set_ylabel(f"Method A ({feat})", fontsize=14)

        txt = (
            f"Pearson r = {pear_r:.3f}\n"
            f"Spearman ρ = {spear_r:.3f}\n"
            f"R² = {r2:.3f}\n"
            f"Slope = {m:.3f}\n"
            f"RMSE vs y=x = {rmse_identity:.4g}"
        )
        ax.text(
            0.03, 0.97, txt,
            transform=ax.transAxes,
            ha="left", va="top", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", alpha=0.9)
        )

        style_axes(ax)
        fig.tight_layout()
        fig.savefig(
            os.path.join(out_dir, f"pair_scatter_identity_{feat}_A_vs_B.pdf"),
            dpi=FIG_DPI,
            bbox_inches="tight",
            transparent=True
        )
        plt.close(fig)


# =========================
# Main
# =========================
def main():
    ensure_dir(OUT_DIR)

    df, files = load_all_pairs(PAIR_CSV_DIR, GLOB_PATTERN)
    print(f"Loaded files: {len(files)}")
    print(f"Rows before filtering: {len(df)}")

    validate_columns(df, FEATURE_PAIRS)
    df = filter_pairs(df)
    print(f"Rows after filtering: {len(df)}")

    sub, A_df, B_df, feature_names = prepare_feature_matrices(df, FEATURE_PAIRS)
    print(f"Valid rows for PCA/similarity: {len(sub)}")

    scaler, ZA, ZB = joint_normalize(A_df, B_df, scaler_type=SCALER_TYPE)
    pca, PA, PB = run_pca(ZA, ZB, n_components=2)

    # scatter only
    plot_featurewise_pair_scatter_with_identity(sub, FEATURE_PAIRS, OUT_DIR)

    print("Saved pair scatter plots with manual axis limits.")
    for feat in FEATURE_PAIRS.keys():
        print(f" - pair_scatter_identity_{feat}_A_vs_B.pdf")


if __name__ == "__main__":
    main()

Loaded files: 6
Rows before filtering: 1071
Rows after filtering: 1071
Valid rows for PCA/similarity: 1071
Saved pair scatter plots with manual axis limits.
 - pair_scatter_identity_aspect_A_vs_B.pdf
 - pair_scatter_identity_circularity_A_vs_B.pdf
 - pair_scatter_identity_solidity_A_vs_B.pdf
 - pair_scatter_identity_eccentricity_A_vs_B.pdf
 - pair_scatter_identity_extent_A_vs_B.pdf


In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# USER SETTINGS
# =========================
LOADINGS_CSV = r".\shape_pair_pca_analysis416\pca_loadings.csv"
OUT_PDF = r".\shape_pair_pca_analysis416\pca_loadings_colormap_table_horizontal.pdf"

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.linewidth": 1.2,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def plot_loadings_colormap_from_csv_horizontal(csv_path, out_pdf):
    loadings = pd.read_csv(csv_path, index_col=0)

    # 가로형으로 바꾸기: 행=PC, 열=feature
    loadings = loadings.T
    vals = loadings.to_numpy(float)

    fig_w = 1.6 + 1.2 * vals.shape[1]
    fig_h = 1.8 + 0.9 * vals.shape[0]
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    vmax = np.nanmax(np.abs(vals))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    im = ax.imshow(vals, cmap="coolwarm", vmin=-vmax, vmax=vmax, aspect="auto")

    # labels
    ax.set_xticks(np.arange(vals.shape[1]))
    ax.set_xticklabels(loadings.columns, rotation=30, ha="right", fontsize=12)
    ax.set_yticks(np.arange(vals.shape[0]))
    ax.set_yticklabels(loadings.index, fontsize=12)

    # cell text
    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = vals[i, j]
            txt_color = "white" if abs(v) > 0.5 * vmax else "black"
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=11, color=txt_color)

    # table-like borders
    ax.set_xticks(np.arange(-0.5, vals.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, vals.shape[0], 1), minor=True)
    ax.grid(which="minor", color="black", linestyle="-", linewidth=0.8)
    ax.tick_params(which="minor", bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
    cbar.set_label("Loading", fontsize=12)

    fig.tight_layout()
    fig.savefig(out_pdf, dpi=300, bbox_inches="tight", transparent=True)
    plt.close(fig)

plot_loadings_colormap_from_csv_horizontal(LOADINGS_CSV, OUT_PDF)
print(f"Saved: {OUT_PDF}")

Saved: .\shape_pair_pca_analysis416\pca_loadings_colormap_table_horizontal.pdf


 For 1D morphological feature

In [6]:
# -*- coding: utf-8 -*-

import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from scipy.stats import pearsonr, spearmanr, gaussian_kde


# =========================
# USER SETTINGS
# =========================
PAIR_CSV_DIR = r"C:\Users\oxfil\pair_10um_CMOS"
GLOB_PATTERN = "*.csv"
OUT_DIR = r".\skeleton_pair_analysis"

SCALER_TYPE = "standard"   # "standard" or "robust"

MIN_IOU = None
MAX_DIST_UM = 100

FIG_DPI = 300
POINT_SIZE = 18
POINT_ALPHA = 0.30

USE_DENSITY_CONTOUR = True
CONTOUR_LEVELS = 5
CONTOUR_GRID_N = 200

# ---------------------------------
# 1D / relative descriptor pairs
# key = display name
# value = (A column, B column)
# ---------------------------------
FEATURE_PAIRS = {
    "skeleton_length_um": ("skel_lenA_um", "skel_lenB_um"),
    "linearity_index": ("linearityA", "linearityB"),
    "major_axis_um": ("majorA_um", "majorB_um"),
    "minor_axis_um": ("minorA_um", "minorB_um"),
}

# Optional: include area as a reference comparison
INCLUDE_AREA_REFERENCE = True
AREA_PAIR = {"area_um2": ("areaA_um2", "areaB_um2")}

# If True, only use masks already marked as line-like
USE_ONLY_LINE_LIKE = False

# If True, require all chosen 1D features to be finite
REQUIRE_FINITE_1D = True

# =========================
# Manual axis limits for pair scatter
# x = Method B, y = Method A
# None -> auto
# =========================
# PAIR_SCATTER_LIMITS = {
#     "skeleton_length_um": None,
#     "linearity_index": None,
#     "major_axis_um": None,
#     "minor_axis_um": None,
#     "area_um2": None,
# }

# Example:
PAIR_SCATTER_LIMITS = {
    "skeleton_length_um": (0, 250),
    "linearity_index": (0, 10),
    "major_axis_um": (0, 80),
    "minor_axis_um": (0, 30),
    "area_um2": (0, 6000),
}

# =========================
# GraphPad-like style
# =========================
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.linewidth": 1.2,
    "axes.grid": False,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COLOR_A = "#000080"
COLOR_B = "#028A0F"
COLOR_PCA_A = "#4C78A8"
COLOR_PCA_B = "#E45756"


# =========================
# Basic utilities
# =========================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def load_all_pairs(pair_dir, pattern):
    files = glob(os.path.join(pair_dir, pattern), recursive=True)
    if not files:
        raise FileNotFoundError(f"No csv files found in: {pair_dir}")

    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["source_file"] = os.path.basename(f)
        df["source_path"] = f
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True)
    return data, files


def add_um_columns_if_needed(df):
    """
    If major/minor axis are only stored in pixel units, convert them to um.
    Existing *_um columns are preserved.
    """
    out = df.copy()

    if "majorA_um" not in out.columns and "majorA" in out.columns and "areaA_um2" in out.columns and "areaA" in out.columns:
        px_per_um_A = np.sqrt(out["areaA"] / (out["areaA_um2"] + 1e-12))
        out["majorA_um"] = out["majorA"] / (px_per_um_A + 1e-12)

    if "majorB_um" not in out.columns and "majorB" in out.columns and "areaB_um2" in out.columns and "areaB" in out.columns:
        px_per_um_B = np.sqrt(out["areaB"] / (out["areaB_um2"] + 1e-12))
        out["majorB_um"] = out["majorB"] / (px_per_um_B + 1e-12)

    if "minorA_um" not in out.columns and "minorA" in out.columns and "areaA_um2" in out.columns and "areaA" in out.columns:
        px_per_um_A = np.sqrt(out["areaA"] / (out["areaA_um2"] + 1e-12))
        out["minorA_um"] = out["minorA"] / (px_per_um_A + 1e-12)

    if "minorB_um" not in out.columns and "minorB" in out.columns and "areaB_um2" in out.columns and "areaB" in out.columns:
        px_per_um_B = np.sqrt(out["areaB"] / (out["areaB_um2"] + 1e-12))
        out["minorB_um"] = out["minorB"] / (px_per_um_B + 1e-12)

    return out


def filter_pairs(df):
    out = df.copy()

    if MIN_IOU is not None and "iou_Bscale" in out.columns:
        out = out[out["iou_Bscale"] >= MIN_IOU].copy()

    if MAX_DIST_UM is not None and "dist_um" in out.columns:
        out = out[out["dist_um"] <= MAX_DIST_UM].copy()

    if USE_ONLY_LINE_LIKE:
        condA = np.ones(len(out), dtype=bool)
        condB = np.ones(len(out), dtype=bool)

        if "line_like_A" in out.columns:
            condA &= out["line_like_A"].astype(bool).to_numpy()
        if "line_like_B" in out.columns:
            condB &= out["line_like_B"].astype(bool).to_numpy()

        out = out[condA & condB].copy()

    return out


def validate_columns(df, feature_pairs):
    missing = []
    for _, (ca, cb) in feature_pairs.items():
        if ca not in df.columns:
            missing.append(ca)
        if cb not in df.columns:
            missing.append(cb)
    if missing:
        raise ValueError(f"Missing columns: {missing}")


def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", width=1.2, length=5)


def get_scaler(kind="standard"):
    if kind.lower() == "robust":
        return RobustScaler()
    return StandardScaler()


def add_density_contour(ax, x, y, color, levels=5, grid_n=200, linewidth=1.2, alpha=0.9):
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 20:
        return

    values = np.vstack([x, y])
    kde = gaussian_kde(values)

    xmin, xmax = np.min(x), np.max(x)
    ymin, ymax = np.min(y), np.max(y)

    xpad = 0.08 * (xmax - xmin + 1e-12)
    ypad = 0.08 * (ymax - ymin + 1e-12)

    xx, yy = np.meshgrid(
        np.linspace(xmin - xpad, xmax + xpad, grid_n),
        np.linspace(ymin - ypad, ymax + ypad, grid_n)
    )
    coords = np.vstack([xx.ravel(), yy.ravel()])
    zz = kde(coords).reshape(xx.shape)

    zmin, zmax = np.nanmin(zz), np.nanmax(zz)
    levs = np.linspace(zmin + 0.2 * (zmax - zmin), zmax, levels)
    ax.contour(xx, yy, zz, levels=levs, linewidths=linewidth, alpha=alpha, colors=[color], zorder=4)


# =========================
# Prepare feature matrices
# =========================
def prepare_feature_matrices(df, feature_pairs):
    feature_names = list(feature_pairs.keys())
    colsA = [feature_pairs[f][0] for f in feature_names]
    colsB = [feature_pairs[f][1] for f in feature_names]

    meta_cols = []
    for c in [
        "idxA", "idxB", "labelA", "labelB", "source_file", "source_path",
        "dist_um", "iou_Bscale",
        "line_like_A", "line_like_B",
        "shape_reliable_2d_A", "shape_reliable_2d_B"
    ]:
        if c in df.columns:
            meta_cols.append(c)

    sub = df[colsA + colsB + meta_cols].copy()
    sub = sub.replace([np.inf, -np.inf], np.nan)

    if REQUIRE_FINITE_1D:
        sub = sub.dropna(subset=colsA + colsB)

    A = sub[colsA].to_numpy(float)
    B = sub[colsB].to_numpy(float)

    A_df = pd.DataFrame(A, columns=feature_names, index=sub.index)
    B_df = pd.DataFrame(B, columns=feature_names, index=sub.index)

    return sub, A_df, B_df, feature_names


def joint_normalize(A_df, B_df, scaler_type="standard"):
    scaler = get_scaler(scaler_type)
    pooled = pd.concat([A_df, B_df], axis=0).to_numpy(float)
    scaler.fit(pooled)

    ZA = scaler.transform(A_df.to_numpy(float))
    ZB = scaler.transform(B_df.to_numpy(float))
    return scaler, ZA, ZB


def run_pca(ZA, ZB, n_components=2):
    pooled = np.vstack([ZA, ZB])
    max_comp = min(2, pooled.shape[1], pooled.shape[0])
    if max_comp < 1:
        raise ValueError("Not enough data/features for PCA.")

    pca = PCA(n_components=max_comp, random_state=80)
    proj = pca.fit_transform(pooled)

    PA = proj[:len(ZA)]
    PB = proj[len(ZA):]
    return pca, PA, PB


# =========================
# Pair scatter
# =========================
def plot_featurewise_pair_scatter_with_identity(sub, feature_pairs, out_dir):
    for feat, (colA, colB) in feature_pairs.items():
        y = pd.to_numeric(sub[colA], errors="coerce").to_numpy(float)  # Method A
        x = pd.to_numeric(sub[colB], errors="coerce").to_numpy(float)  # Method B

        valid = np.isfinite(x) & np.isfinite(y)
        x = x[valid]
        y = y[valid]

        if len(x) < 3:
            print(f"[SKIP] {feat}: not enough valid points")
            continue

        try:
            pear_r, _ = pearsonr(x, y)
        except Exception:
            pear_r = np.nan

        try:
            spear_r, _ = spearmanr(x, y)
        except Exception:
            spear_r = np.nan

        try:
            m, b = np.polyfit(x, y, 1)
            yhat = m * x + b
            ss_res = np.sum((y - yhat) ** 2)
            ss_tot = np.sum((y - np.mean(y)) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        except Exception:
            m, b, r2 = np.nan, np.nan, np.nan

        rmse_identity = np.sqrt(np.mean((y - x) ** 2)) if len(x) > 0 else np.nan

        fig, ax = plt.subplots(figsize=(5.2, 5.2))
        ax.scatter(
            x, y,
            s=POINT_SIZE, alpha=POINT_ALPHA, color=COLOR_A,
            edgecolors="none", zorder=2
        )

        manual_lim = PAIR_SCATTER_LIMITS.get(feat, None)
        if manual_lim is not None:
            lo, hi = manual_lim
        else:
            vmin = np.nanmin(np.r_[x, y])
            vmax = np.nanmax(np.r_[x, y])
            vr = vmax - vmin
            if vr <= 0:
                vr = 1.0
            pad = 0.05 * vr
            lo = vmin - pad
            hi = vmax + pad

        ax.plot([lo, hi], [lo, hi], ls="--", lw=1.2, color="black", alpha=0.8, zorder=1)

        if np.isfinite(m) and np.isfinite(b):
            xx = np.linspace(lo, hi, 200)
            yy = m * xx + b
            ax.plot(xx, yy, lw=1.2, color=COLOR_B, alpha=0.9, zorder=3)

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)

        ax.set_xlabel(f"Method B ({feat})", fontsize=14)
        ax.set_ylabel(f"Method A ({feat})", fontsize=14)

        txt = (
            f"Pearson r = {pear_r:.3f}\n"
            f"Spearman ρ = {spear_r:.3f}\n"
            f"R² = {r2:.3f}\n"
            f"Slope = {m:.3f}\n"
            f"RMSE vs y=x = {rmse_identity:.4g}"
        )
        ax.text(
            0.03, 0.97, txt,
            transform=ax.transAxes,
            ha="left", va="top", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", alpha=0.9)
        )

        style_axes(ax)
        fig.tight_layout()
        fig.savefig(
            os.path.join(out_dir, f"pair_scatter_identity_{feat}_A_vs_B.pdf"),
            dpi=FIG_DPI,
            bbox_inches="tight",
            transparent=True
        )
        plt.close(fig)


# =========================
# PCA / feature-space comparison
# =========================
def plot_pca_scatter(PA, PB, out_dir, use_density_contour=True):
    fig, ax = plt.subplots(figsize=(5.5, 5.0))

    xA = PA[:, 0]
    xB = PB[:, 0]

    if PA.shape[1] >= 2:
        yA = PA[:, 1]
        yB = PB[:, 1]
        ylabel = "PC2"
    else:
        yA = np.zeros_like(xA)
        yB = np.zeros_like(xB)
        ylabel = "0"

    ax.scatter(xA, yA, s=POINT_SIZE, alpha=POINT_ALPHA, color=COLOR_PCA_A, edgecolors="none", label="Method A", zorder=2)
    ax.scatter(xB, yB, s=POINT_SIZE, alpha=POINT_ALPHA, color=COLOR_PCA_B, edgecolors="none", label="Method B", zorder=2)

    if use_density_contour and PA.shape[1] >= 2:
        add_density_contour(ax, xA, yA, COLOR_PCA_A, levels=CONTOUR_LEVELS, grid_n=CONTOUR_GRID_N)
        add_density_contour(ax, xB, yB, COLOR_PCA_B, levels=CONTOUR_LEVELS, grid_n=CONTOUR_GRID_N)

    ax.set_xlabel("PC1")
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False)
    style_axes(ax)
    fig.tight_layout()
    fig.savefig(
        os.path.join(out_dir, "pca_1d_feature_space_A_vs_B.pdf"),
        dpi=FIG_DPI,
        bbox_inches="tight",
        transparent=True
    )
    plt.close(fig)


def save_pca_loadings(pca, feature_names, out_dir):
    comps = pca.components_
    rows = []
    for i in range(comps.shape[0]):
        row = {"component": f"PC{i+1}"}
        for f, v in zip(feature_names, comps[i]):
            row[f] = float(v)
        rows.append(row)

    load_df = pd.DataFrame(rows)
    load_df.to_csv(os.path.join(out_dir, "pca_loadings_1d_features.csv"), index=False)
    return load_df


def save_explained_variance(pca, out_dir):
    ev = pd.DataFrame({
        "component": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "explained_variance_percent": pca.explained_variance_ratio_ * 100.0
    })
    ev.to_csv(os.path.join(out_dir, "pca_explained_variance_1d_features.csv"), index=False)
    return ev


# =========================
# Main
# =========================
def main():
    ensure_dir(OUT_DIR)

    df, files = load_all_pairs(PAIR_CSV_DIR, GLOB_PATTERN)
    print(f"Loaded files: {len(files)}")
    print(f"Rows before filtering: {len(df)}")

    df = add_um_columns_if_needed(df)

    feature_pairs = FEATURE_PAIRS.copy()
    if INCLUDE_AREA_REFERENCE:
        feature_pairs.update(AREA_PAIR)

    validate_columns(df, feature_pairs)

    df = filter_pairs(df)
    print(f"Rows after filtering: {len(df)}")

    sub, A_df, B_df, feature_names = prepare_feature_matrices(df, feature_pairs)
    print(f"Valid rows for 1D-feature comparison: {len(sub)}")

    if len(sub) < 3:
        raise ValueError("Too few valid rows after filtering.")

    sub.to_csv(os.path.join(OUT_DIR, "filtered_pairs_used_for_1d_analysis.csv"), index=False)

    plot_featurewise_pair_scatter_with_identity(sub, feature_pairs, OUT_DIR)

    scaler, ZA, ZB = joint_normalize(A_df, B_df, scaler_type=SCALER_TYPE)
    pca, PA, PB = run_pca(ZA, ZB, n_components=2)

    pca_df_A = pd.DataFrame(PA, columns=[f"PC{i+1}" for i in range(PA.shape[1])], index=sub.index)
    pca_df_B = pd.DataFrame(PB, columns=[f"PC{i+1}" for i in range(PB.shape[1])], index=sub.index)
    pca_df_A["method"] = "A"
    pca_df_B["method"] = "B"
    pca_df_A["source_file"] = sub["source_file"].values if "source_file" in sub.columns else None
    pca_df_B["source_file"] = sub["source_file"].values if "source_file" in sub.columns else None

    pd.concat([pca_df_A, pca_df_B], axis=0, ignore_index=True).to_csv(
        os.path.join(OUT_DIR, "pca_scores_1d_features.csv"), index=False
    )

    plot_pca_scatter(PA, PB, OUT_DIR, use_density_contour=USE_DENSITY_CONTOUR)

    load_df = save_pca_loadings(pca, feature_names, OUT_DIR)
    ev_df = save_explained_variance(pca, OUT_DIR)

    with open(os.path.join(OUT_DIR, "analysis_summary.txt"), "w", encoding="utf-8") as f:
        f.write("1D feature-based pair comparison summary\n")
        f.write("=======================================\n")
        f.write(f"Loaded files: {len(files)}\n")
        f.write(f"Rows after filtering: {len(df)}\n")
        f.write(f"Rows used for analysis: {len(sub)}\n")
        f.write(f"Features used: {feature_names}\n")
        f.write(f"Scaler: {SCALER_TYPE}\n")
        f.write(f"USE_ONLY_LINE_LIKE: {USE_ONLY_LINE_LIKE}\n")
        f.write(f"INCLUDE_AREA_REFERENCE: {INCLUDE_AREA_REFERENCE}\n\n")

        f.write("Explained variance:\n")
        f.write(ev_df.to_string(index=False))
        f.write("\n\nPCA loadings:\n")
        f.write(load_df.to_string(index=False))

    print("Saved outputs:")
    print(" - filtered_pairs_used_for_1d_analysis.csv")
    for feat in feature_pairs.keys():
        print(f" - pair_scatter_identity_{feat}_A_vs_B.pdf")
    print(" - pca_1d_feature_space_A_vs_B.pdf")
    print(" - pca_scores_1d_features.csv")
    print(" - pca_loadings_1d_features.csv")
    print(" - pca_explained_variance_1d_features.csv")
    print(" - analysis_summary.txt")


if __name__ == "__main__":
    main()


Loaded files: 6
Rows before filtering: 782
Rows after filtering: 782
Valid rows for 1D-feature comparison: 782
Saved outputs:
 - filtered_pairs_used_for_1d_analysis.csv
 - pair_scatter_identity_skeleton_length_um_A_vs_B.pdf
 - pair_scatter_identity_linearity_index_A_vs_B.pdf
 - pair_scatter_identity_major_axis_um_A_vs_B.pdf
 - pair_scatter_identity_minor_axis_um_A_vs_B.pdf
 - pair_scatter_identity_area_um2_A_vs_B.pdf
 - pca_1d_feature_space_A_vs_B.pdf
 - pca_scores_1d_features.csv
 - pca_loadings_1d_features.csv
 - pca_explained_variance_1d_features.csv
 - analysis_summary.txt


For subset in Figure E,F

In [19]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.size"] = 28
mpl.rcParams["axes.linewidth"] = 1.2
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42


def plot_boxplot_from_sampled_csv(raw_csv_path, save_path=None):
    df = pd.read_csv(raw_csv_path)

    required_cols = {"region", "intensity"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"CSV must contain columns: {required_cols}")

    cell_vals = df.loc[df["region"] == "Cell", "intensity"].dropna().values
    bg_vals = df.loc[df["region"] == "Background", "intensity"].dropna().values

    print(f"Cell pixels used:       {cell_vals.size}")
    print(f"Background pixels used: {bg_vals.size}")

    if cell_vals.size == 0 or bg_vals.size == 0:
        raise ValueError("Cell or Background data is empty.")

    fig, ax = plt.subplots(figsize=(3.2, 3.2))

    bp = ax.boxplot(
        [cell_vals, bg_vals],
        labels=["Cell", "Background"],
        showfliers=False,
        widths=0.6
    )

    for element in ["boxes", "whiskers", "caps", "medians"]:
        for line in bp[element]:
            line.set_linewidth(1.2)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="in", length=4, width=1.0)

    fig.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.patch.set_alpha(0)
        ax.set_facecolor("none")
        plt.savefig(save_path, bbox_inches="tight", transparent=True)
        plt.close()
        print(f"Saved boxplot: {save_path}")
    else:
        plt.show()


raw_csv = r"C:\Users\oxfil\Cell_optical_media_comparision_new\inside_vs_outside_raw_pixel_intensity_cmos.csv"

out_pdf = r"C:\Users\oxfil\Cell_optical_media_comparision_new\inside_vs_outside_boxplot_pixel_intensity_cmos.pdf"

plot_boxplot_from_sampled_csv(
    raw_csv_path=raw_csv,
    save_path=out_pdf
)

Cell pixels used:       1662549
Background pixels used: 8208851


C:\Users\oxfil\AppData\Local\Temp\ipykernel_15236\3479521449.py:32: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


Saved boxplot: C:\Users\oxfil\Cell_optical_media_comparision_new\inside_vs_outside_boxplot_pixel_intensity_cmos.pdf
